## 01 — Data Inventory and Audit

## Objective

This notebook performs a forensic audit of the raw Flyer × Hartwig genotype and
phenotype workbooks before any cleaning, filtering, linkage mapping, or QTL analysis.

The goals are to determine:

1. workbook sheet names;
2. table dimensions;
3. RIL identifier structure;
4. marker count;
5. phenotype column count;
6. duplicate column names;
7. missing-value structure;
8. genotype allele coding;
9. whether genotype and phenotype RIL identifiers match.

### Important principle

No raw data will be modified in this notebook.

All observations will be documented first. Any later cleaning decisions will be
performed explicitly and saved to `data/interim/` or `data/processed/`.

The historical `fxh_maps.xlsx` file remains excluded from the primary analytical
workflow at this stage.

In [ ]:
# Cell 00.02
# Define the project root and standard project directories.

from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()

# Standard project directories
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_REFERENCE = PROJECT_ROOT / "reference"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

NOTEBOOKS = PROJECT_ROOT / "notebooks"

RESULTS = PROJECT_ROOT / "results"
RESULTS_FIGURES = RESULTS / "figures"
RESULTS_TABLES = RESULTS / "tables"
RESULTS_QTL = RESULTS / "qtl"
RESULTS_MAP = RESULTS / "linkage_map"
RESULTS_CANDIDATES = RESULTS / "candidate_genes"
RESULTS_VALIDATION = RESULTS / "external_validation"

SRC = PROJECT_ROOT / "src"
LOGS = PROJECT_ROOT / "logs"
DOCS = PROJECT_ROOT / "docs"

directories = [
    DATA_RAW,
    DATA_REFERENCE,
    DATA_INTERIM,
    DATA_PROCESSED,
    NOTEBOOKS,
    RESULTS,
    RESULTS_FIGURES,
    RESULTS_TABLES,
    RESULTS_QTL,
    RESULTS_MAP,
    RESULTS_CANDIDATES,
    RESULTS_VALIDATION,
    SRC,
    LOGS,
    DOCS,
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT")
print(PROJECT_ROOT)

print("\nProject root exists:", PROJECT_ROOT.exists())
print("Number of project directories checked:", len(directories))

In [ ]:
# Cell 00.03
# Record the computational environment and establish reproducibility settings.

import sys
import platform
import random

import numpy as np
import pandas as pd

SEED = 20260910

random.seed(SEED)
np.random.seed(SEED)

print("Flyer × Hartwig RIL Project")
print("=" * 50)

print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"NumPy version  : {np.__version__}")
print(f"pandas version : {pd.__version__}")
print(f"Random seed    : {SEED}")

print("\nWorking project root:")
print(PROJECT_ROOT)

In [ ]:
# Cell 00.04
# Locate primary raw input files.
#
# IMPORTANT:
# fxh_maps.xlsx is intentionally NOT included as an analytical input.
# We will reconstruct the linkage map from the genotype dataset if feasible.

GENOTYPE_FILE = DATA_RAW / "fxh_genotypes.xlsx"
PHENOTYPE_FILE = DATA_RAW / "fxh_phenotypes.xlsx"

primary_files = {
    "Genotypes": GENOTYPE_FILE,
    "Phenotypes": PHENOTYPE_FILE,
}

print("PRIMARY ANALYTICAL INPUT FILES")
print("=" * 60)

for label, path in primary_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label:12s}: {status}")
    print(f"              {path}")

print("\nHistorical map:")
print("fxh_maps.xlsx is intentionally excluded from the primary workflow.")
print("Do NOT delete it; retain it as a legacy reference.")

### Cell 01.02 — Inspect workbook structure

In [ ]:
# Cell 01.02
# Inspect workbook sheet names without modifying data.

import pandas as pd

geno_xls = pd.ExcelFile(GENOTYPE_FILE)
pheno_xls = pd.ExcelFile(PHENOTYPE_FILE)

print("GENOTYPE WORKBOOK")
print("=" * 60)
print("File:", GENOTYPE_FILE.name)
print("Sheets:", geno_xls.sheet_names)

print("\nPHENOTYPE WORKBOOK")
print("=" * 60)
print("File:", PHENOTYPE_FILE.name)
print("Sheets:", pheno_xls.sheet_names)

### Cell 01.03 — Load the first sheet exactly as stored
* For now, load only the first sheet from each workbook.

In [ ]:
# Cell 01.03
# Load the first worksheet from each workbook exactly as stored.

geno_raw = pd.read_excel(
    GENOTYPE_FILE,
    sheet_name=geno_xls.sheet_names[0]
)

pheno_raw = pd.read_excel(
    PHENOTYPE_FILE,
    sheet_name=pheno_xls.sheet_names[0]
)

print("GENOTYPE DATA")
print("=" * 60)
print("Shape:", geno_raw.shape)
print("Rows :", geno_raw.shape[0])
print("Columns:", geno_raw.shape[1])

print("\nFirst 10 column names:")
for i, col in enumerate(geno_raw.columns[:10], start=1):
    print(f"{i:>3}. {repr(col)}")

print("\nFirst 5 rows:")
display(geno_raw.head())


print("\n\nPHENOTYPE DATA")
print("=" * 60)
print("Shape:", pheno_raw.shape)
print("Rows :", pheno_raw.shape[0])
print("Columns:", pheno_raw.shape[1])

print("\nFirst 15 column names:")
for i, col in enumerate(pheno_raw.columns[:15], start=1):
    print(f"{i:>3}. {repr(col)}")

print("\nFirst 5 rows:")
display(pheno_raw.head())

### Cell 01.04 — Identify likely ID columns, duplicates, and basic missingness

In [ ]:
# Cell 01.04
# Audit column names, possible identifier columns, duplicates, and missingness.

def audit_dataframe(df, name):
    print(f"\n{name}")
    print("=" * 70)

    # Dimensions
    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    # Duplicate column names
    duplicated_cols = df.columns[df.columns.duplicated()].tolist()

    print("\nDuplicate column names:")
    if duplicated_cols:
        for col in duplicated_cols:
            print(" -", repr(col))
    else:
        print(" None")

    # Likely identifier columns
    id_keywords = [
        "id", "ril", "line", "entry",
        "genotype", "name", "plant"
    ]

    possible_id_cols = [
        col for col in df.columns
        if any(keyword in str(col).lower() for keyword in id_keywords)
    ]

    print("\nPossible identifier columns:")
    if possible_id_cols:
        for col in possible_id_cols:
            print(" -", repr(col))
    else:
        print(" None detected automatically")

    # Missingness
    missing = df.isna().sum().sort_values(ascending=False)

    print("\nColumns with the most missing values:")
    print(missing.head(15))

    # Data types
    print("\nData-type counts:")
    print(df.dtypes.value_counts())


audit_dataframe(geno_raw, "GENOTYPE AUDIT")
audit_dataframe(pheno_raw, "PHENOTYPE AUDIT")

### Cell 01.05 — Markdown

## Genotype coding and sample identity audit

The genotype matrix is organized in the preferred orientation:

- 94 rows corresponding to Flyer × Hartwig RILs;
- 417 molecular marker columns;
- one RIL identifier column (`ril`).

Before genotype filtering or linkage-map reconstruction, we will determine:

1. the complete set of raw genotype codes;
2. the proportion of missing genotype calls;
3. missingness per RIL;
4. missingness per marker;
5. whether genotype values are consistently encoded;
6. whether all expected RIL identifiers are unique; and
7. whether genotype and phenotype RIL identifiers match.

No markers or RILs will be removed at this stage.

The historical 110-marker map remains excluded from the primary analysis and
will be retained only for later comparison and validation.

### Cell 01.06 — Audit genotype coding and missingness

In [ ]:
# Cell 01.06
# Characterize genotype coding and missingness without modifying raw data.

import numpy as np
import pandas as pd

GENO_ID = "ril"

marker_cols = [
    col for col in geno_raw.columns
    if col != GENO_ID
]

print("GENOTYPE MATRIX STRUCTURE")
print("=" * 70)

print(f"RIL rows            : {geno_raw.shape[0]}")
print(f"Marker columns      : {len(marker_cols)}")
print(f"Total columns       : {geno_raw.shape[1]}")
print(f"RIL identifier      : {GENO_ID!r}")


# ------------------------------------------------------------
# Inventory all non-missing genotype values
# ------------------------------------------------------------

geno_values = geno_raw[marker_cols].stack(dropna=True)

print("\nOBSERVED RAW GENOTYPE VALUES")
print("=" * 70)

value_counts = geno_values.value_counts()

for value, count in value_counts.items():
    print(
        f"value={repr(value):>10} | "
        f"type={type(value).__name__:>8} | "
        f"count={count:,}"
    )


# ------------------------------------------------------------
# Overall missingness
# ------------------------------------------------------------

total_cells = geno_raw[marker_cols].size

total_missing = (
    geno_raw[marker_cols]
    .isna()
    .sum()
    .sum()
)

overall_missing_pct = (
    100 * total_missing / total_cells
)

print("\nOVERALL GENOTYPE MISSINGNESS")
print("=" * 70)

print(f"Total genotype cells       : {total_cells:,}")
print(f"Missing genotype cells     : {total_missing:,}")
print(f"Overall missingness (%)    : {overall_missing_pct:.2f}")


# ------------------------------------------------------------
# Missingness by RIL
# ------------------------------------------------------------

ril_missing = pd.DataFrame({
    "ril": geno_raw[GENO_ID].astype(str),
    "n_missing": geno_raw[marker_cols].isna().sum(axis=1)
})

ril_missing["n_observed"] = (
    len(marker_cols) - ril_missing["n_missing"]
)

ril_missing["missing_pct"] = (
    100 * ril_missing["n_missing"] / len(marker_cols)
)

ril_missing = (
    ril_missing
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Missingness by marker
# ------------------------------------------------------------

marker_missing = pd.DataFrame({
    "marker": marker_cols,
    "n_missing": geno_raw[marker_cols].isna().sum(axis=0).values
})

marker_missing["n_observed"] = (
    geno_raw.shape[0] - marker_missing["n_missing"]
)

marker_missing["missing_pct"] = (
    100 * marker_missing["n_missing"] / geno_raw.shape[0]
)

marker_missing = (
    marker_missing
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)


print("\n15 RILs WITH HIGHEST GENOTYPE MISSINGNESS")
display(ril_missing.head(15))

print("\n15 MARKERS WITH HIGHEST GENOTYPE MISSINGNESS")
display(marker_missing.head(15))

### Cell 01.07 — Audit genotype and phenotype IDs

In [ ]:
# Cell 01.07
# Audit RIL identifiers in genotype and phenotype datasets.

GENO_ID = "ril"
PHENO_ID = "ril"

print("GENOTYPE RIL IDENTIFIER AUDIT")
print("=" * 70)

print(f"Rows                  : {len(geno_raw)}")
print(f"Non-missing IDs       : {geno_raw[GENO_ID].notna().sum()}")
print(f"Unique IDs            : {geno_raw[GENO_ID].nunique(dropna=True)}")
print(
    f"Duplicated IDs        : "
    f"{geno_raw[GENO_ID].duplicated(keep=False).sum()}"
)


print("\nPHENOTYPE RIL IDENTIFIER AUDIT")
print("=" * 70)

print(f"Rows                  : {len(pheno_raw)}")
print(f"Non-missing IDs       : {pheno_raw[PHENO_ID].notna().sum()}")
print(f"Unique IDs            : {pheno_raw[PHENO_ID].nunique(dropna=True)}")
print(
    f"Duplicated IDs        : "
    f"{pheno_raw[PHENO_ID].duplicated(keep=False).sum()}"
)


# ------------------------------------------------------------
# Show duplicated IDs if any
# ------------------------------------------------------------

geno_duplicates = geno_raw[
    geno_raw[GENO_ID].duplicated(keep=False)
].copy()

pheno_duplicates = pheno_raw[
    pheno_raw[PHENO_ID].duplicated(keep=False)
].copy()


print("\nDUPLICATED GENOTYPE IDs")
print("=" * 70)

if geno_duplicates.empty:
    print("None")
else:
    display(geno_duplicates)


print("\nDUPLICATED PHENOTYPE IDs")
print("=" * 70)

if pheno_duplicates.empty:
    print("None")
else:
    display(pheno_duplicates)


# ------------------------------------------------------------
# Identify nonstandard phenotype IDs
# ------------------------------------------------------------

expected_prefix = "fxh_ril_"

nonstandard_pheno = pheno_raw[
    ~pheno_raw[PHENO_ID]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.startswith(expected_prefix)
].copy()

print("\nNONSTANDARD PHENOTYPE IDs")
print("=" * 70)

if nonstandard_pheno.empty:
    print("None")
else:
    display(nonstandard_pheno)

### Cell 01.08 — Match genotype and phenotype samples

In [ ]:
# Cell 01.08
# Compare genotype and phenotype identifiers.
# No merging or deletion is performed.

geno_ids = (
    geno_raw[GENO_ID]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

pheno_ids = (
    pheno_raw[PHENO_ID]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

geno_set = set(geno_ids)
pheno_set = set(pheno_ids)

shared_ids = sorted(geno_set & pheno_set)
geno_only = sorted(geno_set - pheno_set)
pheno_only = sorted(pheno_set - geno_set)


print("GENOTYPE ↔ PHENOTYPE SAMPLE MATCH")
print("=" * 70)

print(f"Unique genotype IDs      : {len(geno_set)}")
print(f"Unique phenotype IDs     : {len(pheno_set)}")
print(f"Shared IDs               : {len(shared_ids)}")
print(f"Genotype-only IDs        : {len(geno_only)}")
print(f"Phenotype-only IDs       : {len(pheno_only)}")


print("\nGENOTYPE-ONLY IDs")
print("=" * 70)

if geno_only:
    for x in geno_only:
        print(x)
else:
    print("None")


print("\nPHENOTYPE-ONLY IDs")
print("=" * 70)

if pheno_only:
    for x in pheno_only:
        print(x)
else:
    print("None")


# ------------------------------------------------------------
# Create audit table
# ------------------------------------------------------------

all_ids = sorted(geno_set | pheno_set)

sample_match_audit = pd.DataFrame({
    "id": all_ids
})

sample_match_audit["in_genotype"] = (
    sample_match_audit["id"].isin(geno_set)
)

sample_match_audit["in_phenotype"] = (
    sample_match_audit["id"].isin(pheno_set)
)

sample_match_audit["status"] = np.select(
    [
        sample_match_audit["in_genotype"]
        & sample_match_audit["in_phenotype"],

        sample_match_audit["in_genotype"]
        & ~sample_match_audit["in_phenotype"],

        ~sample_match_audit["in_genotype"]
        & sample_match_audit["in_phenotype"],
    ],
    [
        "matched",
        "genotype_only",
        "phenotype_only",
    ],
    default="check"
)

print("\nSAMPLE MATCH SUMMARY")
print("=" * 70)

display(
    sample_match_audit["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="n")
)

### Cell 01.09 — Markdown

## Marker-level genotype quality control

The raw genotype audit identified two important issues:

1. genotype calls are primarily encoded as `0` and `2`;
2. a small number of cells contain text strings such as `"NaN "` rather than
   true missing values.

Before evaluating segregation or linkage information, these text-based missing
codes will be converted to genuine missing values in a new working copy.

The raw dataframe (`geno_raw`) will remain unchanged.

This section will evaluate:

- marker missingness;
- genotype-class counts;
- monomorphic markers;
- markers with very low minor-allele representation;
- segregation distortion from the expected 1:1 ratio for RILs.

No markers will be removed yet. Instead, each marker will receive QC flags that
can later be used to define alternative linkage-map datasets.

### Cell 01.10 — normalize genotype missing values without warnings

In [ ]:
# Cell 01.10
# Create a cleaned working copy of the genotype matrix.
# Raw data remain unchanged.
#
# This version avoids pandas replace() downcasting warnings.

geno_work = geno_raw.copy()

for col in marker_cols:
    geno_work[col] = (
        geno_work[col]
        .astype("string")
        .str.strip()
        .replace("NaN", pd.NA)
        .pipe(pd.to_numeric, errors="coerce")
    )

print("GENOTYPE NORMALIZATION CHECK")
print("=" * 70)

remaining_values = (
    geno_work[marker_cols]
    .stack(future_stack=True)
    .dropna()
    .value_counts()
    .sort_index()
)

print("Observed genotype values after normalization:")
print(remaining_values)

print("\nUnexpected genotype values:")

unexpected = remaining_values[
    ~remaining_values.index.isin([0, 2])
]

if unexpected.empty:
    print("None — all non-missing calls are 0 or 2.")
else:
    print(unexpected)

n_missing_clean = (
    geno_work[marker_cols]
    .isna()
    .sum()
    .sum()
)

print("\nMissing values after normalization:", f"{n_missing_clean:,}")

print(
    "Change relative to original pandas-missing count:",
    n_missing_clean - total_missing
)

### Cell 01.11 — Build marker QC table

In [ ]:
# Cell 01.11
# Calculate genotype counts, missingness, allele balance,
# and informativeness for every marker.

marker_qc = []

for marker in marker_cols:

    x = geno_work[marker]

    n_total = len(x)
    n_missing = x.isna().sum()
    n_observed = x.notna().sum()

    n_0 = (x == 0).sum()
    n_2 = (x == 2).sum()

    missing_pct = 100 * n_missing / n_total

    if n_observed > 0:
        freq_0 = n_0 / n_observed
        freq_2 = n_2 / n_observed
        minor_class_freq = min(freq_0, freq_2)
    else:
        freq_0 = np.nan
        freq_2 = np.nan
        minor_class_freq = np.nan

    # Marker is monomorphic if only one genotype class is observed.
    n_classes = len(
        set(x.dropna().unique())
    )

    monomorphic = n_classes <= 1

    marker_qc.append({
        "marker": marker,
        "n_total": n_total,
        "n_observed": n_observed,
        "n_missing": n_missing,
        "missing_pct": missing_pct,
        "n_0": n_0,
        "n_2": n_2,
        "freq_0": freq_0,
        "freq_2": freq_2,
        "minor_class_freq": minor_class_freq,
        "n_genotype_classes": n_classes,
        "monomorphic": monomorphic
    })


marker_qc = pd.DataFrame(marker_qc)


print("MARKER QC SUMMARY")
print("=" * 70)

print(f"Total markers                : {len(marker_qc)}")
print(
    f"Monomorphic markers          : "
    f"{marker_qc['monomorphic'].sum()}"
)

print(
    f"Markers missing >50%         : "
    f"{(marker_qc['missing_pct'] > 50).sum()}"
)

print(
    f"Markers missing >30%         : "
    f"{(marker_qc['missing_pct'] > 30).sum()}"
)

print(
    f"Markers minor class <10%     : "
    f"{(marker_qc['minor_class_freq'] < 0.10).sum()}"
)


print("\n15 MARKERS WITH HIGHEST MISSINGNESS")
display(
    marker_qc
    .sort_values("missing_pct", ascending=False)
    .head(15)
)


print("\nMONOMORPHIC MARKERS")
display(
    marker_qc[
        marker_qc["monomorphic"]
    ].sort_values("missing_pct")
)

### Cell 01.12 — Test 1:1 segregation for each marker
* For an advanced RIL derived from two homozygous parents, the expected segregation for a codominant homozygous marker is approximately:

$$ 1:1 $$

* between the two parental allele classes.
* We will use a chi-square test as a first diagnostic.

In [ ]:
# Cell 01.12
# Test segregation against the expected 1:1 ratio.

from scipy.stats import chisquare

seg_results = []

for _, row in marker_qc.iterrows():

    marker = row["marker"]
    n0 = int(row["n_0"])
    n2 = int(row["n_2"])
    n_obs = n0 + n2

    # Require both classes and at least modest sample size.
    if n_obs >= 10 and n0 > 0 and n2 > 0:

        expected = [n_obs / 2, n_obs / 2]

        chi2, p_value = chisquare(
            f_obs=[n0, n2],
            f_exp=expected
        )

    else:
        chi2 = np.nan
        p_value = np.nan

    seg_results.append({
        "marker": marker,
        "chi2_1to1": chi2,
        "segregation_p": p_value
    })


seg_results = pd.DataFrame(seg_results)

marker_qc = marker_qc.merge(
    seg_results,
    on="marker",
    how="left"
)


# Add diagnostic flags.
marker_qc["distorted_p05"] = (
    marker_qc["segregation_p"] < 0.05
)

marker_qc["distorted_p01"] = (
    marker_qc["segregation_p"] < 0.01
)

marker_qc["distorted_p001"] = (
    marker_qc["segregation_p"] < 0.001
)


print("SEGREGATION DISTORTION SUMMARY")
print("=" * 70)

print(
    "Markers tested:",
    marker_qc["segregation_p"].notna().sum()
)

print(
    "P < 0.05:",
    marker_qc["distorted_p05"].sum()
)

print(
    "P < 0.01:",
    marker_qc["distorted_p01"].sum()
)

print(
    "P < 0.001:",
    marker_qc["distorted_p001"].sum()
)


print("\n20 MOST STRONGLY DISTORTED MARKERS")
display(
    marker_qc[
        marker_qc["segregation_p"].notna()
    ]
    .sort_values("segregation_p")
    [
        [
            "marker",
            "n_observed",
            "n_0",
            "n_2",
            "freq_0",
            "freq_2",
            "missing_pct",
            "chi2_1to1",
            "segregation_p"
        ]
    ]
    .head(20)
)

### Cell 01.13 — Markdown

## Redundant and co-segregating marker assessment

Linkage maps should not treat every marker name as necessarily representing an
independent recombination position.

Two or more markers may:

- have identical genotype patterns across all informative RILs;
- be nearly identical because of scoring differences or missing calls;
- represent different molecular assays at the same genomic position;
- contain insufficient overlapping observations to judge similarity reliably.

This section will therefore examine marker redundancy before linkage-group
construction.

Important distinction:

An exact match will only be declared when two markers have the same genotype
calls at every RIL where both markers are observed.

The number of jointly observed RILs will also be recorded because apparent
agreement based on very few samples is not strong evidence of co-segregation.

### Cell 01.14 — Check marker names and obvious duplicates

In [ ]:
# Cell 01.14
# Examine marker identifiers for exact duplicates and suspicious naming patterns.

marker_names = pd.Series(marker_cols, name="marker")

print("MARKER NAME AUDIT")
print("=" * 70)

print(f"Total marker columns      : {len(marker_names)}")
print(f"Unique marker names       : {marker_names.nunique()}")
print(f"Exact duplicate names     : {marker_names.duplicated().sum()}")

duplicate_names = marker_names[
    marker_names.duplicated(keep=False)
]

print("\nEXACT DUPLICATE MARKER NAMES")
print("=" * 70)

if duplicate_names.empty:
    print("None")
else:
    display(
        duplicate_names
        .to_frame()
        .sort_values("marker")
    )


# ------------------------------------------------------------
# Inspect suffix variants that may indicate replicate assays,
# alternative bands, or manually distinguished loci.
# ------------------------------------------------------------

suspicious_suffixes = marker_names[
    marker_names.str.contains(
        r"(?:\.1$|_[12]$|[a-z]$)",
        regex=True,
        na=False
    )
]

print("\nMARKERS WITH POSSIBLE VARIANT SUFFIXES")
print("=" * 70)
print(f"Count: {len(suspicious_suffixes)}")

display(
    suspicious_suffixes
    .to_frame()
    .head(50)
)

### Cell 01.15 — Compute pairwise marker agreement
* This will compare every marker pair, but only where both have observed genotype calls.

In [ ]:
# Cell 01.15
# Calculate pairwise agreement between markers using only jointly observed RILs.

from itertools import combinations

pairwise_results = []

for marker_a, marker_b in combinations(marker_cols, 2):

    a = geno_work[marker_a]
    b = geno_work[marker_b]

    jointly_observed = a.notna() & b.notna()
    n_overlap = jointly_observed.sum()

    if n_overlap == 0:
        continue

    a_obs = a[jointly_observed]
    b_obs = b[jointly_observed]

    n_same = (a_obs == b_obs).sum()
    n_different = (a_obs != b_obs).sum()

    agreement = n_same / n_overlap

    pairwise_results.append({
        "marker_a": marker_a,
        "marker_b": marker_b,
        "n_overlap": int(n_overlap),
        "n_same": int(n_same),
        "n_different": int(n_different),
        "agreement": agreement
    })


marker_pairs = pd.DataFrame(pairwise_results)

print("PAIRWISE MARKER COMPARISON")
print("=" * 70)

print(f"Marker pairs evaluated : {len(marker_pairs):,}")

print("\nOverlap distribution:")
display(
    marker_pairs["n_overlap"]
    .describe()
    .to_frame("n_overlap")
)

print("\n20 PAIRS WITH HIGHEST AGREEMENT")
display(
    marker_pairs
    .sort_values(
        ["agreement", "n_overlap"],
        ascending=[False, False]
    )
    .head(20)
)

### Cell 01.16 — Identify strong co-segregation candidates
* We should not call two markers equivalent merely because they match in 10 RILs. So for now we'll require reasonably substantial overlap.

In [ ]:
# Cell 01.16
# Identify marker pairs with strong or perfect genotype agreement.

MIN_OVERLAP = 40

strong_pairs = marker_pairs[
    marker_pairs["n_overlap"] >= MIN_OVERLAP
].copy()


perfect_pairs = strong_pairs[
    strong_pairs["agreement"] == 1.0
].copy()


near_identical_pairs = strong_pairs[
    (strong_pairs["agreement"] >= 0.98)
    & (strong_pairs["agreement"] < 1.0)
].copy()


print("CO-SEGREGATION CANDIDATES")
print("=" * 70)

print(f"Minimum jointly observed RILs : {MIN_OVERLAP}")
print(f"Pairs meeting overlap rule    : {len(strong_pairs):,}")
print(f"Perfectly agreeing pairs      : {len(perfect_pairs):,}")
print(f"98–<100% agreeing pairs       : {len(near_identical_pairs):,}")


print("\nPERFECTLY AGREEING MARKER PAIRS")
print("=" * 70)

if perfect_pairs.empty:
    print("None")
else:
    display(
        perfect_pairs
        .sort_values("n_overlap", ascending=False)
        .head(50)
    )


print("\nNEAR-IDENTICAL MARKER PAIRS")
print("=" * 70)

if near_identical_pairs.empty:
    print("None")
else:
    display(
        near_identical_pairs
        .sort_values(
            ["agreement", "n_overlap"],
            ascending=[False, False]
        )
        .head(50)
    )

### Cell 01.17 — Build exact co-segregation bins
* We will treat perfect-agreement marker pairs with at least 40 jointly observed RILs as members of the same provisional recombination bin.

In [ ]:
# Cell 01.17
# Construct exact co-segregation bins using perfect marker-pair agreement.
# Uses a simple union-find algorithm; no extra package is required.

# ------------------------------------------------------------
# Union-find helper functions
# ------------------------------------------------------------

parent = {marker: marker for marker in marker_cols}


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a


# ------------------------------------------------------------
# Join marker pairs that meet our exact-agreement criterion
# ------------------------------------------------------------

for _, row in perfect_pairs.iterrows():
    union(row["marker_a"], row["marker_b"])


# ------------------------------------------------------------
# Collect connected components
# ------------------------------------------------------------

components = {}

for marker in marker_cols:
    root = find(marker)
    components.setdefault(root, []).append(marker)


# Retain only bins containing >1 marker.
coseg_bins = [
    sorted(markers)
    for markers in components.values()
    if len(markers) > 1
]

# Sort largest bins first.
coseg_bins = sorted(
    coseg_bins,
    key=lambda x: (-len(x), x[0])
)


print("EXACT CO-SEGREGATION BINS")
print("=" * 70)

print(f"Total markers               : {len(marker_cols)}")
print(f"Multi-marker bins           : {len(coseg_bins)}")
print(
    f"Markers in multi-marker bins: "
    f"{sum(len(x) for x in coseg_bins)}"
)

redundant_positions = sum(
    len(x) - 1
    for x in coseg_bins
)

print(
    f"Redundant marker positions  : "
    f"{redundant_positions}"
)


print("\nBIN MEMBERS")
print("=" * 70)

for i, markers in enumerate(coseg_bins, start=1):
    print(
        f"Bin {i:02d} | n={len(markers):>2} | "
        + " | ".join(markers)
    )

### Cell 01.18 — Choose a provisional representative for each bin
* For the first map, we want one marker per exact recombination bin. 
* The other markers remain available and can later be placed back at the same map position.
* We will choose the representative primarily by maximum number of observed RILs.

In [ ]:
# Cell 01.18
# Select one provisional representative marker from each exact
# co-segregation bin.

bin_records = []
representative_records = []


for bin_number, markers in enumerate(coseg_bins, start=1):

    candidates = marker_qc[
        marker_qc["marker"].isin(markers)
    ].copy()

    # Prefer:
    # 1. highest number of observed genotypes
    # 2. lowest missingness
    # 3. highest minor-class frequency
    # 4. alphabetical marker name for deterministic tie-breaking
    candidates = candidates.sort_values(
        by=[
            "n_observed",
            "missing_pct",
            "minor_class_freq",
            "marker"
        ],
        ascending=[
            False,
            True,
            False,
            True
        ]
    )

    representative = candidates.iloc[0]["marker"]

    representative_records.append({
        "bin_id": f"BIN_{bin_number:03d}",
        "representative": representative,
        "bin_size": len(markers)
    })

    for marker in markers:
        bin_records.append({
            "bin_id": f"BIN_{bin_number:03d}",
            "marker": marker,
            "representative": representative,
            "is_representative": marker == representative
        })


coseg_bin_table = pd.DataFrame(bin_records)

coseg_representatives = pd.DataFrame(
    representative_records
)


print("CO-SEGREGATION BIN REPRESENTATIVES")
print("=" * 70)

display(coseg_representatives)


print("\nDETAILED BIN MEMBERSHIP")
print("=" * 70)

display(
    coseg_bin_table.sort_values(
        ["bin_id", "is_representative", "marker"],
        ascending=[True, False, True]
    )
)

### Cell 01.19 — Examine RIL missingness distribution
* Before deciding whether fxh_ril_03 or other poorly genotyped lines should be excluded from map building, let's see the complete distribution.

In [ ]:
# Cell 01.19
# Recalculate RIL-level missingness using the normalized genotype matrix
# and examine possible QC thresholds.

ril_qc = pd.DataFrame({
    "ril": geno_work[GENO_ID].astype(str),
    "n_missing": geno_work[marker_cols].isna().sum(axis=1)
})

ril_qc["n_observed"] = (
    len(marker_cols) - ril_qc["n_missing"]
)

ril_qc["missing_pct"] = (
    100 * ril_qc["n_missing"] / len(marker_cols)
)


print("RIL GENOTYPE MISSINGNESS DISTRIBUTION")
print("=" * 70)

display(
    ril_qc["missing_pct"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
    .to_frame("missing_pct")
)


print("\nRIL COUNTS ABOVE CANDIDATE MISSINGNESS THRESHOLDS")
print("=" * 70)

for threshold in [20, 25, 30, 35, 40, 50, 60, 70]:
    n = (ril_qc["missing_pct"] > threshold).sum()

    print(
        f"> {threshold:>2}% missing : "
        f"{n:>2} of {len(ril_qc)} RILs"
    )


print("\nALL RILs SORTED BY MISSINGNESS")
print("=" * 70)

display(
    ril_qc
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

### Cell 01.20 — Marker-QC threshold sensitivity
* Now let's see how many of the 417 markers survive several possible missing-data thresholds.

In [ ]:
# Cell 01.20
# Examine how different marker-QC rules affect the number of available markers.
# These are sensitivity calculations only; no markers are removed yet.

threshold_results = []

for max_missing in [10, 20, 30, 40, 50, 60]:

    subset = marker_qc[
        marker_qc["missing_pct"] <= max_missing
    ]

    threshold_results.append({
        "max_missing_pct": max_missing,
        "markers_retained": len(subset),
        "pct_markers_retained":
            100 * len(subset) / len(marker_qc),

        "distorted_p05":
            subset["distorted_p05"].sum(),

        "distorted_p01":
            subset["distorted_p01"].sum(),

        "distorted_p001":
            subset["distorted_p001"].sum()
    })


marker_threshold_summary = pd.DataFrame(
    threshold_results
)

print("MARKER QC THRESHOLD SENSITIVITY")
print("=" * 70)

display(marker_threshold_summary)


# ------------------------------------------------------------
# Additional candidate backbone sets
# ------------------------------------------------------------

candidate_30 = marker_qc[
    marker_qc["missing_pct"] <= 30
].copy()

candidate_30_nosevere_distortion = candidate_30[
    ~candidate_30["distorted_p001"]
].copy()


print("\nPROVISIONAL BACKBONE COUNTS")
print("=" * 70)

print(
    f"Markers with <=30% missing        : "
    f"{len(candidate_30)}"
)

print(
    f"<=30% missing and not P<0.001     : "
    f"{len(candidate_30_nosevere_distortion)}"
)

print(
    "\nThese are diagnostic candidate sets only; "
    "nothing has been permanently filtered."
)

### Cell 01.21 — Define provisional RIL QC rule

In [ ]:
# Cell 01.21
# Define a provisional RIL QC rule based on the observed missingness distribution.
# No raw data are modified.

RIL_MAX_MISSING_PCT = 50.0

ril_qc["retain_for_mapping"] = (
    ril_qc["missing_pct"] <= RIL_MAX_MISSING_PCT
)

retained_rils = ril_qc.loc[
    ril_qc["retain_for_mapping"],
    "ril"
].tolist()

excluded_rils = ril_qc.loc[
    ~ril_qc["retain_for_mapping"],
    ["ril", "n_observed", "n_missing", "missing_pct"]
].copy()


print("PROVISIONAL RIL QC")
print("=" * 70)

print(f"Missingness threshold     : <= {RIL_MAX_MISSING_PCT:.1f}%")
print(f"Original RILs             : {len(ril_qc)}")
print(f"RILs retained             : {len(retained_rils)}")
print(f"RILs excluded             : {len(excluded_rils)}")


print("\nEXCLUDED RILs")
print("=" * 70)

if excluded_rils.empty:
    print("None")
else:
    display(
        excluded_rils.sort_values(
            "missing_pct",
            ascending=False
        )
    )


print("\nRILs CLOSE TO THE THRESHOLD")
print("=" * 70)

display(
    ril_qc[
        ril_qc["missing_pct"] >= 40
    ]
    .sort_values("missing_pct", ascending=False)
)

### Cell 01.22 — Recompute marker QC using the retained 92 RILs

In [ ]:
# Cell 01.22
# Recompute marker-level QC after provisional RIL filtering.

geno_map_rils = (
    geno_work[
        geno_work[GENO_ID].isin(retained_rils)
    ]
    .copy()
    .reset_index(drop=True)
)

print("MAPPING RIL DATASET")
print("=" * 70)

print(f"RILs retained : {geno_map_rils.shape[0]}")
print(f"Markers       : {len(marker_cols)}")


marker_qc_92 = []

for marker in marker_cols:

    x = geno_map_rils[marker]

    n_total = len(x)
    n_missing = x.isna().sum()
    n_observed = x.notna().sum()

    n_0 = (x == 0).sum()
    n_2 = (x == 2).sum()

    missing_pct = (
        100 * n_missing / n_total
    )

    if n_observed > 0:
        freq_0 = n_0 / n_observed
        freq_2 = n_2 / n_observed
        minor_class_freq = min(freq_0, freq_2)
    else:
        freq_0 = np.nan
        freq_2 = np.nan
        minor_class_freq = np.nan

    n_classes = x.dropna().nunique()

    monomorphic = (
        n_classes <= 1
    )

    # Recalculate 1:1 segregation test.
    if (
        n_observed >= 10
        and n_0 > 0
        and n_2 > 0
    ):

        expected = [
            n_observed / 2,
            n_observed / 2
        ]

        chi2, p_value = chisquare(
            f_obs=[n_0, n_2],
            f_exp=expected
        )

    else:
        chi2 = np.nan
        p_value = np.nan

    marker_qc_92.append({
        "marker": marker,
        "n_total": n_total,
        "n_observed": n_observed,
        "n_missing": n_missing,
        "missing_pct": missing_pct,
        "n_0": n_0,
        "n_2": n_2,
        "freq_0": freq_0,
        "freq_2": freq_2,
        "minor_class_freq": minor_class_freq,
        "n_genotype_classes": n_classes,
        "monomorphic": monomorphic,
        "chi2_1to1": chi2,
        "segregation_p": p_value
    })


marker_qc_92 = pd.DataFrame(marker_qc_92)

marker_qc_92["distorted_p05"] = (
    marker_qc_92["segregation_p"] < 0.05
)

marker_qc_92["distorted_p01"] = (
    marker_qc_92["segregation_p"] < 0.01
)

marker_qc_92["distorted_p001"] = (
    marker_qc_92["segregation_p"] < 0.001
)


print("\nUPDATED MARKER QC — 92 RILs")
print("=" * 70)

print(f"Total markers             : {len(marker_qc_92)}")
print(
    "Monomorphic              :",
    marker_qc_92["monomorphic"].sum()
)
print(
    "Missing >30%             :",
    (marker_qc_92["missing_pct"] > 30).sum()
)
print(
    "Missing <=30%            :",
    (marker_qc_92["missing_pct"] <= 30).sum()
)
print(
    "Segregation P < 0.05     :",
    marker_qc_92["distorted_p05"].sum()
)
print(
    "Segregation P < 0.01     :",
    marker_qc_92["distorted_p01"].sum()
)
print(
    "Segregation P < 0.001    :",
    marker_qc_92["distorted_p001"].sum()
)


print("\n15 MARKERS WITH HIGHEST MISSINGNESS")
display(
    marker_qc_92
    .sort_values("missing_pct", ascending=False)
    .head(15)
)

### Cell 01.23 — Construct conservative marker-backbone candidates
* We'll now define the first formal candidate backbone, but still not permanently remove anything.
* Criteria:
    - marker missingness ≤30%;
    - both genotype classes represented;
    - minor genotype class ≥10%;
    - exclude only extreme segregation distortion at P < 0.001.

In [ ]:
# Cell 01.23
# Define the conservative candidate marker backbone.

MARKER_MAX_MISSING_PCT = 30.0
MIN_MINOR_CLASS_FREQ = 0.10
EXTREME_DISTORTION_P = 0.001


marker_qc_92["pass_missing"] = (
    marker_qc_92["missing_pct"]
    <= MARKER_MAX_MISSING_PCT
)

marker_qc_92["pass_polymorphic"] = (
    ~marker_qc_92["monomorphic"]
)

marker_qc_92["pass_minor_class"] = (
    marker_qc_92["minor_class_freq"]
    >= MIN_MINOR_CLASS_FREQ
)

marker_qc_92["pass_distortion"] = (
    marker_qc_92["segregation_p"].isna()
    |
    (
        marker_qc_92["segregation_p"]
        >= EXTREME_DISTORTION_P
    )
)


marker_qc_92["backbone_candidate"] = (
    marker_qc_92["pass_missing"]
    &
    marker_qc_92["pass_polymorphic"]
    &
    marker_qc_92["pass_minor_class"]
    &
    marker_qc_92["pass_distortion"]
)


print("CONSERVATIVE BACKBONE QC")
print("=" * 70)

print(
    f"Maximum marker missingness : "
    f"{MARKER_MAX_MISSING_PCT:.1f}%"
)

print(
    f"Minimum minor class freq   : "
    f"{MIN_MINOR_CLASS_FREQ:.2f}"
)

print(
    f"Extreme distortion cutoff  : "
    f"P < {EXTREME_DISTORTION_P}"
)


print("\nFILTER EFFECTS")
print("=" * 70)

print(
    "Pass missingness       :",
    marker_qc_92["pass_missing"].sum()
)

print(
    "Pass polymorphism      :",
    marker_qc_92["pass_polymorphic"].sum()
)

print(
    "Pass minor class       :",
    marker_qc_92["pass_minor_class"].sum()
)

print(
    "Pass distortion rule   :",
    marker_qc_92["pass_distortion"].sum()
)

print(
    "\nBackbone candidates    :",
    marker_qc_92["backbone_candidate"].sum()
)


failed_backbone = marker_qc_92[
    ~marker_qc_92["backbone_candidate"]
].copy()


print("\nREASONS FOR FAILURE")
print("=" * 70)

print(
    "Fail missingness       :",
    (~marker_qc_92["pass_missing"]).sum()
)

print(
    "Fail polymorphism      :",
    (~marker_qc_92["pass_polymorphic"]).sum()
)

print(
    "Fail minor class       :",
    (~marker_qc_92["pass_minor_class"]).sum()
)

print(
    "Fail extreme distortion:",
    (~marker_qc_92["pass_distortion"]).sum()
)

### Cell 01.24 — Reconcile co-segregation bins with backbone QC
* This is important because we shouldn't blindly retain Satt146 as a bin representative if, after RIL filtering, another member has better coverage.
* We'll choose the best eligible marker within each bin.

In [ ]:
# Cell 01.24
# Reconcile exact co-segregation bins with the 92-RIL marker QC.
# Choose the best backbone-eligible member of each bin.

bin_backbone_records = []

for bin_number, markers in enumerate(
    coseg_bins,
    start=1
):

    bin_id = f"BIN_{bin_number:03d}"

    candidates = marker_qc_92[
        marker_qc_92["marker"].isin(markers)
    ].copy()

    eligible = candidates[
        candidates["backbone_candidate"]
    ].copy()

    if not eligible.empty:

        eligible = eligible.sort_values(
            by=[
                "n_observed",
                "missing_pct",
                "minor_class_freq",
                "marker"
            ],
            ascending=[
                False,
                True,
                False,
                True
            ]
        )

        selected = eligible.iloc[0]["marker"]

    else:
        selected = None

    for marker in markers:

        row = candidates[
            candidates["marker"] == marker
        ].iloc[0]

        bin_backbone_records.append({
            "bin_id": bin_id,
            "marker": marker,
            "n_observed": row["n_observed"],
            "missing_pct": row["missing_pct"],
            "segregation_p": row["segregation_p"],
            "backbone_candidate":
                row["backbone_candidate"],
            "selected_bin_representative":
                marker == selected
                if selected is not None
                else False
        })


bin_backbone_qc = pd.DataFrame(
    bin_backbone_records
)


print("CO-SEGREGATION BIN QC AFTER RIL FILTERING")
print("=" * 70)

display(
    bin_backbone_qc.sort_values(
        [
            "bin_id",
            "selected_bin_representative",
            "n_observed"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


# ------------------------------------------------------------
# Construct deduplicated conservative backbone.
# ------------------------------------------------------------

markers_in_bins = set(
    bin_backbone_qc["marker"]
)

# Eligible markers that do not belong to any multi-marker bin.
unbinned_backbone = marker_qc_92.loc[
    marker_qc_92["backbone_candidate"]
    &
    ~marker_qc_92["marker"].isin(markers_in_bins),
    "marker"
].tolist()


# One selected marker from each eligible co-segregation bin.
bin_representatives_92 = (
    bin_backbone_qc.loc[
        bin_backbone_qc[
            "selected_bin_representative"
        ],
        "marker"
    ]
    .tolist()
)


backbone_markers = (
    unbinned_backbone
    + bin_representatives_92
)


print("\nDEDUPLICATED CONSERVATIVE BACKBONE")
print("=" * 70)

print(
    f"Eligible markers before bin collapse : "
    f"{marker_qc_92['backbone_candidate'].sum()}"
)

print(
    f"Selected bin representatives         : "
    f"{len(bin_representatives_92)}"
)

print(
    f"Unbinned backbone markers            : "
    f"{len(unbinned_backbone)}"
)

print(
    f"Final provisional backbone markers   : "
    f"{len(backbone_markers)}"
)

print(
    f"RILs in provisional mapping dataset  : "
    f"{len(retained_rils)}"
)

### Cell 01.25 — Check opposite-phase co-segregation
* For each marker pair, a perfect relationship can occur either as:
    - same phase: 0↔0, 2↔2
    - opposite phase: 0↔2, 2↔0
* We should detect both.

In [ ]:
# Cell 01.25
# Detect same-phase and opposite-phase co-segregating marker pairs.

phase_pairs = marker_pairs.copy()

phase_pairs["same_phase_agreement"] = (
    phase_pairs["n_same"] / phase_pairs["n_overlap"]
)

phase_pairs["opposite_phase_agreement"] = (
    phase_pairs["n_different"] / phase_pairs["n_overlap"]
)

phase_pairs["best_phase_agreement"] = phase_pairs[
    ["same_phase_agreement", "opposite_phase_agreement"]
].max(axis=1)

phase_pairs["best_phase"] = np.where(
    phase_pairs["same_phase_agreement"]
    >= phase_pairs["opposite_phase_agreement"],
    "same",
    "opposite"
)

phase_pairs["phase_adjusted_differences"] = np.minimum(
    phase_pairs["n_same"],
    phase_pairs["n_different"]
)


MIN_OVERLAP = 40

phase_strong = phase_pairs[
    phase_pairs["n_overlap"] >= MIN_OVERLAP
].copy()


perfect_phase_pairs = phase_strong[
    phase_strong["best_phase_agreement"] == 1.0
].copy()


near_phase_pairs = phase_strong[
    (phase_strong["best_phase_agreement"] >= 0.98)
    &
    (phase_strong["best_phase_agreement"] < 1.0)
].copy()


print("PHASE-INDEPENDENT CO-SEGREGATION AUDIT")
print("=" * 70)

print(f"Pairs with overlap >= {MIN_OVERLAP}       : {len(phase_strong):,}")

print(
    "Perfect pairs, either phase       :",
    len(perfect_phase_pairs)
)

print(
    "  Same-phase perfect              :",
    (perfect_phase_pairs["best_phase"] == "same").sum()
)

print(
    "  Opposite-phase perfect          :",
    (perfect_phase_pairs["best_phase"] == "opposite").sum()
)

print(
    "98–<100% pairs, either phase      :",
    len(near_phase_pairs)
)


print("\nPERFECT OPPOSITE-PHASE PAIRS")
print("=" * 70)

perfect_opposite = perfect_phase_pairs[
    perfect_phase_pairs["best_phase"] == "opposite"
].copy()

if perfect_opposite.empty:
    print("None")
else:
    display(
        perfect_opposite[
            [
                "marker_a",
                "marker_b",
                "n_overlap",
                "n_same",
                "n_different",
                "best_phase_agreement"
            ]
        ]
        .sort_values("n_overlap", ascending=False)
        .head(50)
    )

### Cell 01.26 — Calculate phase-adjusted pairwise linkage statistics

Now we work only with the 241 backbone markers.

For two markers, define the observed RIL recombinant fraction as

$$ R=\frac{\min(n_{\mathrm{different}},n_{\mathrm{same}})} {n_{\mathrm{overlap}}}. $$

Taking the minimum makes the calculation independent of marker coding phase.

For an advanced RIL produced by selfing, the relationship between the observed RIL recombinant fraction \(R\) and the underlying meiotic recombination fraction \(r\) is: r = R/2(1−R)

In [ ]:
# Cell 01.26
# Calculate pairwise phase-adjusted linkage statistics
# for the 241-marker conservative backbone.

from itertools import combinations
import math

backbone_pairwise = []

for marker_a, marker_b in combinations(backbone_markers, 2):

    a = geno_map_rils[marker_a]
    b = geno_map_rils[marker_b]

    valid = a.notna() & b.notna()
    n_overlap = int(valid.sum())

    if n_overlap == 0:
        continue

    a_obs = a[valid]
    b_obs = b[valid]

    n_same = int((a_obs == b_obs).sum())
    n_different = int((a_obs != b_obs).sum())

    # Choose the orientation that minimizes observed recombinants.
    if n_same >= n_different:
        phase = "same"
        n_nonrec = n_same
        n_rec = n_different
    else:
        phase = "opposite"
        n_nonrec = n_different
        n_rec = n_same

    R_ril = n_rec / n_overlap

    # Convert RIL recombinant fraction to meiotic recombination fraction.
    if R_ril < 0.5:
        r_meiotic = R_ril / (2 * (1 - R_ril))
    else:
        r_meiotic = 0.5

    # Pairwise LOD based on observed RIL recombinant states.
    if R_ril == 0:
        lod = n_nonrec * math.log10(2)
    elif R_ril < 0.5:
        lod = (
            n_nonrec * math.log10((1 - R_ril) / 0.5)
            +
            n_rec * math.log10(R_ril / 0.5)
        )
    else:
        lod = 0.0

    backbone_pairwise.append({
        "marker_a": marker_a,
        "marker_b": marker_b,
        "n_overlap": n_overlap,
        "phase": phase,
        "n_nonrec": n_nonrec,
        "n_rec": n_rec,
        "R_ril": R_ril,
        "r_meiotic": r_meiotic,
        "lod": lod
    })


backbone_pairwise = pd.DataFrame(backbone_pairwise)


print("BACKBONE PAIRWISE LINKAGE STATISTICS")
print("=" * 70)

print(f"Backbone markers          : {len(backbone_markers)}")
print(f"Marker pairs              : {len(backbone_pairwise):,}")

print("\nOverlap summary:")
display(
    backbone_pairwise["n_overlap"]
    .describe()
    .to_frame("n_overlap")
)

print("\nRIL recombinant fraction summary:")
display(
    backbone_pairwise["R_ril"]
    .describe()
    .to_frame("R_ril")
)

print("\nLOD summary:")
display(
    backbone_pairwise["lod"]
    .describe()
    .to_frame("lod")
)

### Cell 01.27 — Examine linkage evidence at several thresholds
* We should not choose a linkage-group threshold blindly. Let's see what the data look like.

In [ ]:
# Cell 01.27
# Explore candidate linkage thresholds.

print("PAIRWISE LINKAGE THRESHOLD SENSITIVITY")
print("=" * 78)

threshold_records = []

for min_overlap in [40, 50, 60]:
    for max_R in [0.20, 0.25, 0.30]:
        for min_lod in [3, 4, 5, 6]:

            mask = (
                (backbone_pairwise["n_overlap"] >= min_overlap)
                &
                (backbone_pairwise["R_ril"] <= max_R)
                &
                (backbone_pairwise["lod"] >= min_lod)
            )

            threshold_records.append({
                "min_overlap": min_overlap,
                "max_R_ril": max_R,
                "min_lod": min_lod,
                "linked_pairs": int(mask.sum())
            })


threshold_summary = pd.DataFrame(threshold_records)

display(threshold_summary)


print("\n30 STRONGEST PAIRWISE LINKAGES")
print("=" * 78)

display(
    backbone_pairwise
    .sort_values(
        ["lod", "R_ril", "n_overlap"],
        ascending=[False, True, False]
    )
    [
        [
            "marker_a",
            "marker_b",
            "n_overlap",
            "phase",
            "n_rec",
            "R_ril",
            "r_meiotic",
            "lod"
        ]
    ]
    .head(30)
)

### Cell 01.28 — Provisional linkage-group connectivity
* Let's test several reasonable graph criteria without yet declaring a final map.
* Each marker is a node; a sufficiently strong pairwise linkage is an edge. Connected components then give provisional linkage groups.

In [ ]:
# Cell 01.28
# Examine provisional linkage-group connectivity under several
# candidate pairwise linkage criteria.

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


def linkage_components(
    pairwise_df,
    markers,
    min_overlap=50,
    max_R=0.25,
    min_lod=4.0
):
    marker_index = {
        marker: i
        for i, marker in enumerate(markers)
    }

    rows = []
    cols = []

    linked = pairwise_df[
        (pairwise_df["n_overlap"] >= min_overlap)
        &
        (pairwise_df["R_ril"] <= max_R)
        &
        (pairwise_df["lod"] >= min_lod)
    ]

    for _, row in linked.iterrows():

        i = marker_index[row["marker_a"]]
        j = marker_index[row["marker_b"]]

        rows.extend([i, j])
        cols.extend([j, i])

    data = np.ones(len(rows), dtype=int)

    graph = csr_matrix(
        (
            data,
            (rows, cols)
        ),
        shape=(len(markers), len(markers))
    )

    n_components, labels = connected_components(
        graph,
        directed=False
    )

    result = pd.DataFrame({
        "marker": markers,
        "component": labels
    })

    sizes = (
        result["component"]
        .value_counts()
        .rename_axis("component")
        .reset_index(name="n_markers")
        .sort_values(
            "n_markers",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return linked, result, sizes


candidate_settings = [
    (40, 0.25, 3),
    (50, 0.25, 4),
    (50, 0.20, 4),
    (60, 0.25, 5),
    (60, 0.20, 5)
]


component_summary = []


for min_overlap, max_R, min_lod in candidate_settings:

    linked, membership, sizes = linkage_components(
        backbone_pairwise,
        backbone_markers,
        min_overlap=min_overlap,
        max_R=max_R,
        min_lod=min_lod
    )

    n_singletons = int(
        (sizes["n_markers"] == 1).sum()
    )

    n_multimarker = int(
        (sizes["n_markers"] >= 2).sum()
    )

    component_summary.append({
        "min_overlap": min_overlap,
        "max_R_ril": max_R,
        "min_lod": min_lod,
        "linked_pairs": len(linked),
        "components_total": len(sizes),
        "multi_marker_components": n_multimarker,
        "singletons": n_singletons,
        "largest_component": int(
            sizes["n_markers"].max()
        )
    })


component_summary = pd.DataFrame(
    component_summary
)


print("PROVISIONAL LINKAGE-GROUP CONNECTIVITY")
print("=" * 80)

display(component_summary)


# Also inspect the middle/default candidate in detail.
linked_default, membership_default, sizes_default = (
    linkage_components(
        backbone_pairwise,
        backbone_markers,
        min_overlap=50,
        max_R=0.25,
        min_lod=4
    )
)


print("\nCOMPONENT SIZES FOR:")
print("minimum overlap = 50")
print("maximum RIL recombinant fraction = 0.25")
print("minimum LOD = 4")

display(sizes_default)

### Cell 01.29 — Explore a broader linkage-group threshold grid

In [ ]:
# Cell 01.29
# Explore broader linkage criteria to identify the transition
# from fragmented marker groups to chromosome-scale connectivity.

expanded_settings = []

for min_overlap in [30, 40, 50]:
    for max_R in [0.25, 0.30, 0.325, 0.35]:
        for min_lod in [2.5, 3.0, 3.5, 4.0]:

            linked, membership, sizes = linkage_components(
                backbone_pairwise,
                backbone_markers,
                min_overlap=min_overlap,
                max_R=max_R,
                min_lod=min_lod
            )

            component_sizes = sizes["n_markers"]

            expanded_settings.append({
                "min_overlap": min_overlap,
                "max_R_ril": max_R,
                "min_lod": min_lod,
                "linked_pairs": len(linked),

                "components_total": len(sizes),

                "singletons":
                    int((component_sizes == 1).sum()),

                "groups_ge_2":
                    int((component_sizes >= 2).sum()),

                "groups_ge_3":
                    int((component_sizes >= 3).sum()),

                "groups_ge_5":
                    int((component_sizes >= 5).sum()),

                "groups_ge_8":
                    int((component_sizes >= 8).sum()),

                "largest_component":
                    int(component_sizes.max()),

                "second_largest":
                    (
                        int(component_sizes.iloc[1])
                        if len(component_sizes) > 1
                        else np.nan
                    )
            })


expanded_component_summary = pd.DataFrame(
    expanded_settings
)


print("EXPANDED LINKAGE-GROUP THRESHOLD SEARCH")
print("=" * 90)

display(
    expanded_component_summary.sort_values(
        [
            "groups_ge_3",
            "singletons",
            "largest_component"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
)

### Cell 01.30 — Focus on biologically promising solutions
* Now filter the threshold grid for solutions that produce roughly 15–30 substantial groups.

In [ ]:
# Cell 01.30
# Identify threshold combinations that are reasonably close
# to the expected chromosome-scale structure.

promising_settings = expanded_component_summary[
    (
        expanded_component_summary["groups_ge_3"]
        .between(15, 30)
    )
].copy()


print("PROMISING LINKAGE-GROUP SETTINGS")
print("=" * 90)

if promising_settings.empty:

    print(
        "No settings produced 15–30 groups "
        "containing at least 3 markers."
    )

else:

    promising_settings["largest_pct"] = (
        100
        * promising_settings["largest_component"]
        / len(backbone_markers)
    )

    display(
        promising_settings.sort_values(
            [
                "singletons",
                "largest_pct",
                "components_total"
            ],
            ascending=[
                True,
                True,
                True
            ]
        )
    )

### Cell 01.31 — Determine which markers remain isolated
* A useful question now is whether the singletons are systematically poorer-quality markers.
* We'll use an exploratory criterion of:
    - overlap ≥40
    - R_RIL < 0.30
    - LOD ≥3
* This is a standard-looking exploratory range for our dataset, not yet the final grouping rule.

In [ ]:
# Cell 01.31
# Examine isolated versus connected markers under a broader
# exploratory linkage criterion.

EXP_MIN_OVERLAP = 40
EXP_MAX_R = 0.30
EXP_MIN_LOD = 3.0


linked_exp, membership_exp, sizes_exp = linkage_components(
    backbone_pairwise,
    backbone_markers,
    min_overlap=EXP_MIN_OVERLAP,
    max_R=EXP_MAX_R,
    min_lod=EXP_MIN_LOD
)


# Add component size to every marker.
component_size_lookup = (
    membership_exp["component"]
    .value_counts()
    .to_dict()
)

membership_exp["component_size"] = (
    membership_exp["component"]
    .map(component_size_lookup)
)


marker_connectivity_qc = membership_exp.merge(
    marker_qc_92[
        [
            "marker",
            "n_observed",
            "missing_pct",
            "minor_class_freq",
            "segregation_p"
        ]
    ],
    on="marker",
    how="left"
)


print("EXPLORATORY CONNECTIVITY")
print("=" * 80)

print(f"Criterion: overlap >= {EXP_MIN_OVERLAP}")
print(f"           R <= {EXP_MAX_R}")
print(f"           LOD >= {EXP_MIN_LOD}")

print()
print(f"Linked edges       : {len(linked_exp)}")
print(f"Total components   : {len(sizes_exp)}")
print(
    "Singleton markers :",
    (sizes_exp["n_markers"] == 1).sum()
)


print("\nCOMPONENT SIZE DISTRIBUTION")
print("=" * 80)

display(sizes_exp)


print("\nISOLATED MARKERS")
print("=" * 80)

isolated_markers = marker_connectivity_qc[
    marker_connectivity_qc["component_size"] == 1
].copy()

display(
    isolated_markers
    .sort_values(
        ["missing_pct", "n_observed"],
        ascending=[False, True]
    )
)

### Cell 01.32 — Calculate marker network degree
* This is especially useful for identifying possible problematic "bridge" markers later.

In [ ]:
# Cell 01.32
# Calculate marker degree under the exploratory linkage criterion.

degree_counts = pd.concat(
    [
        linked_exp["marker_a"],
        linked_exp["marker_b"]
    ]
).value_counts()


marker_degree = pd.DataFrame({
    "marker": backbone_markers
})

marker_degree["degree"] = (
    marker_degree["marker"]
    .map(degree_counts)
    .fillna(0)
    .astype(int)
)


marker_degree = marker_degree.merge(
    marker_qc_92[
        [
            "marker",
            "n_observed",
            "missing_pct",
            "minor_class_freq",
            "segregation_p"
        ]
    ],
    on="marker",
    how="left"
)


print("MARKER LINKAGE DEGREE")
print("=" * 80)

display(
    marker_degree["degree"]
    .describe()
    .to_frame("degree")
)


print("\nMARKERS WITH HIGHEST NETWORK DEGREE")
print("=" * 80)

display(
    marker_degree
    .sort_values(
        ["degree", "missing_pct"],
        ascending=[False, True]
    )
    .head(30)
)


print("\nMARKERS WITH DEGREE = 0")
print("=" * 80)

display(
    marker_degree[
        marker_degree["degree"] == 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

### Cell 01.33 — Define the provisional 20 core linkage groups

In [ ]:
# Cell 01.33
# Define provisional core linkage groups using the stable threshold plateau.

CORE_MIN_OVERLAP = 40
CORE_MAX_R = 0.30
CORE_MIN_LOD = 3.5


linked_core, membership_core, sizes_core = linkage_components(
    backbone_pairwise,
    backbone_markers,
    min_overlap=CORE_MIN_OVERLAP,
    max_R=CORE_MAX_R,
    min_lod=CORE_MIN_LOD
)


# Add component sizes.
size_lookup = (
    membership_core["component"]
    .value_counts()
    .to_dict()
)

membership_core["component_size"] = (
    membership_core["component"]
    .map(size_lookup)
)


# Separate substantial groups, small components, and singletons.
core_components = sizes_core[
    sizes_core["n_markers"] >= 3
].copy()

two_marker_components = sizes_core[
    sizes_core["n_markers"] == 2
].copy()

singleton_components = sizes_core[
    sizes_core["n_markers"] == 1
].copy()


print("PROVISIONAL CORE LINKAGE GROUPS")
print("=" * 80)

print(f"Minimum overlap        : {CORE_MIN_OVERLAP}")
print(f"Maximum R_ril          : {CORE_MAX_R}")
print(f"Minimum LOD            : {CORE_MIN_LOD}")

print()
print(f"Linked marker pairs    : {len(linked_core)}")
print(f"Total components       : {len(sizes_core)}")
print(f"Components >=3 markers : {len(core_components)}")
print(f"Two-marker components  : {len(two_marker_components)}")
print(f"Singleton markers      : {len(singleton_components)}")


print("\nCORE COMPONENT SIZES")
print("=" * 80)

display(core_components.reset_index(drop=True))


print("\nTWO-MARKER COMPONENTS")
print("=" * 80)

display(two_marker_components.reset_index(drop=True))

### Cell 01.34 — Assign stable provisional group labels and show members
* Component numbers from graph algorithms are arbitrary, so let's rename the 20 substantial groups as pLG01–pLG20, ordered from largest to smallest.

In [ ]:
# Cell 01.34
# Assign provisional linkage-group labels and inspect marker membership.

core_component_ids = (
    core_components
    .sort_values(
        "n_markers",
        ascending=False
    )["component"]
    .tolist()
)


component_to_plg = {
    component: f"pLG{i:02d}"
    for i, component in enumerate(
        core_component_ids,
        start=1
    )
}


membership_core["provisional_group"] = (
    membership_core["component"]
    .map(component_to_plg)
)


core_group_membership = membership_core[
    membership_core["provisional_group"].notna()
].copy()


print("PROVISIONAL LINKAGE-GROUP MEMBERSHIP")
print("=" * 80)

group_summary = (
    core_group_membership
    .groupby("provisional_group")
    .agg(
        n_markers=("marker", "size")
    )
    .reset_index()
)

display(group_summary)


print("\nMARKERS IN EACH PROVISIONAL GROUP")
print("=" * 80)

for group in sorted(
    core_group_membership["provisional_group"].unique()
):

    markers = (
        core_group_membership.loc[
            core_group_membership["provisional_group"] == group,
            "marker"
        ]
        .sort_values()
        .tolist()
    )

    print(
        f"\n{group} | n={len(markers)}"
    )

    print(
        " | ".join(markers)
    )

### Cell 01.35 — Measure internal support for each provisional group
* A group formed by many mutually supported edges is more convincing than one held together by a single chain.

In [ ]:
# Cell 01.35
# Evaluate internal edge support within each provisional linkage group.

group_support_records = []


for group in sorted(
    core_group_membership["provisional_group"].unique()
):

    markers = set(
        core_group_membership.loc[
            core_group_membership["provisional_group"] == group,
            "marker"
        ]
    )

    edges = linked_core[
        linked_core["marker_a"].isin(markers)
        &
        linked_core["marker_b"].isin(markers)
    ].copy()

    n_markers = len(markers)

    max_possible_edges = (
        n_markers * (n_markers - 1) / 2
    )

    edge_density = (
        len(edges) / max_possible_edges
        if max_possible_edges > 0
        else np.nan
    )

    group_support_records.append({
        "provisional_group": group,
        "n_markers": n_markers,
        "n_supported_edges": len(edges),
        "max_possible_edges":
            int(max_possible_edges),
        "edge_density": edge_density,
        "median_R_ril":
            edges["R_ril"].median()
            if len(edges) > 0
            else np.nan,
        "max_R_ril":
            edges["R_ril"].max()
            if len(edges) > 0
            else np.nan,
        "median_lod":
            edges["lod"].median()
            if len(edges) > 0
            else np.nan,
        "min_lod":
            edges["lod"].min()
            if len(edges) > 0
            else np.nan
    })


core_group_support = pd.DataFrame(
    group_support_records
)


print("INTERNAL SUPPORT OF PROVISIONAL LINKAGE GROUPS")
print("=" * 90)

display(
    core_group_support.sort_values(
        [
            "edge_density",
            "n_markers"
        ],
        ascending=[
            True,
            False
        ]
    )
)

### Cell 01.36 — Find bridges and articulation markers
* This is the most important diagnostic in this block. An articulation marker is a marker whose removal splits a component into separate pieces.
* If one or two markers are holding two genuine chromosomes together, this analysis should expose them.

In [ ]:
# Cell 01.36
# Identify articulation markers and graph bridges within the
# provisional core linkage groups.

import networkx as nx


bridge_records = []
articulation_records = []


for group in sorted(
    core_group_membership["provisional_group"].unique()
):

    markers = (
        core_group_membership.loc[
            core_group_membership["provisional_group"] == group,
            "marker"
        ]
        .tolist()
    )

    group_edges = linked_core[
        linked_core["marker_a"].isin(markers)
        &
        linked_core["marker_b"].isin(markers)
    ]


    G = nx.Graph()

    G.add_nodes_from(markers)

    for _, row in group_edges.iterrows():

        G.add_edge(
            row["marker_a"],
            row["marker_b"],
            lod=row["lod"],
            R_ril=row["R_ril"],
            n_overlap=row["n_overlap"]
        )


    # Articulation markers
    arts = list(
        nx.articulation_points(G)
    )

    for marker in arts:

        articulation_records.append({
            "provisional_group": group,
            "marker": marker,
            "degree": G.degree(marker)
        })


    # Bridge edges
    bridges = list(
        nx.bridges(G)
    )

    for marker_a, marker_b in bridges:

        edge = G[marker_a][marker_b]

        bridge_records.append({
            "provisional_group": group,
            "marker_a": marker_a,
            "marker_b": marker_b,
            "n_overlap": edge["n_overlap"],
            "R_ril": edge["R_ril"],
            "lod": edge["lod"]
        })


articulation_table = pd.DataFrame(
    articulation_records
)

bridge_table = pd.DataFrame(
    bridge_records
)


print("GRAPH ROBUSTNESS OF CORE LINKAGE GROUPS")
print("=" * 90)

print(
    "Total articulation markers:",
    len(articulation_table)
)

print(
    "Total bridge edges:",
    len(bridge_table)
)


print("\nARTICULATION MARKERS")
print("=" * 90)

if articulation_table.empty:
    print("None")
else:
    display(
        articulation_table.sort_values(
            [
                "provisional_group",
                "degree"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


print("\nBRIDGE EDGES")
print("=" * 90)

if bridge_table.empty:
    print("None")
else:
    display(
        bridge_table.sort_values(
            [
                "R_ril",
                "lod"
            ],
            ascending=[
                False,
                True
            ]
        )
    )

### Cell 01.37 — Test how fragile each provisional group is
* This asks how much each pLG fragments when we progressively require stronger edges.

In [ ]:
# Cell 01.37
# Robustness analysis of provisional linkage groups.
#
# The 20 pLGs were defined at:
# overlap >= 40, R_ril <= 0.30, LOD >= 3.5
#
# Here we retain the SAME group membership and ask how each group
# behaves when progressively weaker edges are removed.

robustness_records = []

robustness_settings = [
    (0.30, 3.5),
    (0.275, 3.5),
    (0.25, 3.5),
    (0.30, 4.0),
    (0.25, 4.0),
    (0.25, 5.0)
]


for group in sorted(
    core_group_membership["provisional_group"].unique()
):

    group_markers = (
        core_group_membership.loc[
            core_group_membership["provisional_group"] == group,
            "marker"
        ]
        .tolist()
    )

    for max_R, min_lod in robustness_settings:

        edges = backbone_pairwise[
            backbone_pairwise["marker_a"].isin(group_markers)
            &
            backbone_pairwise["marker_b"].isin(group_markers)
            &
            (backbone_pairwise["n_overlap"] >= 40)
            &
            (backbone_pairwise["R_ril"] <= max_R)
            &
            (backbone_pairwise["lod"] >= min_lod)
        ].copy()

        G = nx.Graph()
        G.add_nodes_from(group_markers)

        G.add_edges_from(
            zip(
                edges["marker_a"],
                edges["marker_b"]
            )
        )

        component_sizes = sorted(
            [
                len(c)
                for c in nx.connected_components(G)
            ],
            reverse=True
        )

        robustness_records.append({
            "provisional_group": group,
            "n_markers": len(group_markers),
            "max_R_ril": max_R,
            "min_lod": min_lod,
            "n_edges": len(edges),
            "n_components": len(component_sizes),
            "largest_piece": component_sizes[0],
            "largest_piece_pct":
                100 * component_sizes[0] / len(group_markers),
            "singletons_after_pruning":
                sum(x == 1 for x in component_sizes)
        })


group_robustness = pd.DataFrame(
    robustness_records
)


print("PROVISIONAL GROUP ROBUSTNESS")
print("=" * 95)

display(
    group_robustness.sort_values(
        [
            "provisional_group",
            "min_lod",
            "max_R_ril"
        ]
    )
)

### Cell 01.38 — Document the de novo mapping decision

## De novo linkage-map reconstruction
* The historical `fxh_maps.xlsx` workbook is excluded from the analytical
workflow.
* The linkage map will be reconstructed de novo from the Flyer × Hartwig
genotype matrix. Historical marker positions or chromosome assignments will
not be used to define, split, merge, order, or validate linkage groups.
* The reconstruction will therefore rely on:
    1. genotype quality control;
    2. co-segregating marker bins;
    3. phase-independent pairwise recombination;
    4. linkage support measured by LOD;
    5. graph robustness and clustering;
    6. marker ordering within linkage groups;
    7. recombination-based map distances.
* Modern external genomic resources may be used later, independently, to assign
the reconstructed linkage groups to physical soybean chromosomes.

### Cell 01.39 — Match historical markers to the de novo pLGs
* We'll first use exact marker-name matching. No fuzzy matching yet.

In [ ]:
# Cell 01.39
# Audit the ORIGINAL genotype workbook headers.
#
# Important because pandas can automatically make duplicate
# column names unique by appending suffixes such as ".1".

from pathlib import Path
from openpyxl import load_workbook
from collections import Counter

project_path = Path(PROJECT_ROOT)

geno_files = list(
    project_path.rglob("fxh_genotypes.xlsx")
)

if len(geno_files) == 0:
    raise FileNotFoundError(
        "fxh_genotypes.xlsx was not found under PROJECT_ROOT."
    )

print("GENOTYPE FILE SEARCH")
print("=" * 85)

for path in geno_files:
    print(path)

GENO_FILE = geno_files[0]

print("\nUsing genotype file:")
print(GENO_FILE)


wb = load_workbook(
    GENO_FILE,
    read_only=True,
    data_only=True
)

ws = wb["fxh_genotypes"]


raw_headers = [
    cell.value
    for cell in next(
        ws.iter_rows(
            min_row=1,
            max_row=1
        )
    )
]


header_counts = Counter(raw_headers)

raw_duplicate_headers = {
    header: count
    for header, count in header_counts.items()
    if header is not None and count > 1
}


print("\nRAW EXCEL GENOTYPE HEADER AUDIT")
print("=" * 85)

print(
    "Number of raw columns      :",
    len(raw_headers)
)

print(
    "Unique nonmissing headers  :",
    len({
        h
        for h in raw_headers
        if h is not None
    })
)

print(
    "Raw duplicated header names:",
    len(raw_duplicate_headers)
)


print("\nDUPLICATED RAW HEADERS")
print("=" * 85)

if raw_duplicate_headers:

    for marker, count in sorted(
        raw_duplicate_headers.items()
    ):
        print(f"{marker}: {count}")

else:
    print("None")

### Cell 01.40 — permanent marker identity table
* This is important for reproducibility because the biological/raw name and computational ID should now be separate concepts.

In [ ]:
# Cell 01.40
# Construct stable unique computational IDs while preserving
# the original marker names exactly as they appear in Excel.

from collections import defaultdict

occurrence = defaultdict(int)

marker_identity_records = []


for excel_col, raw_name in enumerate(
    raw_headers,
    start=1
):

    # First Excel column is the RIL identifier.
    if excel_col == 1:
        continue

    raw_name = str(raw_name).strip()

    occurrence[raw_name] += 1

    copy_number = occurrence[raw_name]


    if header_counts[raw_name] > 1:

        internal_id = (
            f"{raw_name}__dup{copy_number}"
        )

    else:

        internal_id = raw_name


    marker_identity_records.append({
        "excel_column": excel_col,
        "raw_marker_name": raw_name,
        "occurrence": copy_number,
        "internal_marker_id": internal_id
    })


marker_identity = pd.DataFrame(
    marker_identity_records
)


print("MARKER IDENTITY TABLE")
print("=" * 90)

print(
    "Marker columns        :",
    len(marker_identity)
)

print(
    "Unique raw names      :",
    marker_identity[
        "raw_marker_name"
    ].nunique()
)

print(
    "Unique internal IDs   :",
    marker_identity[
        "internal_marker_id"
    ].nunique()
)


print("\nDUPLICATED RAW MARKER RECORDS")
print("=" * 90)

display(
    marker_identity[
        marker_identity[
            "raw_marker_name"
        ].duplicated(
            keep=False
        )
    ]
)

### Cell 01.41 — Characterize the 20 core groups without outside information
* Now we assess the provisional groups using only the genotype evidence.

In [ ]:
# Cell 01.41 — corrected
# Genotype-only summary of the 20 provisional linkage groups.
#
# Self-contained: recomputes group support directly from
# linked_core + core_group_membership.
#
# No historical map information is used.

import numpy as np
import pandas as pd
import networkx as nx


group_summary_records = []
articulation_records = []
bridge_records = []


for group in sorted(
    core_group_membership["provisional_group"].unique()
):

    # Markers belonging to this provisional linkage group
    markers = (
        core_group_membership.loc[
            core_group_membership["provisional_group"] == group,
            "marker"
        ]
        .tolist()
    )

    n_markers = len(markers)


    # Edges defining this group under the current core criterion
    group_edges = linked_core[
        linked_core["marker_a"].isin(markers)
        &
        linked_core["marker_b"].isin(markers)
    ].copy()


    n_edges = len(group_edges)

    max_possible_edges = (
        n_markers * (n_markers - 1) / 2
    )

    edge_density = (
        n_edges / max_possible_edges
        if max_possible_edges > 0
        else np.nan
    )


    # Build graph
    G = nx.Graph()

    G.add_nodes_from(markers)

    G.add_edges_from(
        zip(
            group_edges["marker_a"],
            group_edges["marker_b"]
        )
    )


    # Articulation markers
    articulation_points = list(
        nx.articulation_points(G)
    )

    for marker in articulation_points:

        articulation_records.append({
            "provisional_group": group,
            "marker": marker,
            "degree": G.degree(marker)
        })


    # Bridge edges
    group_bridges = list(
        nx.bridges(G)
    )

    for marker_a, marker_b in group_bridges:

        edge_match = group_edges[
            (
                (
                    group_edges["marker_a"] == marker_a
                )
                &
                (
                    group_edges["marker_b"] == marker_b
                )
            )
            |
            (
                (
                    group_edges["marker_a"] == marker_b
                )
                &
                (
                    group_edges["marker_b"] == marker_a
                )
            )
        ]

        if len(edge_match) > 0:

            row = edge_match.iloc[0]

            bridge_records.append({
                "provisional_group": group,
                "marker_a": marker_a,
                "marker_b": marker_b,
                "n_overlap": row["n_overlap"],
                "R_ril": row["R_ril"],
                "lod": row["lod"]
            })


    # Group-level summary
    group_summary_records.append({
        "provisional_group": group,
        "n_markers": n_markers,
        "n_supported_edges": n_edges,
        "max_possible_edges": int(max_possible_edges),
        "edge_density": edge_density,

        "median_R_ril":
            group_edges["R_ril"].median()
            if n_edges > 0
            else np.nan,

        "max_R_ril":
            group_edges["R_ril"].max()
            if n_edges > 0
            else np.nan,

        "median_lod":
            group_edges["lod"].median()
            if n_edges > 0
            else np.nan,

        "min_lod":
            group_edges["lod"].min()
            if n_edges > 0
            else np.nan,

        "n_articulation_markers":
            len(articulation_points),

        "n_bridge_edges":
            len(group_bridges)
    })


# Create clean output objects
group_genetic_summary = pd.DataFrame(
    group_summary_records
)

articulation_df = pd.DataFrame(
    articulation_records
)

bridge_df = pd.DataFrame(
    bridge_records
)


print("GENOTYPE-ONLY CORE LINKAGE-GROUP SUMMARY")
print("=" * 115)

display(
    group_genetic_summary.sort_values(
        [
            "edge_density",
            "n_markers"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print("\nSUMMARY")
print("=" * 115)

print(
    "Provisional linkage groups :",
    len(group_genetic_summary)
)

print(
    "Total markers in groups    :",
    group_genetic_summary[
        "n_markers"
    ].sum()
)

print(
    "Total articulation markers :",
    len(articulation_df)
)

print(
    "Total bridge edges         :",
    len(bridge_df)
)

### Cell 01.42 — Assess subgroup stability across thresholds
* This replaces the historical chromosome-purity analysis.
* We want to know whether each pLG represents a stable genotype-defined unit or whether it repeatedly fragments when thresholds change.

In [ ]:
# Cell 01.42
# Summarize component stability across several reasonable
# linkage thresholds without reference to any historical map.

stability_settings = [
    (40, 0.30, 3.5),
    (40, 0.275, 3.5),
    (40, 0.25, 3.5),
    (40, 0.30, 4.0),
    (40, 0.25, 4.0),
    (40, 0.25, 5.0)
]


stability_records = []


for group in sorted(
    core_group_membership[
        "provisional_group"
    ].unique()
):

    group_markers = (
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


    for min_overlap, max_R, min_lod in stability_settings:

        edges = backbone_pairwise[
            backbone_pairwise[
                "marker_a"
            ].isin(group_markers)
            &
            backbone_pairwise[
                "marker_b"
            ].isin(group_markers)
            &
            (
                backbone_pairwise[
                    "n_overlap"
                ] >= min_overlap
            )
            &
            (
                backbone_pairwise[
                    "R_ril"
                ] <= max_R
            )
            &
            (
                backbone_pairwise[
                    "lod"
                ] >= min_lod
            )
        ]


        G = nx.Graph()

        G.add_nodes_from(
            group_markers
        )

        G.add_edges_from(
            zip(
                edges["marker_a"],
                edges["marker_b"]
            )
        )


        pieces = sorted(
            [
                len(c)
                for c in nx.connected_components(G)
            ],
            reverse=True
        )


        stability_records.append({
            "provisional_group": group,
            "n_markers": len(group_markers),
            "min_overlap": min_overlap,
            "max_R_ril": max_R,
            "min_lod": min_lod,
            "n_components": len(pieces),
            "largest_component": pieces[0],
            "largest_component_pct":
                100 * pieces[0] /
                len(group_markers),
            "n_singletons":
                sum(
                    size == 1
                    for size in pieces
                )
        })


group_stability = pd.DataFrame(
    stability_records
)


print("GENOTYPE-ONLY LINKAGE-GROUP STABILITY")
print("=" * 105)

display(
    group_stability.sort_values(
        [
            "provisional_group",
            "min_lod",
            "max_R_ril"
        ]
    )
)

### Cell 01.43 — Quantify the weakest connection required to hold each group together
* This is more useful now than chromosome purity.
* For each provisional linkage group, we'll compute a maximum spanning tree using LOD, which exposes the minimum-support edges needed to connect the whole component.

In [ ]:
# Cell 01.43
# Determine the essential connecting edges in each provisional group.
#
# A maximum spanning tree retains the strongest set of edges that
# connects all markers in a group.
#
# Weak edges in this tree are especially important because removing
# one can divide the group.

mst_records = []


for group in sorted(
    core_group_membership[
        "provisional_group"
    ].unique()
):

    markers = (
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


    edges = linked_core[
        linked_core["marker_a"].isin(markers)
        &
        linked_core["marker_b"].isin(markers)
    ].copy()


    G = nx.Graph()

    G.add_nodes_from(markers)


    for _, row in edges.iterrows():

        G.add_edge(
            row["marker_a"],
            row["marker_b"],
            lod=row["lod"],
            R_ril=row["R_ril"],
            n_overlap=row["n_overlap"]
        )


    if len(markers) <= 1:
        continue


    T = nx.maximum_spanning_tree(
        G,
        weight="lod"
    )


    for marker_a, marker_b, data in T.edges(
        data=True
    ):

        mst_records.append({
            "provisional_group": group,
            "marker_a": marker_a,
            "marker_b": marker_b,
            "n_overlap": data[
                "n_overlap"
            ],
            "R_ril": data[
                "R_ril"
            ],
            "lod": data[
                "lod"
            ]
        })


mst_edges = pd.DataFrame(
    mst_records
)


print("MAXIMUM-SPANNING-TREE CONNECTIONS")
print("=" * 105)


display(
    mst_edges.sort_values(
        [
            "provisional_group",
            "lod"
        ],
        ascending=[
            True,
            True
        ]
    )
)

### Cell 01.44 — Weakest essential edge per provisional group

In [ ]:
# Cell 01.44
# Summarize the weakest essential MST edge for each pLG.

mst_group_summary = (
    mst_edges
    .groupby(
        "provisional_group"
    )
    .agg(
        n_tree_edges=(
            "lod",
            "size"
        ),
        weakest_tree_lod=(
            "lod",
            "min"
        ),
        median_tree_lod=(
            "lod",
            "median"
        ),
        strongest_tree_lod=(
            "lod",
            "max"
        ),
        largest_tree_R=(
            "R_ril",
            "max"
        ),
        median_tree_R=(
            "R_ril",
            "median"
        ),
        minimum_tree_overlap=(
            "n_overlap",
            "min"
        )
    )
    .reset_index()
)


print("ESSENTIAL CONNECTION STRENGTH BY LINKAGE GROUP")
print("=" * 105)

display(
    mst_group_summary.sort_values(
        [
            "weakest_tree_lod",
            "largest_tree_R"
        ],
        ascending=[
            True,
            False
        ]
    )
)

### Cell 01.45 — identify weakest essential MST edge in each linkage group

In [ ]:
# Cell 01.45
# Identify the weakest essential MST edge for each provisional linkage group.
#
# These are the edges most likely to cause artificial group fusion.

weakest_mst_edges = (
    mst_edges
    .sort_values(
        [
            "provisional_group",
            "lod",
            "R_ril"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .groupby(
        "provisional_group",
        as_index=False
    )
    .first()
)


print("WEAKEST ESSENTIAL MST EDGE PER LINKAGE GROUP")
print("=" * 115)

display(
    weakest_mst_edges[
        [
            "provisional_group",
            "marker_a",
            "marker_b",
            "n_overlap",
            "R_ril",
            "lod"
        ]
    ]
    .sort_values(
        [
            "lod",
            "R_ril"
        ],
        ascending=[
            True,
            False
        ]
    )
)

### Cell 01.46 — test how each group breaks if its weakest MST edge is removed
* This is a controlled robustness test.

In [ ]:
# Cell 01.46
# Remove the single weakest MST edge from each provisional linkage group
# and observe whether the group splits into meaningful fragments.

weak_edge_split_records = []


for group in sorted(
    core_group_membership["provisional_group"].unique()
):

    markers = (
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


    group_edges = linked_core[
        linked_core["marker_a"].isin(markers)
        &
        linked_core["marker_b"].isin(markers)
    ].copy()


    G = nx.Graph()

    G.add_nodes_from(markers)

    G.add_edges_from(
        zip(
            group_edges["marker_a"],
            group_edges["marker_b"]
        )
    )


    weakest_row = weakest_mst_edges[
        weakest_mst_edges[
            "provisional_group"
        ] == group
    ]


    if len(weakest_row) == 0:

        pieces = sorted(
            nx.connected_components(G),
            key=len,
            reverse=True
        )

    else:

        weakest_row = weakest_row.iloc[0]

        a = weakest_row["marker_a"]
        b = weakest_row["marker_b"]

        if G.has_edge(a, b):
            G.remove_edge(a, b)

        pieces = sorted(
            nx.connected_components(G),
            key=len,
            reverse=True
        )


    sizes = [
        len(piece)
        for piece in pieces
    ]


    weak_edge_split_records.append({
        "provisional_group": group,
        "n_markers": len(markers),
        "n_components_after_removal": len(pieces),
        "largest_component": max(sizes),
        "largest_component_pct":
            100 * max(sizes) / len(markers),
        "second_component":
            sizes[1]
            if len(sizes) > 1
            else 0,
        "n_singletons":
            sum(
                size == 1
                for size in sizes
            )
    })


weak_edge_split_summary = pd.DataFrame(
    weak_edge_split_records
)


print("ROBUSTNESS AFTER REMOVING WEAKEST MST EDGE")
print("=" * 115)

display(
    weak_edge_split_summary.sort_values(
        [
            "largest_component_pct",
            "n_markers"
        ],
        ascending=[
            True,
            False
        ]
    )
)

### Cell 01.47 — inspect the actual fragments created by weak-edge removal

In [ ]:
# Cell 01.47
# Show marker membership of groups that split substantially
# after removal of their weakest essential edge.

substantial_split_groups = (
    weak_edge_split_summary[
        (
            weak_edge_split_summary[
                "n_components_after_removal"
            ] > 1
        )
        &
        (
            weak_edge_split_summary[
                "second_component"
            ] >= 2
        )
    ]
    ["provisional_group"]
    .tolist()
)


print("GROUPS WITH SUBSTANTIAL WEAK-EDGE SPLITS")
print("=" * 115)

print(substantial_split_groups)


weak_split_membership_records = []


for group in substantial_split_groups:

    markers = (
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


    group_edges = linked_core[
        linked_core["marker_a"].isin(markers)
        &
        linked_core["marker_b"].isin(markers)
    ].copy()


    G = nx.Graph()

    G.add_nodes_from(markers)

    G.add_edges_from(
        zip(
            group_edges["marker_a"],
            group_edges["marker_b"]
        )
    )


    weakest_row = weakest_mst_edges[
        weakest_mst_edges[
            "provisional_group"
        ] == group
    ].iloc[0]


    a = weakest_row["marker_a"]
    b = weakest_row["marker_b"]


    if G.has_edge(a, b):
        G.remove_edge(a, b)


    pieces = sorted(
        nx.connected_components(G),
        key=len,
        reverse=True
    )


    for i, piece in enumerate(
        pieces,
        start=1
    ):

        piece_id = (
            f"{group}_W{i:02d}"
        )

        for marker in sorted(piece):

            weak_split_membership_records.append({
                "provisional_group": group,
                "weak_split_group": piece_id,
                "split_size": len(piece),
                "marker": marker
            })


weak_split_membership = pd.DataFrame(
    weak_split_membership_records
)


for group in substantial_split_groups:

    print("\n" + "=" * 115)
    print(group)
    print("=" * 115)

    display(
        weak_split_membership[
            weak_split_membership[
                "provisional_group"
            ] == group
        ]
        .sort_values(
            [
                "weak_split_group",
                "marker"
            ]
        )
    )

### Cell 01.48 — build a provisional robustness classification
* This does not finalize the linkage groups. It simply tells us which groups are currently strong, intermediate, or fragile.

In [ ]:
# Cell 01.48
# Combine several genotype-only robustness metrics
# into a provisional diagnostic classification.

robustness_table = (
    group_genetic_summary
    .merge(
        mst_group_summary,
        on="provisional_group",
        how="left"
    )
    .merge(
        weak_edge_split_summary,
        on=[
            "provisional_group",
            "n_markers"
        ],
        how="left"
    )
)


def classify_group(row):

    # Strong:
    # dense graph, strong weakest MST edge,
    # and remains mostly intact after weakest-edge removal.
    if (
        row["edge_density"] >= 0.50
        and
        row["weakest_tree_lod"] >= 5.0
        and
        row["largest_component_pct"] >= 85
    ):
        return "strong"

    # Fragile:
    # weak essential connection or major fragmentation.
    elif (
        row["weakest_tree_lod"] < 4.0
        or
        row["largest_component_pct"] < 70
    ):
        return "fragile"

    else:
        return "intermediate"


robustness_table[
    "robustness_class"
] = robustness_table.apply(
    classify_group,
    axis=1
)


print("PROVISIONAL GENOTYPE-ONLY LINKAGE-GROUP ROBUSTNESS")
print("=" * 120)

display(
    robustness_table[
        [
            "provisional_group",
            "n_markers",
            "edge_density",
            "median_lod",
            "min_lod",
            "n_articulation_markers",
            "n_bridge_edges",
            "weakest_tree_lod",
            "largest_tree_R",
            "largest_component_pct",
            "second_component",
            "robustness_class"
        ]
    ]
    .sort_values(
        [
            "robustness_class",
            "weakest_tree_lod"
        ]
    )
)


print("\nROBUSTNESS CLASS COUNTS")
print("=" * 120)

print(
    robustness_table[
        "robustness_class"
    ].value_counts()
)

### Cell 01.49 — threshold-stable subgroup membership

In [ ]:
# Cell 01.49
# Examine whether potentially fragile linkage groups split into
# reproducible subgroups across nearby linkage thresholds.
#
# Focus only on groups with substantial weak-edge splits.

fragile_focus_groups = [
    "pLG01",
    "pLG03",
    "pLG04",
    "pLG12"
]

refinement_settings = [
    (40, 0.30, 3.5),
    (40, 0.275, 3.5),
    (40, 0.25, 3.5),
    (40, 0.30, 4.0),
    (40, 0.275, 4.0),
    (40, 0.25, 4.0),
    (40, 0.25, 5.0)
]

refinement_records = []


for group in fragile_focus_groups:

    markers = (
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )

    for min_overlap, max_R, min_lod in refinement_settings:

        edges = backbone_pairwise[
            backbone_pairwise["marker_a"].isin(markers)
            &
            backbone_pairwise["marker_b"].isin(markers)
            &
            (
                backbone_pairwise["n_overlap"]
                >= min_overlap
            )
            &
            (
                backbone_pairwise["R_ril"]
                <= max_R
            )
            &
            (
                backbone_pairwise["lod"]
                >= min_lod
            )
        ].copy()


        G = nx.Graph()

        G.add_nodes_from(markers)

        G.add_edges_from(
            zip(
                edges["marker_a"],
                edges["marker_b"]
            )
        )


        components = sorted(
            nx.connected_components(G),
            key=len,
            reverse=True
        )


        for comp_idx, component in enumerate(
            components,
            start=1
        ):

            component_id = (
                f"{group}_"
                f"R{max_R:.3f}_"
                f"L{min_lod:.1f}_"
                f"C{comp_idx:02d}"
            )

            for marker in sorted(component):

                refinement_records.append({
                    "provisional_group": group,
                    "min_overlap": min_overlap,
                    "max_R_ril": max_R,
                    "min_lod": min_lod,
                    "component_number": comp_idx,
                    "component_id": component_id,
                    "component_size": len(component),
                    "marker": marker
                })


refinement_membership = pd.DataFrame(
    refinement_records
)


print("THRESHOLD-BASED SUBGROUP MEMBERSHIP CREATED")
print("=" * 105)

print(
    "Rows:",
    len(refinement_membership)
)

print(
    "Groups:",
    refinement_membership[
        "provisional_group"
    ].nunique()
)

print(
    "Threshold settings:",
    refinement_membership[
        [
            "min_overlap",
            "max_R_ril",
            "min_lod"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

### Cell 01.50 — component sizes across thresholds

In [ ]:
# Cell 01.50
# Summarize how each fragile group fragments across thresholds.

refinement_size_summary = (
    refinement_membership[
        [
            "provisional_group",
            "min_overlap",
            "max_R_ril",
            "min_lod",
            "component_number",
            "component_size"
        ]
    ]
    .drop_duplicates()
)


print("FRAGILE-GROUP COMPONENT SIZES ACROSS THRESHOLDS")
print("=" * 115)


for group in fragile_focus_groups:

    print("\n" + group)
    print("-" * 115)

    temp = (
        refinement_size_summary[
            refinement_size_summary[
                "provisional_group"
            ] == group
        ]
        .sort_values(
            [
                "min_lod",
                "max_R_ril",
                "component_number"
            ]
        )
    )

    display(temp)

### Cell 01.51 — pairwise co-membership stability
* This is the key cell. For every pair of markers within the four fragile groups, we count how often they remain in the same connected component across the seven threshold settings.

In [ ]:
# Cell 01.51
# Calculate pairwise subgroup stability.
#
# For each marker pair, determine the proportion of threshold
# settings in which both markers remain in the same component.

from itertools import combinations

pair_stability_records = []


for group in fragile_focus_groups:

    markers = sorted(
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )

    settings = (
        refinement_membership.loc[
            refinement_membership[
                "provisional_group"
            ] == group,
            [
                "min_overlap",
                "max_R_ril",
                "min_lod"
            ]
        ]
        .drop_duplicates()
    )


    for marker_a, marker_b in combinations(
        markers,
        2
    ):

        same_component_count = 0
        total_settings = 0


        for _, setting in settings.iterrows():

            temp = refinement_membership[
                (
                    refinement_membership[
                        "provisional_group"
                    ] == group
                )
                &
                (
                    refinement_membership[
                        "min_overlap"
                    ] == setting[
                        "min_overlap"
                    ]
                )
                &
                (
                    refinement_membership[
                        "max_R_ril"
                    ] == setting[
                        "max_R_ril"
                    ]
                )
                &
                (
                    refinement_membership[
                        "min_lod"
                    ] == setting[
                        "min_lod"
                    ]
                )
            ]


            comp_a = temp.loc[
                temp["marker"] == marker_a,
                "component_number"
            ]

            comp_b = temp.loc[
                temp["marker"] == marker_b,
                "component_number"
            ]


            if (
                len(comp_a) == 1
                and
                len(comp_b) == 1
            ):

                total_settings += 1

                if (
                    comp_a.iloc[0]
                    ==
                    comp_b.iloc[0]
                ):
                    same_component_count += 1


        stability = (
            same_component_count /
            total_settings
            if total_settings > 0
            else np.nan
        )


        pair_stability_records.append({
            "provisional_group": group,
            "marker_a": marker_a,
            "marker_b": marker_b,
            "same_component_count":
                same_component_count,
            "total_settings":
                total_settings,
            "co_membership_stability":
                stability
        })


pair_stability = pd.DataFrame(
    pair_stability_records
)


print("PAIRWISE CO-MEMBERSHIP STABILITY")
print("=" * 110)

display(
    pair_stability.sort_values(
        [
            "provisional_group",
            "co_membership_stability"
        ]
    )
    .head(100)
)

### Cell 01.52 — marker-level stability within each fragile group
* Now calculate how consistently each marker remains associated with the rest of its group.

In [ ]:
# Cell 01.52
# Marker-level cohesion score.
#
# For each marker, calculate its average co-membership
# stability with every other marker in its provisional group.

marker_stability_a = (
    pair_stability[
        [
            "provisional_group",
            "marker_a",
            "co_membership_stability"
        ]
    ]
    .rename(
        columns={
            "marker_a": "marker"
        }
    )
)

marker_stability_b = (
    pair_stability[
        [
            "provisional_group",
            "marker_b",
            "co_membership_stability"
        ]
    ]
    .rename(
        columns={
            "marker_b": "marker"
        }
    )
)


marker_stability_long = pd.concat(
    [
        marker_stability_a,
        marker_stability_b
    ],
    ignore_index=True
)


marker_stability_summary = (
    marker_stability_long
    .groupby(
        [
            "provisional_group",
            "marker"
        ],
        as_index=False
    )
    .agg(
        mean_co_membership=(
            "co_membership_stability",
            "mean"
        ),
        median_co_membership=(
            "co_membership_stability",
            "median"
        ),
        minimum_co_membership=(
            "co_membership_stability",
            "min"
        )
    )
)


print("MARKER-LEVEL GROUP COHESION")
print("=" * 110)


for group in fragile_focus_groups:

    print("\n" + group)
    print("-" * 110)

    display(
        marker_stability_summary[
            marker_stability_summary[
                "provisional_group"
            ] == group
        ]
        .sort_values(
            "mean_co_membership"
        )
    )

### Cell 01.53 — build consensus stability graphs for the four fragile groups
* Here, two markers are connected if they remain in the same component in at least 70% of the tested threshold settings.

In [ ]:
# Cell 01.53
# Build consensus subgroup graphs from pairwise co-membership stability.
#
# Two markers are connected when they remain together in at least
# 70% of the tested linkage-threshold settings.
#
# This is a consensus criterion, not a single arbitrary R/LOD cutoff.

CONSENSUS_STABILITY = 0.70

consensus_membership_records = []
consensus_summary_records = []


for group in fragile_focus_groups:

    markers = sorted(
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ].tolist()
    )


    pair_df = pair_stability[
        pair_stability[
            "provisional_group"
        ] == group
    ].copy()


    stable_pairs = pair_df[
        pair_df[
            "co_membership_stability"
        ] >= CONSENSUS_STABILITY
    ]


    G = nx.Graph()

    G.add_nodes_from(markers)

    G.add_edges_from(
        zip(
            stable_pairs["marker_a"],
            stable_pairs["marker_b"]
        )
    )


    components = sorted(
        nx.connected_components(G),
        key=len,
        reverse=True
    )


    sizes = [
        len(component)
        for component in components
    ]


    consensus_summary_records.append({
        "provisional_group": group,
        "n_original_markers": len(markers),
        "n_consensus_components": len(components),
        "largest_component": sizes[0],
        "second_component":
            sizes[1]
            if len(sizes) > 1
            else 0,
        "n_singletons":
            sum(
                size == 1
                for size in sizes
            )
    })


    for idx, component in enumerate(
        components,
        start=1
    ):

        subgroup = (
            f"{group}_C{idx:02d}"
        )

        for marker in sorted(component):

            consensus_membership_records.append({
                "provisional_group": group,
                "consensus_group": subgroup,
                "consensus_size": len(component),
                "marker": marker
            })


consensus_membership = pd.DataFrame(
    consensus_membership_records
)

consensus_summary = pd.DataFrame(
    consensus_summary_records
)


print("CONSENSUS STABILITY COMPONENT SUMMARY")
print("=" * 105)

display(consensus_summary)

### Cell 01.54 — inspect the consensus subgroup memberships

In [ ]:
# Cell 01.54
# Display consensus subgroup membership for the fragile groups.

print(
    f"CONSENSUS GROUPS USING "
    f"CO-MEMBERSHIP >= {CONSENSUS_STABILITY:.2f}"
)
print("=" * 110)


for group in fragile_focus_groups:

    print("\n" + group)
    print("-" * 110)

    temp = (
        consensus_membership[
            consensus_membership[
                "provisional_group"
            ] == group
        ]
        .sort_values(
            [
                "consensus_group",
                "marker"
            ]
        )
    )

    display(temp)

### Cell 01.55 — quantify direct linkage between consensus subgroups
* This is critical before accepting any split. We want to know whether two candidate subgroups are still directly connected by several convincing marker pairs or only by one marginal edge.

In [ ]:
# Cell 01.55
# Quantify direct pairwise linkage between consensus subgroups.
#
# For every pair of consensus subgroups within the same provisional LG,
# summarize the strongest available genotype linkage between them.

from itertools import combinations

cross_consensus_records = []


for group in fragile_focus_groups:

    group_membership = (
        consensus_membership[
            consensus_membership[
                "provisional_group"
            ] == group
        ]
    )

    subgroup_names = sorted(
        group_membership[
            "consensus_group"
        ].unique()
    )


    for sg_a, sg_b in combinations(
        subgroup_names,
        2
    ):

        markers_a = set(
            group_membership.loc[
                group_membership[
                    "consensus_group"
                ] == sg_a,
                "marker"
            ]
        )

        markers_b = set(
            group_membership.loc[
                group_membership[
                    "consensus_group"
                ] == sg_b,
                "marker"
            ]
        )


        cross_pairs = backbone_pairwise[
            (
                backbone_pairwise[
                    "marker_a"
                ].isin(markers_a)
                &
                backbone_pairwise[
                    "marker_b"
                ].isin(markers_b)
            )
            |
            (
                backbone_pairwise[
                    "marker_a"
                ].isin(markers_b)
                &
                backbone_pairwise[
                    "marker_b"
                ].isin(markers_a)
            )
        ].copy()


        if len(cross_pairs) == 0:

            cross_consensus_records.append({
                "provisional_group": group,
                "subgroup_a": sg_a,
                "subgroup_b": sg_b,
                "n_pairwise_comparisons": 0,
                "best_R_ril": np.nan,
                "best_lod": np.nan,
                "median_R_ril": np.nan,
                "median_lod": np.nan,
                "n_core_links": 0,
                "n_strict_links": 0
            })

            continue


        n_core = (
            (
                cross_pairs["n_overlap"] >= 40
            )
            &
            (
                cross_pairs["R_ril"] <= 0.30
            )
            &
            (
                cross_pairs["lod"] >= 3.5
            )
        ).sum()


        n_strict = (
            (
                cross_pairs["n_overlap"] >= 40
            )
            &
            (
                cross_pairs["R_ril"] <= 0.25
            )
            &
            (
                cross_pairs["lod"] >= 3.5
            )
        ).sum()


        cross_consensus_records.append({
            "provisional_group": group,
            "subgroup_a": sg_a,
            "subgroup_b": sg_b,
            "n_pairwise_comparisons":
                len(cross_pairs),

            "best_R_ril":
                cross_pairs["R_ril"].min(),

            "best_lod":
                cross_pairs["lod"].max(),

            "median_R_ril":
                cross_pairs["R_ril"].median(),

            "median_lod":
                cross_pairs["lod"].median(),

            "n_core_links":
                int(n_core),

            "n_strict_links":
                int(n_strict)
        })


cross_consensus_summary = pd.DataFrame(
    cross_consensus_records
)


print("DIRECT LINKAGE BETWEEN CONSENSUS SUBGROUPS")
print("=" * 120)

display(
    cross_consensus_summary.sort_values(
        [
            "provisional_group",
            "best_R_ril",
            "best_lod"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

### Cell 01.56 — show strongest individual cross-subgroup marker pairs
* This is more informative than only summary statistics.

In [ ]:
# Cell 01.56
# Show the strongest individual marker-pair connections
# between consensus subgroups.

cross_pair_records = []


for group in fragile_focus_groups:

    group_membership = (
        consensus_membership[
            consensus_membership[
                "provisional_group"
            ] == group
        ]
    )


    marker_to_consensus = dict(
        zip(
            group_membership["marker"],
            group_membership["consensus_group"]
        )
    )


    group_markers = set(
        group_membership["marker"]
    )


    pairs = backbone_pairwise[
        backbone_pairwise[
            "marker_a"
        ].isin(group_markers)
        &
        backbone_pairwise[
            "marker_b"
        ].isin(group_markers)
    ].copy()


    pairs["consensus_a"] = (
        pairs["marker_a"]
        .map(marker_to_consensus)
    )

    pairs["consensus_b"] = (
        pairs["marker_b"]
        .map(marker_to_consensus)
    )


    cross = pairs[
        pairs["consensus_a"]
        !=
        pairs["consensus_b"]
    ].copy()


    if len(cross) == 0:
        continue


    cross["passes_core_rule"] = (
        (cross["n_overlap"] >= 40)
        &
        (cross["R_ril"] <= 0.30)
        &
        (cross["lod"] >= 3.5)
    )

    cross["passes_strict_rule"] = (
        (cross["n_overlap"] >= 40)
        &
        (cross["R_ril"] <= 0.25)
        &
        (cross["lod"] >= 3.5)
    )


    cross["provisional_group"] = group


    cross_pair_records.append(
        cross[
            [
                "provisional_group",
                "marker_a",
                "consensus_a",
                "marker_b",
                "consensus_b",
                "n_overlap",
                "R_ril",
                "r_meiotic",
                "lod",
                "passes_core_rule",
                "passes_strict_rule"
            ]
        ]
    )


cross_consensus_pairs = pd.concat(
    cross_pair_records,
    ignore_index=True
)


print("STRONGEST CROSS-CONSENSUS MARKER PAIRS")
print("=" * 125)


for group in fragile_focus_groups:

    print("\n" + group)
    print("-" * 125)

    temp = (
        cross_consensus_pairs[
            cross_consensus_pairs[
                "provisional_group"
            ] == group
        ]
        .sort_values(
            [
                "R_ril",
                "lod"
            ],
            ascending=[
                True,
                False
            ]
        )
        .head(20)
    )

    display(temp)

### Cell 01.57 — internal support of each consensus subgroup

In [ ]:
# Cell 01.57
# Quantify internal pairwise support within each consensus subgroup.

consensus_internal_records = []


for group in fragile_focus_groups:

    group_membership = consensus_membership[
        consensus_membership[
            "provisional_group"
        ] == group
    ]


    for subgroup in sorted(
        group_membership[
            "consensus_group"
        ].unique()
    ):

        markers = (
            group_membership.loc[
                group_membership[
                    "consensus_group"
                ] == subgroup,
                "marker"
            ]
            .tolist()
        )

        n_markers = len(markers)


        pairs = backbone_pairwise[
            backbone_pairwise[
                "marker_a"
            ].isin(markers)
            &
            backbone_pairwise[
                "marker_b"
            ].isin(markers)
        ].copy()


        max_possible = (
            n_markers * (n_markers - 1) / 2
        )


        core_mask = (
            (pairs["n_overlap"] >= 40)
            &
            (pairs["R_ril"] <= 0.30)
            &
            (pairs["lod"] >= 3.5)
        )


        strict_mask = (
            (pairs["n_overlap"] >= 40)
            &
            (pairs["R_ril"] <= 0.25)
            &
            (pairs["lod"] >= 3.5)
        )


        consensus_internal_records.append({
            "provisional_group": group,
            "consensus_group": subgroup,
            "n_markers": n_markers,
            "n_possible_pairs": int(max_possible),
            "n_observed_pairs": len(pairs),
            "n_core_links": int(core_mask.sum()),
            "n_strict_links": int(strict_mask.sum()),

            "core_edge_density":
                (
                    core_mask.sum() / max_possible
                    if max_possible > 0
                    else np.nan
                ),

            "strict_edge_density":
                (
                    strict_mask.sum() / max_possible
                    if max_possible > 0
                    else np.nan
                ),

            "median_R_ril":
                (
                    pairs["R_ril"].median()
                    if len(pairs) > 0
                    else np.nan
                ),

            "min_R_ril":
                (
                    pairs["R_ril"].min()
                    if len(pairs) > 0
                    else np.nan
                ),

            "median_lod":
                (
                    pairs["lod"].median()
                    if len(pairs) > 0
                    else np.nan
                ),

            "max_lod":
                (
                    pairs["lod"].max()
                    if len(pairs) > 0
                    else np.nan
                )
        })


consensus_internal_support = pd.DataFrame(
    consensus_internal_records
)


print("INTERNAL SUPPORT OF CONSENSUS SUBGROUPS")
print("=" * 120)

display(
    consensus_internal_support.sort_values(
        [
            "provisional_group",
            "consensus_group"
        ]
    )
)

### Cell 01.58 — compare internal versus cross-subgroup linkage
* This gives us a quantitative "separation" measure.

In [ ]:
# Cell 01.58
# Compare internal subgroup support with the strongest
# cross-subgroup linkage evidence.

separation_records = []


for group in fragile_focus_groups:

    subgroup_df = consensus_internal_support[
        consensus_internal_support[
            "provisional_group"
        ] == group
    ]


    for _, row in subgroup_df.iterrows():

        subgroup = row[
            "consensus_group"
        ]


        cross = cross_consensus_summary[
            (
                cross_consensus_summary[
                    "provisional_group"
                ] == group
            )
            &
            (
                (
                    cross_consensus_summary[
                        "subgroup_a"
                    ] == subgroup
                )
                |
                (
                    cross_consensus_summary[
                        "subgroup_b"
                    ] == subgroup
                )
            )
        ]


        strongest_cross_lod = (
            cross["best_lod"].max()
            if len(cross) > 0
            else np.nan
        )


        best_cross_R = (
            cross["best_R_ril"].min()
            if len(cross) > 0
            else np.nan
        )


        total_cross_core_links = (
            cross["n_core_links"].sum()
            if len(cross) > 0
            else 0
        )


        total_cross_strict_links = (
            cross["n_strict_links"].sum()
            if len(cross) > 0
            else 0
        )


        separation_records.append({
            "provisional_group": group,
            "consensus_group": subgroup,
            "n_markers": row["n_markers"],

            "internal_median_lod":
                row["median_lod"],

            "internal_max_lod":
                row["max_lod"],

            "strongest_cross_lod":
                strongest_cross_lod,

            "lod_separation_ratio":
                (
                    row["median_lod"]
                    / strongest_cross_lod
                    if pd.notna(strongest_cross_lod)
                    and strongest_cross_lod > 0
                    else np.nan
                ),

            "internal_median_R":
                row["median_R_ril"],

            "best_cross_R":
                best_cross_R,

            "n_internal_core_links":
                row["n_core_links"],

            "n_internal_strict_links":
                row["n_strict_links"],

            "n_cross_core_links":
                int(total_cross_core_links),

            "n_cross_strict_links":
                int(total_cross_strict_links)
        })


subgroup_separation = pd.DataFrame(
    separation_records
)


print("INTERNAL VS CROSS-SUBGROUP LINKAGE")
print("=" * 125)

display(
    subgroup_separation.sort_values(
        [
            "provisional_group",
            "consensus_group"
        ]
    )
)

### Cell 01.59 — identify candidate standalone groups
* This is still diagnostic; it does not rename anything yet.

In [ ]:
# Cell 01.59
# Flag consensus subgroups that may represent independent linkage groups.
#
# Conservative criteria:
# - at least 3 markers
# - internally supported by multiple core links
# - very few strict links to other consensus components

candidate_standalone = (
    subgroup_separation.copy()
)


candidate_standalone[
    "candidate_independent_group"
] = (
    (candidate_standalone["n_markers"] >= 3)
    &
    (candidate_standalone[
        "n_internal_core_links"
    ] >= 2)
    &
    (candidate_standalone[
        "n_cross_strict_links"
    ] <= 1)
)


print("CANDIDATE INDEPENDENT CONSENSUS GROUPS")
print("=" * 120)

display(
    candidate_standalone[
        [
            "provisional_group",
            "consensus_group",
            "n_markers",
            "internal_median_R",
            "internal_median_lod",
            "best_cross_R",
            "strongest_cross_lod",
            "n_internal_core_links",
            "n_cross_core_links",
            "n_cross_strict_links",
            "candidate_independent_group"
        ]
    ]
)

### Cell 01.60 — build a candidate revised linkage-group membership table
* This creates a proposal only. It does not overwrite the original 20-group membership.
* For the four fragile groups, use the consensus components. All other groups remain unchanged.

In [ ]:
# Cell 01.60
# Construct a candidate revised linkage-group membership table.
#
# IMPORTANT:
# This does NOT overwrite core_group_membership.
# It creates a separate proposed structure for evaluation.

revised_records = []


fragile_set = set(
    fragile_focus_groups
)


for _, row in core_group_membership.iterrows():

    old_group = row[
        "provisional_group"
    ]

    marker = row[
        "marker"
    ]


    if old_group not in fragile_set:

        revised_group = old_group


    else:

        match = consensus_membership[
            (
                consensus_membership[
                    "provisional_group"
                ] == old_group
            )
            &
            (
                consensus_membership[
                    "marker"
                ] == marker
            )
        ]


        if len(match) != 1:

            raise ValueError(
                f"Unexpected consensus membership for "
                f"{old_group} / {marker}"
            )


        revised_group = match.iloc[0][
            "consensus_group"
        ]


    revised_records.append({
        "marker": marker,
        "original_group": old_group,
        "candidate_group": revised_group
    })


candidate_revised_membership = pd.DataFrame(
    revised_records
)


candidate_group_sizes = (
    candidate_revised_membership
    .groupby(
        "candidate_group"
    )
    .size()
    .reset_index(
        name="n_markers"
    )
    .sort_values(
        "n_markers",
        ascending=False
    )
)


print("CANDIDATE REVISED LINKAGE-GROUP STRUCTURE")
print("=" * 110)

print(
    "Original number of groups :",
    core_group_membership[
        "provisional_group"
    ].nunique()
)

print(
    "Candidate number of groups:",
    candidate_revised_membership[
        "candidate_group"
    ].nunique()
)


display(
    candidate_group_sizes
)

### Cell 01.61 — strongest linkage between every candidate group pair
* This uses all pairwise information among the 241 backbone markers.

In [ ]:
# Cell 01.61
# Evaluate linkage between every pair of candidate groups.
#
# Goal:
# determine whether the 28-component diagnostic solution contains
# candidate groups that should remain connected.

from itertools import combinations
import numpy as np
import pandas as pd


candidate_groups = sorted(
    candidate_revised_membership[
        "candidate_group"
    ].unique()
)


between_group_records = []


for group_a, group_b in combinations(
    candidate_groups,
    2
):

    markers_a = set(
        candidate_revised_membership.loc[
            candidate_revised_membership[
                "candidate_group"
            ] == group_a,
            "marker"
        ]
    )

    markers_b = set(
        candidate_revised_membership.loc[
            candidate_revised_membership[
                "candidate_group"
            ] == group_b,
            "marker"
        ]
    )


    pairs = backbone_pairwise[
        (
            backbone_pairwise[
                "marker_a"
            ].isin(markers_a)
            &
            backbone_pairwise[
                "marker_b"
            ].isin(markers_b)
        )
        |
        (
            backbone_pairwise[
                "marker_a"
            ].isin(markers_b)
            &
            backbone_pairwise[
                "marker_b"
            ].isin(markers_a)
        )
    ].copy()


    if len(pairs) == 0:
        continue


    # Only reasonably informative comparisons
    informative = pairs[
        pairs["n_overlap"] >= 40
    ].copy()


    if len(informative) == 0:
        continue


    # Sort strongest evidence first:
    # lowest R, then highest LOD
    strongest = (
        informative
        .sort_values(
            ["R_ril", "lod"],
            ascending=[True, False]
        )
        .iloc[0]
    )


    n_core = (
        (informative["R_ril"] <= 0.30)
        &
        (informative["lod"] >= 3.5)
    ).sum()


    n_strict = (
        (informative["R_ril"] <= 0.25)
        &
        (informative["lod"] >= 3.5)
    ).sum()


    n_lod4 = (
        (informative["R_ril"] <= 0.30)
        &
        (informative["lod"] >= 4.0)
    ).sum()


    n_lod5 = (
        (informative["R_ril"] <= 0.30)
        &
        (informative["lod"] >= 5.0)
    ).sum()


    between_group_records.append({
        "group_a": group_a,
        "group_b": group_b,

        "n_pairwise_comparisons":
            len(informative),

        "best_marker_a":
            strongest["marker_a"],

        "best_marker_b":
            strongest["marker_b"],

        "best_R_ril":
            strongest["R_ril"],

        "best_lod":
            strongest["lod"],

        "n_core_links":
            int(n_core),

        "n_strict_links":
            int(n_strict),

        "n_lod4_links":
            int(n_lod4),

        "n_lod5_links":
            int(n_lod5)
    })


candidate_between_groups = pd.DataFrame(
    between_group_records
)


print("STRONGEST LINKAGE BETWEEN CANDIDATE GROUPS")
print("=" * 125)

display(
    candidate_between_groups
    .sort_values(
        [
            "n_lod5_links",
            "n_lod4_links",
            "n_core_links",
            "best_lod"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .head(100)
)

### Cell 01.62 — isolate plausible candidate-group joins
* Now restrict attention to connections that are actually worth considering.

In [ ]:
# Cell 01.62
# Identify candidate group pairs with non-trivial linkage evidence.
#
# We deliberately distinguish:
#   strong multi-edge support
#   moderate multi-edge support
#   single-edge / borderline support

plausible_group_links = (
    candidate_between_groups[
        (
            candidate_between_groups[
                "n_core_links"
            ] >= 1
        )
    ]
    .copy()
)


def classify_between_group_link(row):

    # Multiple stronger links
    if (
        row["n_lod5_links"] >= 2
        or
        (
            row["n_lod4_links"] >= 3
            and row["n_strict_links"] >= 1
        )
    ):
        return "strong_multi_edge"

    # Multiple core links but not especially strong
    elif row["n_core_links"] >= 2:
        return "moderate_multi_edge"

    # Only one core link
    else:
        return "single_borderline_edge"


plausible_group_links[
    "link_class"
] = plausible_group_links.apply(
    classify_between_group_link,
    axis=1
)


print("PLAUSIBLE CONNECTIONS BETWEEN CANDIDATE GROUPS")
print("=" * 130)

display(
    plausible_group_links[
        [
            "group_a",
            "group_b",
            "best_marker_a",
            "best_marker_b",
            "best_R_ril",
            "best_lod",
            "n_core_links",
            "n_strict_links",
            "n_lod4_links",
            "n_lod5_links",
            "link_class"
        ]
    ]
    .sort_values(
        [
            "link_class",
            "n_core_links",
            "best_lod"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)

### Cell 01.63 — candidate-group meta-network
* Here we treat each candidate component as a node. This gives us a much simpler view of the reconstruction problem.

In [ ]:
# Cell 01.63
# Build a meta-network in which each candidate component is one node.
#
# Edges represent linkage evidence between candidate components.

import networkx as nx


group_meta_graph = nx.Graph()

group_meta_graph.add_nodes_from(
    candidate_groups
)


# Add only multi-edge evidence initially.
meta_edges = plausible_group_links[
    plausible_group_links[
        "link_class"
    ].isin(
        [
            "strong_multi_edge",
            "moderate_multi_edge"
        ]
    )
]


for _, row in meta_edges.iterrows():

    group_meta_graph.add_edge(
        row["group_a"],
        row["group_b"],
        n_core_links=row["n_core_links"],
        best_lod=row["best_lod"],
        best_R=row["best_R_ril"],
        link_class=row["link_class"]
    )


meta_components = sorted(
    nx.connected_components(
        group_meta_graph
    ),
    key=len,
    reverse=True
)


meta_component_records = []


for idx, component in enumerate(
    meta_components,
    start=1
):

    meta_id = f"metaLG_{idx:02d}"

    for group in sorted(component):

        meta_component_records.append({
            "meta_group": meta_id,
            "n_candidate_components":
                len(component),
            "candidate_group": group
        })


meta_membership = pd.DataFrame(
    meta_component_records
)


print("CANDIDATE-GROUP META-NETWORK")
print("=" * 110)

print(
    "Candidate components:",
    len(candidate_groups)
)

print(
    "Meta-components using multi-edge evidence:",
    len(meta_components)
)


display(
    meta_membership.sort_values(
        [
            "n_candidate_components",
            "meta_group",
            "candidate_group"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

### Cell 01.64 — summarize which diagnostic splits are supported
* This focuses specifically on the pieces derived from pLG01, pLG03, pLG04, and pLG12.

In [ ]:
# Cell 01.64
# Summarize evidence connecting the diagnostic subdivisions
# of the four fragile original groups.

split_prefixes = [
    "pLG01_",
    "pLG03_",
    "pLG04_",
    "pLG12_"
]


original_split_link_records = []


for old_group in fragile_focus_groups:

    split_groups = sorted(
        candidate_revised_membership.loc[
            candidate_revised_membership[
                "original_group"
            ] == old_group,
            "candidate_group"
        ].unique()
    )


    if len(split_groups) < 2:
        continue


    for group_a, group_b in combinations(
        split_groups,
        2
    ):

        match = candidate_between_groups[
            (
                (
                    candidate_between_groups[
                        "group_a"
                    ] == group_a
                )
                &
                (
                    candidate_between_groups[
                        "group_b"
                    ] == group_b
                )
            )
            |
            (
                (
                    candidate_between_groups[
                        "group_a"
                    ] == group_b
                )
                &
                (
                    candidate_between_groups[
                        "group_b"
                    ] == group_a
                )
            )
        ]


        if len(match) == 0:
            continue


        row = match.iloc[0]


        original_split_link_records.append({
            "original_group": old_group,
            "component_a": group_a,
            "component_b": group_b,
            "best_R_ril":
                row["best_R_ril"],
            "best_lod":
                row["best_lod"],
            "n_core_links":
                row["n_core_links"],
            "n_strict_links":
                row["n_strict_links"],
            "n_lod4_links":
                row["n_lod4_links"],
            "n_lod5_links":
                row["n_lod5_links"]
        })


split_connection_summary = pd.DataFrame(
    original_split_link_records
)


print("EVIDENCE BETWEEN COMPONENTS OF THE FOUR FRAGILE GROUPS")
print("=" * 125)

display(
    split_connection_summary.sort_values(
        [
            "original_group",
            "n_core_links",
            "best_lod"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)

### Cell 01.65 — construct conservative refined seed groups

In [ ]:
# Cell 01.65
# Construct conservative genotype-only seed groups.
#
# Evidence-based decisions:
#
# pLG01:
#   C01 + C03 reconnect through multiple core links.
#   C02 and C04 remain separate candidate fragments.
#
# pLG03:
#   C01 + C03 reconnect through multiple core links.
#   C02 remains separate.
#
# pLG04:
#   C01 + C03 reconnect through multiple core links.
#   C02 remains separate.
#
# pLG12:
#   retain the original 5-marker group provisionally because
#   its 3+2 split is supported by one strict cross-component link.
#
# All other provisional groups remain unchanged.
#
# These are temporary SEED groups, not final chromosomes.

seed_records = []


for _, row in candidate_revised_membership.iterrows():

    marker = row["marker"]
    original_group = row["original_group"]
    candidate_group = row["candidate_group"]


    # pLG01
    if original_group == "pLG01":

        if candidate_group in [
            "pLG01_C01",
            "pLG01_C03"
        ]:
            seed_group = "pLG01_seed_main"

        elif candidate_group == "pLG01_C02":
            seed_group = "pLG01_seed_C02"

        elif candidate_group == "pLG01_C04":
            seed_group = "pLG01_seed_C04"

        else:
            raise ValueError(
                f"Unexpected pLG01 component: "
                f"{candidate_group}"
            )


    # pLG03
    elif original_group == "pLG03":

        if candidate_group in [
            "pLG03_C01",
            "pLG03_C03"
        ]:
            seed_group = "pLG03_seed_main"

        elif candidate_group == "pLG03_C02":
            seed_group = "pLG03_seed_C02"

        else:
            raise ValueError(
                f"Unexpected pLG03 component: "
                f"{candidate_group}"
            )


    # pLG04
    elif original_group == "pLG04":

        if candidate_group in [
            "pLG04_C01",
            "pLG04_C03"
        ]:
            seed_group = "pLG04_seed_main"

        elif candidate_group == "pLG04_C02":
            seed_group = "pLG04_seed_C02"

        else:
            raise ValueError(
                f"Unexpected pLG04 component: "
                f"{candidate_group}"
            )


    # pLG12 retained intact
    elif original_group == "pLG12":

        seed_group = "pLG12"


    # All unaffected original groups
    else:

        seed_group = original_group


    seed_records.append({
        "marker": marker,
        "original_group": original_group,
        "candidate_group": candidate_group,
        "seed_group": seed_group
    })


seed_group_membership = pd.DataFrame(
    seed_records
)


seed_group_sizes = (
    seed_group_membership
    .groupby(
        "seed_group"
    )
    .size()
    .reset_index(
        name="n_markers"
    )
    .sort_values(
        "n_markers",
        ascending=False
    )
)


print("CONSERVATIVE SEED-GROUP STRUCTURE")
print("=" * 105)

print(
    "Markers currently assigned:",
    seed_group_membership[
        "marker"
    ].nunique()
)

print(
    "Number of seed groups:",
    seed_group_membership[
        "seed_group"
    ].nunique()
)

display(seed_group_sizes)

### Cell 01.66 — identify the backbone markers not yet in seed groups
* The conservative backbone contains 241 markers. The original 20 components accounted for only part of them.

In [ ]:
# Cell 01.66
# Identify conservative-backbone markers not yet included
# in any seed linkage group.

backbone_markers = sorted(
    set(
        backbone_pairwise[
            "marker_a"
        ]
    )
    |
    set(
        backbone_pairwise[
            "marker_b"
        ]
    )
)


assigned_seed_markers = set(
    seed_group_membership[
        "marker"
    ]
)


unassigned_backbone_markers = sorted(
    set(backbone_markers)
    -
    assigned_seed_markers
)


print("BACKBONE MARKER ACCOUNTING")
print("=" * 100)

print(
    "Total conservative backbone markers :",
    len(backbone_markers)
)

print(
    "Markers already in seed groups      :",
    len(assigned_seed_markers)
)

print(
    "Currently unassigned backbone markers:",
    len(unassigned_backbone_markers)
)


print("\nUNASSIGNED BACKBONE MARKERS")
print("-" * 100)

print(
    unassigned_backbone_markers
)

### Cell 01.67 — test every unassigned marker against every seed group
* This is an important cell. Rather than globally lowering the threshold and merging everything, each unassigned marker gets evaluated against every existing seed group.

In [ ]:
# Cell 01.67
# Evaluate linkage evidence from every currently unassigned
# backbone marker to every seed group.
#
# The objective is targeted marker placement rather than
# globally relaxing linkage thresholds.

assignment_evidence_records = []


for marker in unassigned_backbone_markers:

    for seed_group in sorted(
        seed_group_membership[
            "seed_group"
        ].unique()
    ):

        group_markers = set(
            seed_group_membership.loc[
                seed_group_membership[
                    "seed_group"
                ] == seed_group,
                "marker"
            ]
        )


        pairs = backbone_pairwise[
            (
                (
                    backbone_pairwise[
                        "marker_a"
                    ] == marker
                )
                &
                (
                    backbone_pairwise[
                        "marker_b"
                    ].isin(group_markers)
                )
            )
            |
            (
                (
                    backbone_pairwise[
                        "marker_b"
                    ] == marker
                )
                &
                (
                    backbone_pairwise[
                        "marker_a"
                    ].isin(group_markers)
                )
            )
        ].copy()


        # Require at least 40 shared RILs
        pairs = pairs[
            pairs[
                "n_overlap"
            ] >= 40
        ].copy()


        if len(pairs) == 0:
            continue


        core_mask = (
            (pairs["R_ril"] <= 0.30)
            &
            (pairs["lod"] >= 3.5)
        )


        strict_mask = (
            (pairs["R_ril"] <= 0.25)
            &
            (pairs["lod"] >= 3.5)
        )


        lod4_mask = (
            (pairs["R_ril"] <= 0.30)
            &
            (pairs["lod"] >= 4.0)
        )


        lod5_mask = (
            (pairs["R_ril"] <= 0.30)
            &
            (pairs["lod"] >= 5.0)
        )


        strongest = (
            pairs
            .sort_values(
                [
                    "R_ril",
                    "lod"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .iloc[0]
        )


        partner = (
            strongest["marker_b"]
            if strongest["marker_a"] == marker
            else strongest["marker_a"]
        )


        assignment_evidence_records.append({
            "marker": marker,
            "seed_group": seed_group,

            "group_size":
                len(group_markers),

            "n_informative_pairs":
                len(pairs),

            "n_core_links":
                int(core_mask.sum()),

            "n_strict_links":
                int(strict_mask.sum()),

            "n_lod4_links":
                int(lod4_mask.sum()),

            "n_lod5_links":
                int(lod5_mask.sum()),

            "best_partner":
                partner,

            "best_R_ril":
                strongest["R_ril"],

            "best_lod":
                strongest["lod"],

            "median_R_ril":
                pairs["R_ril"].median(),

            "median_lod":
                pairs["lod"].median()
        })


unassigned_to_seed = pd.DataFrame(
    assignment_evidence_records
)


print("UNASSIGNED MARKER → SEED-GROUP EVIDENCE")
print("=" * 125)

print(
    "Unassigned markers evaluated:",
    unassigned_to_seed[
        "marker"
    ].nunique()
)

print(
    "Seed groups evaluated:",
    unassigned_to_seed[
        "seed_group"
    ].nunique()
)

### Cell 01.68 — rank the best and second-best seed group for each marker
* This is the crucial output I want to see next.

In [ ]:
# Cell 01.68
# Rank candidate seed groups for every unassigned marker.
#
# Ranking hierarchy:
#   1. number of core links
#   2. number of LOD >= 4 links
#   3. number of strict links
#   4. number of LOD >= 5 links
#   5. strongest LOD
#   6. lowest R
#
# No marker is assigned automatically yet.

ranked_assignment = (
    unassigned_to_seed
    .sort_values(
        [
            "marker",
            "n_core_links",
            "n_lod4_links",
            "n_strict_links",
            "n_lod5_links",
            "best_lod",
            "best_R_ril"
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True
        ]
    )
    .copy()
)


ranked_assignment[
    "group_rank"
] = (
    ranked_assignment
    .groupby(
        "marker"
    )
    .cumcount()
    + 1
)


top1 = (
    ranked_assignment[
        ranked_assignment[
            "group_rank"
        ] == 1
    ]
    .copy()
)


top2 = (
    ranked_assignment[
        ranked_assignment[
            "group_rank"
        ] == 2
    ][
        [
            "marker",
            "seed_group",
            "n_core_links",
            "n_strict_links",
            "n_lod4_links",
            "n_lod5_links",
            "best_R_ril",
            "best_lod"
        ]
    ]
    .rename(
        columns={
            "seed_group":
                "second_seed_group",

            "n_core_links":
                "second_n_core_links",

            "n_strict_links":
                "second_n_strict_links",

            "n_lod4_links":
                "second_n_lod4_links",

            "n_lod5_links":
                "second_n_lod5_links",

            "best_R_ril":
                "second_best_R_ril",

            "best_lod":
                "second_best_lod"
        }
    )
)


best_assignment_summary = (
    top1.merge(
        top2,
        on="marker",
        how="left"
    )
)


best_assignment_summary[
    "core_link_margin"
] = (
    best_assignment_summary[
        "n_core_links"
    ]
    -
    best_assignment_summary[
        "second_n_core_links"
    ].fillna(0)
)


best_assignment_summary[
    "lod_margin"
] = (
    best_assignment_summary[
        "best_lod"
    ]
    -
    best_assignment_summary[
        "second_best_lod"
    ]
)


print("BEST SEED-GROUP EVIDENCE FOR EACH UNASSIGNED MARKER")
print("=" * 145)

display(
    best_assignment_summary[
        [
            "marker",

            "seed_group",
            "best_partner",

            "n_core_links",
            "n_strict_links",
            "n_lod4_links",
            "n_lod5_links",

            "best_R_ril",
            "best_lod",

            "second_seed_group",
            "second_n_core_links",
            "second_n_lod4_links",
            "second_best_R_ril",
            "second_best_lod",

            "core_link_margin",
            "lod_margin"
        ]
    ]
    .sort_values(
        [
            "n_core_links",
            "n_lod4_links",
            "best_lod"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)

### Cell 01.69 — quantify secondary evidence to each seed group
* We'll use two weaker evidence tiers:
    - near-core: R ≤ 0.30, LOD ≥ 2.5
    - supporting: R ≤ 0.325, LOD ≥ 2.0
* These do not redefine linkage groups. They are used only for placing previously isolated markers.

In [ ]:
# Cell 01.69
# Secondary evidence for previously unassigned backbone markers.
#
# Original core rule remains unchanged:
#   overlap >= 40
#   R_ril <= 0.30
#   LOD >= 3.5
#
# Secondary tiers are used ONLY to investigate isolated markers.

secondary_records = []


for marker in unassigned_backbone_markers:

    for seed_group in sorted(
        seed_group_membership["seed_group"].unique()
    ):

        group_markers = set(
            seed_group_membership.loc[
                seed_group_membership["seed_group"] == seed_group,
                "marker"
            ]
        )


        pairs = backbone_pairwise[
            (
                (
                    backbone_pairwise["marker_a"] == marker
                )
                &
                (
                    backbone_pairwise["marker_b"].isin(group_markers)
                )
            )
            |
            (
                (
                    backbone_pairwise["marker_b"] == marker
                )
                &
                (
                    backbone_pairwise["marker_a"].isin(group_markers)
                )
            )
        ].copy()


        pairs = pairs[
            pairs["n_overlap"] >= 40
        ].copy()


        if len(pairs) == 0:
            continue


        near_core = (
            (pairs["R_ril"] <= 0.30)
            &
            (pairs["lod"] >= 2.5)
        )


        supporting = (
            (pairs["R_ril"] <= 0.325)
            &
            (pairs["lod"] >= 2.0)
        )


        strongest_lod_row = (
            pairs.sort_values(
                ["lod", "R_ril"],
                ascending=[False, True]
            )
            .iloc[0]
        )


        best_R_row = (
            pairs.sort_values(
                ["R_ril", "lod"],
                ascending=[True, False]
            )
            .iloc[0]
        )


        secondary_records.append({
            "marker": marker,
            "seed_group": seed_group,
            "seed_group_size": len(group_markers),

            "n_pairs": len(pairs),

            "n_near_core_links":
                int(near_core.sum()),

            "n_supporting_links":
                int(supporting.sum()),

            "max_lod":
                pairs["lod"].max(),

            "min_R_ril":
                pairs["R_ril"].min(),

            "median_lod":
                pairs["lod"].median(),

            "median_R_ril":
                pairs["R_ril"].median(),

            "strongest_lod_partner":
                (
                    strongest_lod_row["marker_b"]
                    if strongest_lod_row["marker_a"] == marker
                    else strongest_lod_row["marker_a"]
                ),

            "strongest_lod":
                strongest_lod_row["lod"],

            "strongest_lod_R":
                strongest_lod_row["R_ril"],

            "best_R_partner":
                (
                    best_R_row["marker_b"]
                    if best_R_row["marker_a"] == marker
                    else best_R_row["marker_a"]
                ),

            "best_R":
                best_R_row["R_ril"],

            "best_R_lod":
                best_R_row["lod"]
        })


secondary_assignment_evidence = pd.DataFrame(
    secondary_records
)


print("SECONDARY PLACEMENT EVIDENCE CREATED")
print("=" * 110)

print(
    "Markers evaluated:",
    secondary_assignment_evidence[
        "marker"
    ].nunique()
)

print(
    "Seed groups evaluated:",
    secondary_assignment_evidence[
        "seed_group"
    ].nunique()
)

### Cell 01.70 — rank first and second seed-group evidence
* The difference between the best and second-best group is at least as important as the best score itself.

In [ ]:
# Cell 01.70
# Rank secondary group evidence for each unassigned marker.

secondary_ranked = (
    secondary_assignment_evidence
    .sort_values(
        [
            "marker",
            "n_near_core_links",
            "n_supporting_links",
            "max_lod",
            "min_R_ril"
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True
        ]
    )
    .copy()
)


secondary_ranked["group_rank"] = (
    secondary_ranked
    .groupby("marker")
    .cumcount()
    + 1
)


secondary_best = (
    secondary_ranked[
        secondary_ranked["group_rank"] == 1
    ]
    .copy()
)


secondary_second = (
    secondary_ranked[
        secondary_ranked["group_rank"] == 2
    ][
        [
            "marker",
            "seed_group",
            "n_near_core_links",
            "n_supporting_links",
            "max_lod",
            "min_R_ril"
        ]
    ]
    .rename(
        columns={
            "seed_group":
                "second_seed_group",

            "n_near_core_links":
                "second_near_core_links",

            "n_supporting_links":
                "second_supporting_links",

            "max_lod":
                "second_max_lod",

            "min_R_ril":
                "second_min_R_ril"
        }
    )
)


secondary_best_summary = (
    secondary_best
    .merge(
        secondary_second,
        on="marker",
        how="left"
    )
)


secondary_best_summary[
    "near_core_margin"
] = (
    secondary_best_summary[
        "n_near_core_links"
    ]
    -
    secondary_best_summary[
        "second_near_core_links"
    ].fillna(0)
)


secondary_best_summary[
    "supporting_link_margin"
] = (
    secondary_best_summary[
        "n_supporting_links"
    ]
    -
    secondary_best_summary[
        "second_supporting_links"
    ].fillna(0)
)


secondary_best_summary[
    "lod_margin"
] = (
    secondary_best_summary[
        "max_lod"
    ]
    -
    secondary_best_summary[
        "second_max_lod"
    ]
)


print("SECONDARY EVIDENCE — BEST VS SECOND-BEST GROUP")
print("=" * 150)

display(
    secondary_best_summary[
        [
            "marker",
            "seed_group",

            "n_near_core_links",
            "n_supporting_links",

            "strongest_lod_partner",
            "max_lod",
            "min_R_ril",

            "second_seed_group",
            "second_near_core_links",
            "second_supporting_links",
            "second_max_lod",
            "second_min_R_ril",

            "near_core_margin",
            "supporting_link_margin",
            "lod_margin"
        ]
    ]
    .sort_values(
        [
            "n_near_core_links",
            "n_supporting_links",
            "max_lod"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)

### Cell 01.71 — classify placement confidence conservatively
* This will deliberately leave many markers unresolved.

In [ ]:
# Cell 01.71
# Conservative secondary-placement classification.
#
# No assignment is final.
#
# HIGH:
#   >=2 near-core links to best group
#   AND clear advantage over second group
#
# MODERATE:
#   >=1 near-core link
#   OR >=2 supporting links with clear group advantage
#
# AMBIGUOUS:
#   appreciable evidence for more than one group
#
# WEAK:
#   no meaningful targeted evidence

def classify_secondary_assignment(row):

    best_near = row["n_near_core_links"]
    second_near = row["second_near_core_links"]

    best_support = row["n_supporting_links"]
    second_support = row["second_supporting_links"]

    lod_margin = row["lod_margin"]


    # High-confidence targeted placement
    if (
        best_near >= 2
        and best_near > second_near
        and best_support > second_support
    ):
        return "high"


    # Competing near-core evidence
    if (
        best_near >= 1
        and second_near >= 1
    ):
        return "ambiguous"


    # One near-core link with reasonable separation
    if (
        best_near >= 1
        and second_near == 0
        and lod_margin >= 0.5
    ):
        return "moderate"


    # Several weaker supporting links uniquely favor one group
    if (
        best_near == 0
        and best_support >= 2
        and best_support > second_support
        and lod_margin >= 0.5
    ):
        return "moderate"


    # Similar evidence to two candidate groups
    if (
        best_support >= 1
        and second_support >= 1
        and abs(lod_margin) < 0.5
    ):
        return "ambiguous"


    return "weak"


secondary_best_summary[
    "placement_class"
] = (
    secondary_best_summary.apply(
        classify_secondary_assignment,
        axis=1
    )
)


print("SECONDARY PLACEMENT CLASSIFICATION")
print("=" * 145)

print(
    secondary_best_summary[
        "placement_class"
    ].value_counts()
)


display(
    secondary_best_summary[
        [
            "marker",
            "seed_group",
            "placement_class",

            "n_near_core_links",
            "n_supporting_links",
            "max_lod",
            "min_R_ril",

            "second_seed_group",
            "second_near_core_links",
            "second_supporting_links",
            "second_max_lod",

            "near_core_margin",
            "supporting_link_margin",
            "lod_margin"
        ]
    ]
    .sort_values(
        [
            "placement_class",
            "n_near_core_links",
            "n_supporting_links",
            "max_lod"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
)

### Cell 01.72 — look specifically for bridge markers
* This is especially important because an unassigned marker might connect two of our separated seed fragments.

In [ ]:
# Cell 01.72
# Identify potential bridge markers.
#
# A bridge candidate has secondary evidence to at least two seed groups.
# We inspect these separately rather than assigning them immediately.

bridge_records = []


for marker in unassigned_backbone_markers:

    temp = secondary_assignment_evidence[
        secondary_assignment_evidence[
            "marker"
        ] == marker
    ].copy()


    # Retain groups with at least some meaningful secondary evidence
    supported = temp[
        (
            temp["n_near_core_links"] >= 1
        )
        |
        (
            temp["n_supporting_links"] >= 1
        )
    ].copy()


    if supported["seed_group"].nunique() >= 2:

        supported = supported.sort_values(
            [
                "n_near_core_links",
                "n_supporting_links",
                "max_lod"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )


        top_groups = supported.head(4)


        for _, row in top_groups.iterrows():

            bridge_records.append({
                "marker": marker,
                "seed_group":
                    row["seed_group"],

                "n_near_core_links":
                    row["n_near_core_links"],

                "n_supporting_links":
                    row["n_supporting_links"],

                "max_lod":
                    row["max_lod"],

                "min_R_ril":
                    row["min_R_ril"],

                "strongest_partner":
                    row["strongest_lod_partner"]
            })


bridge_candidates = pd.DataFrame(
    bridge_records
)


print("POTENTIAL BRIDGE MARKERS")
print("=" * 125)


if len(bridge_candidates) == 0:

    print(
        "No unassigned markers show meaningful "
        "secondary support to >=2 seed groups."
    )

else:

    print(
        "Number of possible bridge markers:",
        bridge_candidates[
            "marker"
        ].nunique()
    )

    display(
        bridge_candidates.sort_values(
            [
                "marker",
                "n_near_core_links",
                "n_supporting_links",
                "max_lod"
            ],
            ascending=[
                True,
                False,
                False,
                False
            ]
        )
    )

### Cell 01.73 — reconstruct the 92-RIL mapping matrix and define pair statistics

In [ ]:
# Cell 01.73
# Reconstruct the 92-RIL mapping genotype matrix and define
# a reusable phase-independent pairwise linkage function.

import numpy as np
import pandas as pd


excluded_rils = [
    "fxh_ril_03",
    "fxh_ril_43"
]


mapping_geno = (
    geno_work[
        ~geno_work["ril"].isin(
            excluded_rils
        )
    ]
    .copy()
    .reset_index(drop=True)
)


print("MAPPING MATRIX")
print("=" * 90)

print(
    "RILs:",
    mapping_geno.shape[0]
)

print(
    "Markers:",
    mapping_geno.shape[1] - 1
)


def pair_stats_from_df(df, marker_a, marker_b):
    """
    Phase-independent pairwise linkage statistics.

    Returns:
        n_overlap
        R_ril
        lod
    """

    x = pd.to_numeric(
        df[marker_a],
        errors="coerce"
    )

    y = pd.to_numeric(
        df[marker_b],
        errors="coerce"
    )


    valid = (
        x.notna()
        &
        y.notna()
    )


    x = x[valid].to_numpy()
    y = y[valid].to_numpy()

    n = len(x)


    if n == 0:
        return (
            0,
            np.nan,
            np.nan
        )


    n_same = np.sum(
        x == y
    )

    n_diff = (
        n - n_same
    )


    # Phase-independent:
    # choose smaller recombinant count
    n_rec = min(
        n_same,
        n_diff
    )


    R = (
        n_rec / n
    )


    # LOD versus unlinked R = 0.5
    if R <= 0:

        lod = (
            n
            *
            np.log10(2)
        )

    elif R >= 0.5:

        lod = 0.0

    else:

        lod = (
            (n - n_rec)
            *
            np.log10(
                (1 - R) / 0.5
            )
            +
            n_rec
            *
            np.log10(
                R / 0.5
            )
        )


    return (
        n,
        R,
        lod
    )


print(
    "\nPair-statistics function ready."
)

### Cell 01.74 — define the critical borderline edges
* These are the actual edges determining our questionable splits.

In [ ]:
# Cell 01.74
# Define critical inter-component edges for resampling.
#
# These are genotype-derived links that currently determine
# whether fragile seed components remain connected.

critical_edges = pd.DataFrame(
    [
        {
            "original_group": "pLG01",
            "marker_a": "Satt357a",
            "marker_b": "Satt316",
            "edge_role": "main_to_C03"
        },

        {
            "original_group": "pLG01",
            "marker_a": "Satt424b",
            "marker_b": "PWL1",
            "edge_role": "main_to_C02"
        },

        {
            "original_group": "pLG01",
            "marker_a": "Satt363",
            "marker_b": "Satt305a",
            "edge_role": "main_to_C04"
        },

        {
            "original_group": "pLG03",
            "marker_a": "Satt208",
            "marker_b": "Sat_092",
            "edge_role": "main_to_C03"
        },

        {
            "original_group": "pLG03",
            "marker_a": "Satt310",
            "marker_b": "Satt001b",
            "edge_role": "main_to_C02"
        },

        {
            "original_group": "pLG04",
            "marker_a": "Satt600",
            "marker_b": "Satt546",
            "edge_role": "main_to_C03"
        },

        {
            "original_group": "pLG04",
            "marker_a": "Satt359",
            "marker_b": "Satt444",
            "edge_role": "main_to_C02"
        },

        {
            "original_group": "pLG12",
            "marker_a": "Satt251",
            "marker_b": "Satt132",
            "edge_role": "C01_to_C02"
        }
    ]
)


# Recalculate original statistics directly
original_edge_stats = []


for _, row in critical_edges.iterrows():

    n, R, lod = pair_stats_from_df(
        mapping_geno,
        row["marker_a"],
        row["marker_b"]
    )


    original_edge_stats.append({
        **row.to_dict(),
        "n_overlap": n,
        "R_ril": R,
        "lod": lod
    })


critical_edge_stats = pd.DataFrame(
    original_edge_stats
)


print("CRITICAL BORDERLINE EDGES")
print("=" * 110)

display(
    critical_edge_stats
)

### Cell 01.75 — leave-one-RIL-out stability
* This is deterministic and asks whether one individual is disproportionately responsible for an edge meeting the linkage criterion.

In [ ]:
# Cell 01.75
# Leave-one-RIL-out stability analysis for critical edges.
#
# Core criterion:
#   n_overlap >= 40
#   R_ril <= 0.30
#   LOD >= 3.5

CORE_MIN_OVERLAP = 40
CORE_MAX_R = 0.30
CORE_MIN_LOD = 3.5


loo_records = []


for leave_idx in range(
    len(mapping_geno)
):

    loo_df = (
        mapping_geno
        .drop(
            index=leave_idx
        )
        .reset_index(drop=True)
    )


    left_out_ril = (
        mapping_geno.loc[
            leave_idx,
            "ril"
        ]
    )


    for _, edge in critical_edges.iterrows():

        n, R, lod = pair_stats_from_df(
            loo_df,
            edge["marker_a"],
            edge["marker_b"]
        )


        passes_core = (
            n >= CORE_MIN_OVERLAP
            and R <= CORE_MAX_R
            and lod >= CORE_MIN_LOD
        )


        loo_records.append({
            "left_out_ril":
                left_out_ril,

            "original_group":
                edge["original_group"],

            "edge_role":
                edge["edge_role"],

            "marker_a":
                edge["marker_a"],

            "marker_b":
                edge["marker_b"],

            "n_overlap":
                n,

            "R_ril":
                R,

            "lod":
                lod,

            "passes_core":
                passes_core
        })


loo_edge_results = pd.DataFrame(
    loo_records
)


loo_edge_summary = (
    loo_edge_results
    .groupby(
        [
            "original_group",
            "edge_role",
            "marker_a",
            "marker_b"
        ]
    )
    .agg(
        n_loo=("passes_core", "size"),

        pass_count=(
            "passes_core",
            "sum"
        ),

        pass_rate=(
            "passes_core",
            "mean"
        ),

        R_min=(
            "R_ril",
            "min"
        ),

        R_median=(
            "R_ril",
            "median"
        ),

        R_max=(
            "R_ril",
            "max"
        ),

        lod_min=(
            "lod",
            "min"
        ),

        lod_median=(
            "lod",
            "median"
        ),

        lod_max=(
            "lod",
            "max"
        )
    )
    .reset_index()
)


print("LEAVE-ONE-RIL-OUT EDGE STABILITY")
print("=" * 135)

display(
    loo_edge_summary.sort_values(
        "pass_rate",
        ascending=False
    )
)

### Cell 01.76 — bootstrap stability of the same edges
* Use 500 bootstrap replicates with a fixed seed.

In [ ]:
# Cell 01.76
# Bootstrap stability of critical inter-component edges.
#
# Resample the 92 RILs with replacement.
#
# The bootstrap pass rate estimates how often each edge
# satisfies the original linkage rule under population
# resampling.

N_BOOT = 500
RANDOM_SEED = 20260910

rng = np.random.default_rng(
    RANDOM_SEED
)


bootstrap_records = []


n_rils = len(
    mapping_geno
)


for bootstrap_id in range(
    1,
    N_BOOT + 1
):

    sample_idx = rng.integers(
        low=0,
        high=n_rils,
        size=n_rils
    )


    boot_df = (
        mapping_geno
        .iloc[sample_idx]
        .reset_index(drop=True)
    )


    for _, edge in critical_edges.iterrows():

        n, R, lod = pair_stats_from_df(
            boot_df,
            edge["marker_a"],
            edge["marker_b"]
        )


        passes_core = (
            n >= CORE_MIN_OVERLAP
            and R <= CORE_MAX_R
            and lod >= CORE_MIN_LOD
        )


        bootstrap_records.append({
            "bootstrap_id":
                bootstrap_id,

            "original_group":
                edge["original_group"],

            "edge_role":
                edge["edge_role"],

            "marker_a":
                edge["marker_a"],

            "marker_b":
                edge["marker_b"],

            "n_overlap":
                n,

            "R_ril":
                R,

            "lod":
                lod,

            "passes_core":
                passes_core
        })


bootstrap_edge_results = pd.DataFrame(
    bootstrap_records
)


bootstrap_edge_summary = (
    bootstrap_edge_results
    .groupby(
        [
            "original_group",
            "edge_role",
            "marker_a",
            "marker_b"
        ]
    )
    .agg(
        n_boot=(
            "passes_core",
            "size"
        ),

        pass_count=(
            "passes_core",
            "sum"
        ),

        pass_rate=(
            "passes_core",
            "mean"
        ),

        R_q025=(
            "R_ril",
            lambda x:
                x.quantile(0.025)
        ),

        R_median=(
            "R_ril",
            "median"
        ),

        R_q975=(
            "R_ril",
            lambda x:
                x.quantile(0.975)
        ),

        lod_q025=(
            "lod",
            lambda x:
                x.quantile(0.025)
        ),

        lod_median=(
            "lod",
            "median"
        ),

        lod_q975=(
            "lod",
            lambda x:
                x.quantile(0.975)
        )
    )
    .reset_index()
)


print("BOOTSTRAP STABILITY OF CRITICAL EDGES")
print("=" * 145)

display(
    bootstrap_edge_summary.sort_values(
        "pass_rate",
        ascending=False
    )
)

### Cell 01.77 — summarize critical-edge resampling evidence

In [ ]:
# Cell 01.77
# Combine original, leave-one-out, and bootstrap evidence
# for the critical inter-component edges.

critical_resampling_summary = (
    critical_edge_stats
    .merge(
        loo_edge_summary[
            [
                "original_group",
                "edge_role",
                "marker_a",
                "marker_b",
                "pass_rate"
            ]
        ].rename(
            columns={
                "pass_rate": "loo_pass_rate"
            }
        ),
        on=[
            "original_group",
            "edge_role",
            "marker_a",
            "marker_b"
        ],
        how="left"
    )
    .merge(
        bootstrap_edge_summary[
            [
                "original_group",
                "edge_role",
                "marker_a",
                "marker_b",
                "pass_rate",
                "R_q025",
                "R_median",
                "R_q975",
                "lod_q025",
                "lod_median",
                "lod_q975"
            ]
        ].rename(
            columns={
                "pass_rate": "bootstrap_pass_rate"
            }
        ),
        on=[
            "original_group",
            "edge_role",
            "marker_a",
            "marker_b"
        ],
        how="left"
    )
)


def edge_stability_class(row):

    b = row["bootstrap_pass_rate"]
    l = row["loo_pass_rate"]

    if l >= 0.95 and b >= 0.70:
        return "relatively_robust"

    elif l >= 0.95 and b >= 0.60:
        return "moderately_robust"

    elif b >= 0.45:
        return "borderline"

    else:
        return "unstable"


critical_resampling_summary[
    "stability_class"
] = (
    critical_resampling_summary.apply(
        edge_stability_class,
        axis=1
    )
)


print("CRITICAL EDGE RESAMPLING SUMMARY")
print("=" * 145)

display(
    critical_resampling_summary[
        [
            "original_group",
            "edge_role",
            "marker_a",
            "marker_b",
            "R_ril",
            "lod",
            "loo_pass_rate",
            "bootstrap_pass_rate",
            "R_q025",
            "R_median",
            "R_q975",
            "lod_q025",
            "lod_median",
            "lod_q975",
            "stability_class"
        ]
    ]
    .sort_values(
        "bootstrap_pass_rate",
        ascending=False
    )
)

### Cell 01.78 — bootstrap the entire fragile-group graph
* This asks whether each original fragile group remains connected under repeated RIL resampling, using all pairwise marker relationships, not just one selected edge.

In [ ]:
# Cell 01.78
# Bootstrap connectivity of the complete fragile-group graphs.
#
# This is more informative than testing only the weakest edge,
# because alternate paths can preserve linkage-group connectivity.

import networkx as nx
import numpy as np
import pandas as pd


N_GRAPH_BOOT = 500
GRAPH_SEED = 20260910

rng_graph = np.random.default_rng(
    GRAPH_SEED
)


# Convert genotype columns once for speed
geno_array_dict = {}

for marker in seed_group_membership["marker"].unique():

    geno_array_dict[marker] = (
        pd.to_numeric(
            mapping_geno[marker],
            errors="coerce"
        )
        .to_numpy()
    )


def pair_stats_bootstrap_arrays(
    marker_a,
    marker_b,
    sample_idx
):

    x = geno_array_dict[marker_a][sample_idx]
    y = geno_array_dict[marker_b][sample_idx]

    valid = (
        ~np.isnan(x)
        &
        ~np.isnan(y)
    )

    x = x[valid]
    y = y[valid]

    n = len(x)

    if n == 0:
        return 0, np.nan, np.nan

    n_same = np.sum(
        x == y
    )

    n_diff = (
        n - n_same
    )

    n_rec = min(
        n_same,
        n_diff
    )

    R = (
        n_rec / n
    )

    if R <= 0:

        lod = (
            n * np.log10(2)
        )

    elif R >= 0.5:

        lod = 0.0

    else:

        lod = (
            (n - n_rec)
            * np.log10(
                (1 - R) / 0.5
            )
            +
            n_rec
            * np.log10(
                R / 0.5
            )
        )

    return n, R, lod


fragile_original_markers = {}

for group in fragile_focus_groups:

    fragile_original_markers[group] = (
        core_group_membership.loc[
            core_group_membership[
                "provisional_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


graph_boot_records = []

n_rils = len(mapping_geno)


for boot_id in range(
    1,
    N_GRAPH_BOOT + 1
):

    sample_idx = rng_graph.integers(
        0,
        n_rils,
        size=n_rils
    )


    for group, markers in fragile_original_markers.items():

        G = nx.Graph()

        G.add_nodes_from(
            markers
        )


        for i in range(
            len(markers)
        ):

            for j in range(
                i + 1,
                len(markers)
            ):

                a = markers[i]
                b = markers[j]

                n, R, lod = (
                    pair_stats_bootstrap_arrays(
                        a,
                        b,
                        sample_idx
                    )
                )


                if (
                    n >= 40
                    and R <= 0.30
                    and lod >= 3.5
                ):

                    G.add_edge(
                        a,
                        b
                    )


        components = sorted(
            nx.connected_components(G),
            key=len,
            reverse=True
        )


        sizes = [
            len(c)
            for c in components
        ]


        graph_boot_records.append({
            "bootstrap_id":
                boot_id,

            "provisional_group":
                group,

            "n_markers":
                len(markers),

            "n_components":
                len(components),

            "largest_component":
                sizes[0],

            "largest_fraction":
                sizes[0]
                / len(markers),

            "fully_connected":
                len(components) == 1
        })


fragile_graph_bootstrap = pd.DataFrame(
    graph_boot_records
)


fragile_graph_bootstrap_summary = (
    fragile_graph_bootstrap
    .groupby(
        "provisional_group"
    )
    .agg(
        n_boot=(
            "bootstrap_id",
            "size"
        ),

        full_connectivity_rate=(
            "fully_connected",
            "mean"
        ),

        median_n_components=(
            "n_components",
            "median"
        ),

        mean_n_components=(
            "n_components",
            "mean"
        ),

        median_largest_fraction=(
            "largest_fraction",
            "median"
        ),

        q025_largest_fraction=(
            "largest_fraction",
            lambda x:
                x.quantile(0.025)
        ),

        q975_largest_fraction=(
            "largest_fraction",
            lambda x:
                x.quantile(0.975)
        )
    )
    .reset_index()
)


print("BOOTSTRAP CONNECTIVITY OF COMPLETE FRAGILE GROUPS")
print("=" * 130)

display(
    fragile_graph_bootstrap_summary
)

### Cell 01.79 — bootstrap connectivity between the consensus subgroups
* This is the most important of these cells. Instead of asking whether one marker pair survives, it asks:
* Is there any path through the resampled linkage graph connecting the two proposed subgroups?

In [ ]:
# Cell 01.79
# Estimate bootstrap graph-connectivity probability between
# consensus components within each fragile original group.

consensus_lookup = (
    consensus_membership
    .set_index("marker")[
        "consensus_group"
    ]
    .to_dict()
)


subgroup_pairs = []


for group in fragile_focus_groups:

    subgroups = sorted(
        consensus_membership.loc[
            consensus_membership[
                "provisional_group"
            ] == group,
            "consensus_group"
        ].unique()
    )


    for i in range(
        len(subgroups)
    ):

        for j in range(
            i + 1,
            len(subgroups)
        ):

            subgroup_pairs.append(
                (
                    group,
                    subgroups[i],
                    subgroups[j]
                )
            )


subgroup_connect_records = []


# Reuse same bootstrap seed for reproducibility
rng_connect = np.random.default_rng(
    GRAPH_SEED
)


for boot_id in range(
    1,
    N_GRAPH_BOOT + 1
):

    sample_idx = rng_connect.integers(
        0,
        n_rils,
        size=n_rils
    )


    for group in fragile_focus_groups:

        markers = (
            fragile_original_markers[
                group
            ]
        )


        G = nx.Graph()

        G.add_nodes_from(
            markers
        )


        for i in range(
            len(markers)
        ):

            for j in range(
                i + 1,
                len(markers)
            ):

                a = markers[i]
                b = markers[j]

                n, R, lod = (
                    pair_stats_bootstrap_arrays(
                        a,
                        b,
                        sample_idx
                    )
                )


                if (
                    n >= 40
                    and R <= 0.30
                    and lod >= 3.5
                ):

                    G.add_edge(
                        a,
                        b
                    )


        group_subgroups = sorted(
            consensus_membership.loc[
                consensus_membership[
                    "provisional_group"
                ] == group,
                "consensus_group"
            ].unique()
        )


        for i in range(
            len(group_subgroups)
        ):

            for j in range(
                i + 1,
                len(group_subgroups)
            ):

                sg_a = group_subgroups[i]
                sg_b = group_subgroups[j]


                markers_a = (
                    consensus_membership.loc[
                        consensus_membership[
                            "consensus_group"
                        ] == sg_a,
                        "marker"
                    ]
                    .tolist()
                )


                markers_b = (
                    consensus_membership.loc[
                        consensus_membership[
                            "consensus_group"
                        ] == sg_b,
                        "marker"
                    ]
                    .tolist()
                )


                connected = False


                for a in markers_a:

                    for b in markers_b:

                        if nx.has_path(
                            G,
                            a,
                            b
                        ):

                            connected = True
                            break

                    if connected:
                        break


                subgroup_connect_records.append({
                    "bootstrap_id":
                        boot_id,

                    "provisional_group":
                        group,

                    "subgroup_a":
                        sg_a,

                    "subgroup_b":
                        sg_b,

                    "graph_connected":
                        connected
                })


bootstrap_subgroup_connectivity = pd.DataFrame(
    subgroup_connect_records
)


bootstrap_subgroup_summary = (
    bootstrap_subgroup_connectivity
    .groupby(
        [
            "provisional_group",
            "subgroup_a",
            "subgroup_b"
        ]
    )
    .agg(
        n_boot=(
            "bootstrap_id",
            "size"
        ),

        connectivity_rate=(
            "graph_connected",
            "mean"
        )
    )
    .reset_index()
)


print("BOOTSTRAP CONNECTIVITY BETWEEN CONSENSUS COMPONENTS")
print("=" * 125)

display(
    bootstrap_subgroup_summary.sort_values(
        [
            "provisional_group",
            "connectivity_rate"
        ],
        ascending=[
            True,
            False
        ]
    )
)

### Cell 01.80 — generate a provisional connectivity interpretation
* This doesn't change marker membership. It simply categorizes the bootstrap results.

In [ ]:
# Cell 01.80
# Descriptive interpretation of bootstrap subgroup connectivity.
#
# These labels remain provisional.

def classify_graph_connectivity(rate):

    if rate >= 0.80:
        return "strong_reconnection"

    elif rate >= 0.60:
        return "moderate_reconnection"

    elif rate >= 0.40:
        return "uncertain_boundary"

    else:
        return "persistent_separation"


bootstrap_subgroup_summary[
    "connectivity_class"
] = (
    bootstrap_subgroup_summary[
        "connectivity_rate"
    ]
    .apply(
        classify_graph_connectivity
    )
)


print("PROVISIONAL SUBGROUP CONNECTIVITY INTERPRETATION")
print("=" * 125)


display(
    bootstrap_subgroup_summary[
        [
            "provisional_group",
            "subgroup_a",
            "subgroup_b",
            "connectivity_rate",
            "connectivity_class"
        ]
    ]
    .sort_values(
        [
            "provisional_group",
            "connectivity_rate"
        ],
        ascending=[
            True,
            False
        ]
    )
)


print("\nCONNECTIVITY CLASS COUNTS")
print("-" * 60)

print(
    bootstrap_subgroup_summary[
        "connectivity_class"
    ].value_counts()
)

### Cell 01.81 — quantify attachment of every consensus component to its main core

In [ ]:
# Cell 01.81
# Summarize bootstrap attachment of each fragile-group
# consensus component to its designated main component.
#
# This converts the bootstrap graph results into a
# component-level confidence table.

main_component_lookup = {
    "pLG01": "pLG01_C01",
    "pLG03": "pLG03_C01",
    "pLG04": "pLG04_C01",
    "pLG12": "pLG12_C01"
}


attachment_records = []


for group in fragile_focus_groups:

    main_component = (
        main_component_lookup[group]
    )

    components = sorted(
        consensus_membership.loc[
            consensus_membership[
                "provisional_group"
            ] == group,
            "consensus_group"
        ].unique()
    )


    for component in components:

        n_markers = (
            consensus_membership.loc[
                consensus_membership[
                    "consensus_group"
                ] == component,
                "marker"
            ]
            .nunique()
        )


        # Main component is its own reference core
        if component == main_component:

            attachment_rate = 1.0
            component_role = "main_core"

        else:

            temp = bootstrap_subgroup_summary[
                (
                    bootstrap_subgroup_summary[
                        "provisional_group"
                    ] == group
                )
                &
                (
                    (
                        (
                            bootstrap_subgroup_summary[
                                "subgroup_a"
                            ] == main_component
                        )
                        &
                        (
                            bootstrap_subgroup_summary[
                                "subgroup_b"
                            ] == component
                        )
                    )
                    |
                    (
                        (
                            bootstrap_subgroup_summary[
                                "subgroup_b"
                            ] == main_component
                        )
                        &
                        (
                            bootstrap_subgroup_summary[
                                "subgroup_a"
                            ] == component
                        )
                    )
                )
            ]


            if len(temp) != 1:

                raise ValueError(
                    f"Expected one bootstrap comparison for "
                    f"{main_component} vs {component}"
                )


            attachment_rate = (
                temp.iloc[0][
                    "connectivity_rate"
                ]
            )


            if attachment_rate >= 0.80:
                component_role = "supported_attachment"

            elif attachment_rate >= 0.60:
                component_role = "provisional_attachment"

            else:
                component_role = "uncertain_satellite"


        attachment_records.append({
            "provisional_group":
                group,

            "main_component":
                main_component,

            "consensus_group":
                component,

            "n_markers":
                n_markers,

            "attachment_rate":
                attachment_rate,

            "component_role":
                component_role
        })


component_attachment_summary = pd.DataFrame(
    attachment_records
)


print("BOOTSTRAP-SUPPORTED COMPONENT ATTACHMENT")
print("=" * 115)

display(
    component_attachment_summary.sort_values(
        [
            "provisional_group",
            "attachment_rate"
        ],
        ascending=[
            True,
            False
        ]
    )
)

### Cell 01.82 — restore 20 provisional LG identities but preserve confidence annotations
* This is important: we are not returning blindly to the original graph. We retain the original group identity while recording which markers are core versus uncertain.

In [ ]:
# Cell 01.82
# Construct a resampling-supported working membership table.
#
# We preserve the original 20 provisional linkage-group IDs.
# Component uncertainty is stored separately rather than
# creating artificial new linkage groups.

working_records = []


attachment_role_lookup = (
    component_attachment_summary
    .set_index("consensus_group")[
        [
            "attachment_rate",
            "component_role"
        ]
    ]
    .to_dict("index")
)


consensus_marker_lookup = (
    consensus_membership
    .set_index("marker")[
        "consensus_group"
    ]
    .to_dict()
)


for _, row in core_group_membership.iterrows():

    marker = row["marker"]
    group = row["provisional_group"]


    if group in fragile_focus_groups:

        consensus_group = (
            consensus_marker_lookup[
                marker
            ]
        )

        attachment_info = (
            attachment_role_lookup[
                consensus_group
            ]
        )

        attachment_rate = (
            attachment_info[
                "attachment_rate"
            ]
        )

        component_role = (
            attachment_info[
                "component_role"
            ]
        )

    else:

        consensus_group = pd.NA
        attachment_rate = 1.0
        component_role = "established_core"


    working_records.append({
        "marker":
            marker,

        "working_group":
            group,

        "consensus_group":
            consensus_group,

        "attachment_rate":
            attachment_rate,

        "component_role":
            component_role
    })


working_group_membership = pd.DataFrame(
    working_records
)


print("RESAMPLING-SUPPORTED WORKING GROUP STRUCTURE")
print("=" * 120)

print(
    "Working linkage groups:",
    working_group_membership[
        "working_group"
    ].nunique()
)

print(
    "Markers represented:",
    working_group_membership[
        "marker"
    ].nunique()
)


print("\nMARKERS BY COMPONENT ROLE")
print("-" * 70)

print(
    working_group_membership[
        "component_role"
    ].value_counts()
)

### Cell 01.83 — define the high-confidence ordering backbone
* Now we decide which markers are safe enough to use for initial marker ordering.
* For the fragile groups:
    - main cores: include;
    - supported attachments ≥0.80: include;
    - pLG01_C03 is slightly below 0.80 at 0.776, but because it also has two direct core links and the strongest direct edge bootstraps at 0.644, I would include it as a special moderately supported core extension;
    - other provisional/uncertain satellites: hold out initially.

In [ ]:
# Cell 01.83
# Define markers used for INITIAL marker ordering.
#
# Satellite fragments are not deleted.
# They are held out for later insertion after the core
# marker order is established.

ordering_records = []


for _, row in working_group_membership.iterrows():

    marker = row["marker"]
    group = row["working_group"]
    consensus_group = row["consensus_group"]
    role = row["component_role"]


    include = False
    reason = None


    # Unaffected groups
    if role == "established_core":

        include = True
        reason = "established_core"


    # Main components of fragile groups
    elif role == "main_core":

        include = True
        reason = "main_core"


    # Strong bootstrap-supported attachment
    elif role == "supported_attachment":

        include = True
        reason = "bootstrap_supported_attachment"


    # Specific pLG01 C03 extension:
    # 0.776 graph connectivity + multiple direct core links
    elif (
        group == "pLG01"
        and consensus_group == "pLG01_C03"
    ):

        include = True
        reason = "moderately_supported_core_extension"


    else:

        include = False
        reason = "holdout_satellite"


    ordering_records.append({
        "marker":
            marker,

        "working_group":
            group,

        "consensus_group":
            consensus_group,

        "component_role":
            role,

        "attachment_rate":
            row["attachment_rate"],

        "include_initial_ordering":
            include,

        "ordering_reason":
            reason
    })


ordering_marker_status = pd.DataFrame(
    ordering_records
)


initial_ordering_membership = (
    ordering_marker_status[
        ordering_marker_status[
            "include_initial_ordering"
        ]
    ]
    .copy()
)


ordering_holdouts = (
    ordering_marker_status[
        ~ordering_marker_status[
            "include_initial_ordering"
        ]
    ]
    .copy()
)


print("INITIAL ORDERING BACKBONE")
print("=" * 110)

print(
    "Markers used for initial ordering:",
    initial_ordering_membership[
        "marker"
    ].nunique()
)

print(
    "Markers held out temporarily:",
    ordering_holdouts[
        "marker"
    ].nunique()
)


print("\nINITIAL ORDERING MARKERS BY GROUP")
print("-" * 80)

display(
    initial_ordering_membership
    .groupby("working_group")
    .size()
    .reset_index(
        name="n_ordering_markers"
    )
    .sort_values(
        "n_ordering_markers",
        ascending=False
    )
)


print("\nHELD-OUT SATELLITE MARKERS")
print("-" * 100)

display(
    ordering_holdouts[
        [
            "working_group",
            "consensus_group",
            "marker",
            "component_role",
            "attachment_rate"
        ]
    ]
    .sort_values(
        [
            "working_group",
            "consensus_group",
            "marker"
        ]
    )
)

### Cell 01.84 — verify core connectivity before ordering
* Before trying to order anything, every initial ordering group should be tested again under the original linkage rule.

In [ ]:
# Cell 01.84
# Check connectivity of each initial ordering group using
# the original genotype-only core linkage criterion.

import networkx as nx


ordering_connectivity_records = []


for group in sorted(
    initial_ordering_membership[
        "working_group"
    ].unique()
):

    markers = (
        initial_ordering_membership.loc[
            initial_ordering_membership[
                "working_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


    G = nx.Graph()

    G.add_nodes_from(
        markers
    )


    internal_pairs = backbone_pairwise[
        (
            backbone_pairwise[
                "marker_a"
            ].isin(markers)
        )
        &
        (
            backbone_pairwise[
                "marker_b"
            ].isin(markers)
        )
        &
        (
            backbone_pairwise[
                "n_overlap"
            ] >= 40
        )
        &
        (
            backbone_pairwise[
                "R_ril"
            ] <= 0.30
        )
        &
        (
            backbone_pairwise[
                "lod"
            ] >= 3.5
        )
    ].copy()


    for _, edge in internal_pairs.iterrows():

        G.add_edge(
            edge["marker_a"],
            edge["marker_b"]
        )


    components = sorted(
        nx.connected_components(G),
        key=len,
        reverse=True
    )


    sizes = [
        len(c)
        for c in components
    ]


    ordering_connectivity_records.append({
        "working_group":
            group,

        "n_markers":
            len(markers),

        "n_core_edges":
            G.number_of_edges(),

        "n_components":
            len(components),

        "largest_component":
            sizes[0],

        "largest_fraction":
            sizes[0] / len(markers),

        "fully_connected":
            len(components) == 1
    })


ordering_connectivity = pd.DataFrame(
    ordering_connectivity_records
)


print("CONNECTIVITY OF INITIAL ORDERING BACKBONE")
print("=" * 115)

display(
    ordering_connectivity.sort_values(
        [
            "fully_connected",
            "largest_fraction",
            "n_markers"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)


print("\nSUMMARY")
print("-" * 70)

print(
    "Fully connected groups:",
    ordering_connectivity[
        "fully_connected"
    ].sum(),
    "/",
    len(ordering_connectivity)
)

### Cell 01.85 — build and validate within-group recombination matrices

In [ ]:
# Cell 01.85
# Build complete pairwise R_ril matrices for each
# initial-ordering linkage group.
#
# No physical-map or historical-map information is used.

import numpy as np
import pandas as pd


ordering_groups = (
    initial_ordering_membership
    .groupby("working_group")["marker"]
    .apply(list)
    .to_dict()
)


R_matrices = {}
LOD_matrices = {}
N_matrices = {}

matrix_validation_records = []


for group, markers in ordering_groups.items():

    markers = list(markers)
    n_markers = len(markers)

    Rmat = pd.DataFrame(
        np.nan,
        index=markers,
        columns=markers
    )

    Lmat = pd.DataFrame(
        np.nan,
        index=markers,
        columns=markers
    )

    Nmat = pd.DataFrame(
        np.nan,
        index=markers,
        columns=markers
    )


    # Diagonal
    for marker in markers:
        Rmat.loc[marker, marker] = 0.0
        Lmat.loc[marker, marker] = np.inf
        Nmat.loc[marker, marker] = 92


    group_pairs = backbone_pairwise[
        (
            backbone_pairwise["marker_a"].isin(markers)
        )
        &
        (
            backbone_pairwise["marker_b"].isin(markers)
        )
    ].copy()


    for _, row in group_pairs.iterrows():

        a = row["marker_a"]
        b = row["marker_b"]

        Rmat.loc[a, b] = row["R_ril"]
        Rmat.loc[b, a] = row["R_ril"]

        Lmat.loc[a, b] = row["lod"]
        Lmat.loc[b, a] = row["lod"]

        Nmat.loc[a, b] = row["n_overlap"]
        Nmat.loc[b, a] = row["n_overlap"]


    expected_pairs = (
        n_markers * (n_markers - 1) // 2
    )

    observed_pairs = len(group_pairs)

    missing_pairs = (
        expected_pairs - observed_pairs
    )


    R_matrices[group] = Rmat
    LOD_matrices[group] = Lmat
    N_matrices[group] = Nmat


    matrix_validation_records.append({
        "working_group": group,
        "n_markers": n_markers,
        "expected_pairs": expected_pairs,
        "observed_pairs": observed_pairs,
        "missing_pairs": missing_pairs,
        "min_overlap":
            group_pairs["n_overlap"].min()
            if len(group_pairs) > 0
            else np.nan,
        "median_overlap":
            group_pairs["n_overlap"].median()
            if len(group_pairs) > 0
            else np.nan
    })


ordering_matrix_validation = pd.DataFrame(
    matrix_validation_records
)


print("WITHIN-GROUP PAIRWISE MATRIX VALIDATION")
print("=" * 110)

display(
    ordering_matrix_validation.sort_values(
        "working_group"
    )
)


print("\nTotal missing within-group pairs:",
      ordering_matrix_validation["missing_pairs"].sum())

### Cell 01.86 — deterministic multi-start path ordering with 2-opt
* This minimizes the sum of recombination fractions between adjacent markers.

In [ ]:
# Cell 01.86
# Initial de novo marker ordering.
#
# Method:
#   1. nearest-neighbor path from every possible start marker
#   2. improve each path with deterministic 2-opt
#   3. retain the path with the smallest total adjacent R_ril
#
# Reverse paths are equivalent genetically.

def path_cost(order, Rmat):

    if len(order) <= 1:
        return 0.0

    return sum(
        Rmat.loc[
            order[i],
            order[i + 1]
        ]
        for i in range(len(order) - 1)
    )


def nearest_neighbor_path(markers, Rmat, start):

    unvisited = set(markers)

    path = [start]
    unvisited.remove(start)

    current = start


    while unvisited:

        next_marker = min(
            unvisited,
            key=lambda m: (
                Rmat.loc[current, m],
                m
            )
        )

        path.append(next_marker)

        unvisited.remove(next_marker)

        current = next_marker


    return path


def two_opt_path(order, Rmat):

    best = list(order)
    best_cost = path_cost(
        best,
        Rmat
    )

    improved = True


    while improved:

        improved = False

        n = len(best)


        for i in range(1, n - 1):

            for j in range(
                i + 1,
                n
            ):

                candidate = (
                    best[:i]
                    +
                    best[i:j + 1][::-1]
                    +
                    best[j + 1:]
                )

                candidate_cost = path_cost(
                    candidate,
                    Rmat
                )


                if candidate_cost < (
                    best_cost - 1e-12
                ):

                    best = candidate
                    best_cost = candidate_cost

                    improved = True

                    break

            if improved:
                break


    return best, best_cost


def canonical_orientation(order):
    """
    Marker order and its reverse are genetically equivalent.
    Store a deterministic orientation for reproducibility.
    """

    forward = tuple(order)
    reverse = tuple(order[::-1])

    return list(
        min(
            forward,
            reverse
        )
    )


def optimize_marker_order(markers, Rmat):

    candidate_paths = []


    for start in sorted(markers):

        initial = nearest_neighbor_path(
            markers,
            Rmat,
            start
        )

        improved, cost = two_opt_path(
            initial,
            Rmat
        )

        improved = canonical_orientation(
            improved
        )

        cost = path_cost(
            improved,
            Rmat
        )


        candidate_paths.append(
            (
                cost,
                tuple(improved)
            )
        )


    candidate_paths = sorted(
        candidate_paths,
        key=lambda x: (
            x[0],
            x[1]
        )
    )


    best_cost, best_order = (
        candidate_paths[0]
    )


    return (
        list(best_order),
        best_cost
    )


print("Ordering functions ready.")

### Cell 01.87 — generate initial order for all 20 groups

In [ ]:
# Cell 01.87
# Generate initial recombination-minimizing marker order
# for every working linkage group.

initial_order_records = []
order_summary_records = []


for group in sorted(ordering_groups):

    markers = ordering_groups[group]

    Rmat = R_matrices[group]
    Lmat = LOD_matrices[group]
    Nmat = N_matrices[group]


    order, total_R = optimize_marker_order(
        markers,
        Rmat
    )


    adjacent_R = []
    adjacent_LOD = []
    adjacent_N = []


    for i in range(
        len(order) - 1
    ):

        a = order[i]
        b = order[i + 1]

        r = Rmat.loc[a, b]
        lod = Lmat.loc[a, b]
        n = Nmat.loc[a, b]

        adjacent_R.append(r)
        adjacent_LOD.append(lod)
        adjacent_N.append(n)


    for position, marker in enumerate(
        order,
        start=1
    ):

        initial_order_records.append({
            "working_group":
                group,

            "order_position":
                position,

            "marker":
                marker
        })


    order_summary_records.append({
        "working_group":
            group,

        "n_markers":
            len(order),

        "total_adjacent_R":
            total_R,

        "mean_adjacent_R":
            np.mean(adjacent_R)
            if adjacent_R
            else np.nan,

        "median_adjacent_R":
            np.median(adjacent_R)
            if adjacent_R
            else np.nan,

        "max_adjacent_R":
            np.max(adjacent_R)
            if adjacent_R
            else np.nan,

        "median_adjacent_lod":
            np.median(adjacent_LOD)
            if adjacent_LOD
            else np.nan,

        "min_adjacent_lod":
            np.min(adjacent_LOD)
            if adjacent_LOD
            else np.nan,

        "min_adjacent_overlap":
            np.min(adjacent_N)
            if adjacent_N
            else np.nan
    })


initial_marker_order = pd.DataFrame(
    initial_order_records
)


initial_order_summary = pd.DataFrame(
    order_summary_records
)


print("INITIAL DE NOVO MARKER ORDERS")
print("=" * 115)

display(
    initial_marker_order
    .groupby("working_group")["marker"]
    .apply(
        lambda x:
            " → ".join(x)
    )
    .reset_index(
        name="marker_order"
    )
)


print("\nORDERING DIAGNOSTICS")
print("=" * 115)

display(
    initial_order_summary.sort_values(
        "max_adjacent_R",
        ascending=False
    )
)

### Cell 01.88 — inspect every adjacent interval
* This will tell us where an apparently good overall order contains a weak local transition.

In [ ]:
# Cell 01.88
# Examine support for every adjacent marker pair in
# the initial de novo orders.

adjacency_records = []


for group in sorted(
    initial_marker_order[
        "working_group"
    ].unique()
):

    order = (
        initial_marker_order.loc[
            initial_marker_order[
                "working_group"
            ] == group
        ]
        .sort_values(
            "order_position"
        )[
            "marker"
        ]
        .tolist()
    )


    Rmat = R_matrices[group]
    Lmat = LOD_matrices[group]
    Nmat = N_matrices[group]


    for i in range(
        len(order) - 1
    ):

        a = order[i]
        b = order[i + 1]

        R = Rmat.loc[a, b]
        lod = Lmat.loc[a, b]
        n = Nmat.loc[a, b]


        if (
            R <= 0.25
            and lod >= 5
        ):

            interval_class = (
                "strong"
            )

        elif (
            R <= 0.30
            and lod >= 3.5
        ):

            interval_class = (
                "core_supported"
            )

        elif (
            R <= 0.325
            and lod >= 2.0
        ):

            interval_class = (
                "weak_supported"
            )

        else:

            interval_class = (
                "poor"
            )


        adjacency_records.append({
            "working_group":
                group,

            "left_position":
                i + 1,

            "right_position":
                i + 2,

            "marker_a":
                a,

            "marker_b":
                b,

            "n_overlap":
                n,

            "R_ril":
                R,

            "lod":
                lod,

            "interval_class":
                interval_class
        })


initial_order_adjacencies = pd.DataFrame(
    adjacency_records
)


print("ADJACENT INTERVAL SUPPORT")
print("=" * 125)

print(
    initial_order_adjacencies[
        "interval_class"
    ].value_counts()
)


print("\nWEAKEST ADJACENT INTERVALS")
print("-" * 125)

display(
    initial_order_adjacencies
    .sort_values(
        [
            "lod",
            "R_ril"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(40)
)


print("\nGROUP-LEVEL ADJACENCY QUALITY")
print("-" * 125)

display(
    initial_order_adjacencies
    .groupby("working_group")
    .agg(
        n_intervals=(
            "marker_a",
            "size"
        ),

        n_strong=(
            "interval_class",
            lambda x:
                (x == "strong").sum()
        ),

        n_core_supported=(
            "interval_class",
            lambda x:
                (x == "core_supported").sum()
        ),

        n_weak_supported=(
            "interval_class",
            lambda x:
                (x == "weak_supported").sum()
        ),

        n_poor=(
            "interval_class",
            lambda x:
                (x == "poor").sum()
        ),

        max_R_ril=(
            "R_ril",
            "max"
        ),

        min_lod=(
            "lod",
            "min"
        )
    )
    .reset_index()
    .sort_values(
        [
            "n_poor",
            "n_weak_supported",
            "max_R_ril"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)

### Cell 01.89 — diagnose whether weak adjacencies are structurally necessary
* This checks the core-linkage degree of the markers involved and lists their strongest alternative partners within the same group.

In [ ]:
# Cell 01.89
# Diagnose poor and weak-supported adjacent intervals.
#
# Question:
# Are these weak jumps forced by graph structure,
# or are stronger alternative neighboring markers available?

problem_intervals = (
    initial_order_adjacencies[
        initial_order_adjacencies[
            "interval_class"
        ].isin(
            [
                "poor",
                "weak_supported"
            ]
        )
    ]
    .copy()
)


problem_marker_records = []


for group in sorted(
    problem_intervals["working_group"].unique()
):

    group_markers = (
        initial_ordering_membership.loc[
            initial_ordering_membership[
                "working_group"
            ] == group,
            "marker"
        ]
        .tolist()
    )


    # Core graph for this group
    core_pairs = backbone_pairwise[
        (
            backbone_pairwise[
                "marker_a"
            ].isin(group_markers)
        )
        &
        (
            backbone_pairwise[
                "marker_b"
            ].isin(group_markers)
        )
        &
        (
            backbone_pairwise[
                "n_overlap"
            ] >= 40
        )
        &
        (
            backbone_pairwise[
                "R_ril"
            ] <= 0.30
        )
        &
        (
            backbone_pairwise[
                "lod"
            ] >= 3.5
        )
    ].copy()


    G = nx.Graph()

    G.add_nodes_from(
        group_markers
    )


    for _, row in core_pairs.iterrows():

        G.add_edge(
            row["marker_a"],
            row["marker_b"],
            R_ril=row["R_ril"],
            lod=row["lod"]
        )


    affected_markers = sorted(
        set(
            problem_intervals.loc[
                problem_intervals[
                    "working_group"
                ] == group,
                [
                    "marker_a",
                    "marker_b"
                ]
            ]
            .to_numpy()
            .ravel()
        )
    )


    for marker in affected_markers:

        all_partners = []


        for partner in group_markers:

            if partner == marker:
                continue


            R = R_matrices[
                group
            ].loc[
                marker,
                partner
            ]

            lod = LOD_matrices[
                group
            ].loc[
                marker,
                partner
            ]

            n = N_matrices[
                group
            ].loc[
                marker,
                partner
            ]


            if (
                R <= 0.25
                and lod >= 5
            ):
                support_class = "strong"
                support_rank = 0

            elif (
                R <= 0.30
                and lod >= 3.5
            ):
                support_class = "core_supported"
                support_rank = 1

            elif (
                R <= 0.325
                and lod >= 2.0
            ):
                support_class = "weak_supported"
                support_rank = 2

            else:
                support_class = "poor"
                support_rank = 3


            all_partners.append({
                "partner":
                    partner,

                "R_ril":
                    R,

                "lod":
                    lod,

                "n_overlap":
                    n,

                "support_class":
                    support_class,

                "support_rank":
                    support_rank
            })


        partner_df = pd.DataFrame(
            all_partners
        ).sort_values(
            [
                "support_rank",
                "R_ril",
                "lod"
            ],
            ascending=[
                True,
                True,
                False
            ]
        )


        best = partner_df.iloc[0]


        problem_marker_records.append({
            "working_group":
                group,

            "marker":
                marker,

            "core_degree":
                G.degree(marker),

            "best_available_partner":
                best["partner"],

            "best_available_class":
                best["support_class"],

            "best_available_R":
                best["R_ril"],

            "best_available_lod":
                best["lod"]
        })


problem_marker_diagnostics = pd.DataFrame(
    problem_marker_records
)


print("DIAGNOSTICS FOR MARKERS IN WEAK/POOR ADJACENCIES")
print("=" * 125)

display(
    problem_marker_diagnostics.sort_values(
        [
            "working_group",
            "core_degree",
            "best_available_R"
        ]
    )
)

### Cell 01.90 — define a support-aware ordering objective
* We now optimize paths lexicographically:
    - fewest poor intervals;
    - fewest weak_supported intervals;
    - fewest merely core_supported intervals;
    - lowest total R_RIL.
* Thus a tiny improvement in total recombination can no longer justify introducing an unsupported jump.

In [ ]:
# Cell 01.90
# Support-aware ordering objective.
#
# Lexicographic priority:
#   1. minimize poor intervals
#   2. minimize weak-supported intervals
#   3. minimize core-supported intervals
#      (therefore maximize strong intervals)
#   4. minimize total adjacent R_ril


def edge_support_class(
    group,
    marker_a,
    marker_b
):

    R = R_matrices[
        group
    ].loc[
        marker_a,
        marker_b
    ]

    lod = LOD_matrices[
        group
    ].loc[
        marker_a,
        marker_b
    ]


    if (
        R <= 0.25
        and lod >= 5
    ):
        return "strong"

    elif (
        R <= 0.30
        and lod >= 3.5
    ):
        return "core_supported"

    elif (
        R <= 0.325
        and lod >= 2.0
    ):
        return "weak_supported"

    else:
        return "poor"


def support_aware_path_score(
    group,
    order
):

    n_poor = 0
    n_weak = 0
    n_core = 0
    total_R = 0.0


    for i in range(
        len(order) - 1
    ):

        a = order[i]
        b = order[i + 1]

        cls = edge_support_class(
            group,
            a,
            b
        )

        R = R_matrices[
            group
        ].loc[
            a,
            b
        ]

        total_R += R


        if cls == "poor":
            n_poor += 1

        elif cls == "weak_supported":
            n_weak += 1

        elif cls == "core_supported":
            n_core += 1


    return (
        n_poor,
        n_weak,
        n_core,
        total_R
    )


def support_aware_nearest_neighbor(
    group,
    markers,
    start
):

    class_rank = {
        "strong": 0,
        "core_supported": 1,
        "weak_supported": 2,
        "poor": 3
    }


    unvisited = set(
        markers
    )

    path = [
        start
    ]

    unvisited.remove(
        start
    )

    current = start


    while unvisited:

        next_marker = min(
            unvisited,
            key=lambda m: (
                class_rank[
                    edge_support_class(
                        group,
                        current,
                        m
                    )
                ],
                R_matrices[
                    group
                ].loc[
                    current,
                    m
                ],
                -LOD_matrices[
                    group
                ].loc[
                    current,
                    m
                ],
                m
            )
        )


        path.append(
            next_marker
        )

        unvisited.remove(
            next_marker
        )

        current = next_marker


    return path


def support_aware_two_opt(
    group,
    order
):

    best = list(
        order
    )

    best_score = (
        support_aware_path_score(
            group,
            best
        )
    )


    improved = True


    while improved:

        improved = False

        n = len(
            best
        )


        for i in range(
            1,
            n - 1
        ):

            for j in range(
                i + 1,
                n
            ):

                candidate = (
                    best[:i]
                    +
                    best[i:j + 1][::-1]
                    +
                    best[j + 1:]
                )


                score = (
                    support_aware_path_score(
                        group,
                        candidate
                    )
                )


                if score < best_score:

                    best = candidate
                    best_score = score

                    improved = True
                    break


            if improved:
                break


    return (
        best,
        best_score
    )


def optimize_support_aware_order(
    group,
    markers
):

    candidates = []


    for start in sorted(
        markers
    ):

        initial = (
            support_aware_nearest_neighbor(
                group,
                markers,
                start
            )
        )


        improved, score = (
            support_aware_two_opt(
                group,
                initial
            )
        )


        improved = canonical_orientation(
            improved
        )


        score = support_aware_path_score(
            group,
            improved
        )


        candidates.append(
            (
                score,
                tuple(improved)
            )
        )


    candidates = sorted(
        candidates,
        key=lambda x: (
            x[0],
            x[1]
        )
    )


    best_score, best_order = (
        candidates[0]
    )


    return (
        list(best_order),
        best_score
    )


print(
    "Support-aware ordering functions ready."
)

### Cell 01.91 — calculate support-aware orders and compare with the original orders

In [ ]:
# Cell 01.91
# Reorder all 20 groups using the support-aware objective
# and compare against the original minimum-R solution.

support_order_records = []
support_order_summary_records = []


for group in sorted(
    ordering_groups
):

    markers = ordering_groups[
        group
    ]


    new_order, new_score = (
        optimize_support_aware_order(
            group,
            markers
        )
    )


    old_order = (
        initial_marker_order.loc[
            initial_marker_order[
                "working_group"
            ] == group
        ]
        .sort_values(
            "order_position"
        )[
            "marker"
        ]
        .tolist()
    )


    old_score = (
        support_aware_path_score(
            group,
            old_order
        )
    )


    for position, marker in enumerate(
        new_order,
        start=1
    ):

        support_order_records.append({
            "working_group":
                group,

            "order_position":
                position,

            "marker":
                marker
        })


    support_order_summary_records.append({
        "working_group":
            group,

        "n_markers":
            len(markers),

        "old_n_poor":
            old_score[0],

        "new_n_poor":
            new_score[0],

        "old_n_weak":
            old_score[1],

        "new_n_weak":
            new_score[1],

        "old_n_core":
            old_score[2],

        "new_n_core":
            new_score[2],

        "old_total_R":
            old_score[3],

        "new_total_R":
            new_score[3],

        "poor_improvement":
            old_score[0]
            -
            new_score[0],

        "weak_improvement":
            old_score[1]
            -
            new_score[1]
    })


support_aware_marker_order = pd.DataFrame(
    support_order_records
)


support_aware_order_summary = pd.DataFrame(
    support_order_summary_records
)


print("SUPPORT-AWARE ORDERING COMPARISON")
print("=" * 130)

display(
    support_aware_order_summary.sort_values(
        [
            "poor_improvement",
            "weak_improvement"
        ],
        ascending=[
            False,
            False
        ]
    )
)


print("\nSUPPORT-AWARE MARKER ORDERS")
print("=" * 130)

display(
    support_aware_marker_order
    .groupby(
        "working_group"
    )[
        "marker"
    ]
    .apply(
        lambda x:
            " → ".join(x)
    )
    .reset_index(
        name="marker_order"
    )
)

### Cell 01.92 — evaluate the revised adjacent intervals

In [ ]:
# Cell 01.92
# Evaluate adjacency quality after support-aware ordering.

support_adjacency_records = []


for group in sorted(
    support_aware_marker_order[
        "working_group"
    ].unique()
):

    order = (
        support_aware_marker_order.loc[
            support_aware_marker_order[
                "working_group"
            ] == group
        ]
        .sort_values(
            "order_position"
        )[
            "marker"
        ]
        .tolist()
    )


    for i in range(
        len(order) - 1
    ):

        a = order[i]
        b = order[i + 1]


        R = R_matrices[
            group
        ].loc[
            a,
            b
        ]

        lod = LOD_matrices[
            group
        ].loc[
            a,
            b
        ]

        n = N_matrices[
            group
        ].loc[
            a,
            b
        ]


        cls = edge_support_class(
            group,
            a,
            b
        )


        support_adjacency_records.append({
            "working_group":
                group,

            "left_position":
                i + 1,

            "right_position":
                i + 2,

            "marker_a":
                a,

            "marker_b":
                b,

            "n_overlap":
                n,

            "R_ril":
                R,

            "lod":
                lod,

            "interval_class":
                cls
        })


support_aware_adjacencies = pd.DataFrame(
    support_adjacency_records
)


print("SUPPORT-AWARE ADJACENT INTERVALS")
print("=" * 125)

print(
    support_aware_adjacencies[
        "interval_class"
    ].value_counts()
)


print("\nWEAKEST REMAINING INTERVALS")
print("-" * 125)

display(
    support_aware_adjacencies
    .sort_values(
        [
            "lod",
            "R_ril"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(40)
)


print("\nGROUP-LEVEL SUPPORT-AWARE ORDER QUALITY")
print("-" * 130)

support_group_quality = (
    support_aware_adjacencies
    .groupby(
        "working_group"
    )
    .agg(
        n_intervals=(
            "marker_a",
            "size"
        ),

        n_strong=(
            "interval_class",
            lambda x:
                (x == "strong").sum()
        ),

        n_core_supported=(
            "interval_class",
            lambda x:
                (
                    x
                    ==
                    "core_supported"
                ).sum()
        ),

        n_weak_supported=(
            "interval_class",
            lambda x:
                (
                    x
                    ==
                    "weak_supported"
                ).sum()
        ),

        n_poor=(
            "interval_class",
            lambda x:
                (x == "poor").sum()
        ),

        max_R_ril=(
            "R_ril",
            "max"
        ),

        min_lod=(
            "lod",
            "min"
        )
    )
    .reset_index()
)


display(
    support_group_quality.sort_values(
        [
            "n_poor",
            "n_weak_supported",
            "max_R_ril"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)

### Cell 01.93 — freeze candidate framework order v1

In [ ]:
# Cell 01.93
# Freeze the current support-aware order as candidate framework v1.
#
# "Freeze" here means preserve it as a reproducible checkpoint.
# It is NOT yet the final genetic map.

framework_order_v1 = (
    support_aware_marker_order
    .copy()
    .sort_values(
        [
            "working_group",
            "order_position"
        ]
    )
    .reset_index(drop=True)
)


framework_adjacencies_v1 = (
    support_aware_adjacencies
    .copy()
    .sort_values(
        [
            "working_group",
            "left_position"
        ]
    )
    .reset_index(drop=True)
)


print("CANDIDATE FRAMEWORK ORDER V1")
print("=" * 100)

print(
    "Linkage groups:",
    framework_order_v1[
        "working_group"
    ].nunique()
)

print(
    "Framework markers:",
    framework_order_v1[
        "marker"
    ].nunique()
)

print(
    "Adjacent intervals:",
    len(
        framework_adjacencies_v1
    )
)


print("\nINTERVAL CLASSES")
print("-" * 60)

print(
    framework_adjacencies_v1[
        "interval_class"
    ].value_counts()
)


print("\nGROUPS CONTAINING WEAK-SUPPORTED INTERVALS")
print("-" * 80)

display(
    framework_adjacencies_v1[
        framework_adjacencies_v1[
            "interval_class"
        ] == "weak_supported"
    ][
        [
            "working_group",
            "left_position",
            "right_position",
            "marker_a",
            "marker_b",
            "n_overlap",
            "R_ril",
            "lod"
        ]
    ]
)

### Cell 01.94 — bootstrap all 146 framework adjacencies
* Here we distinguish two useful criteria.
* **Core pass:** R ≤ 0.30, LOD ≥ 3.5
* **Supporting pass:** R ≤ 0.325, LOD ≥ 2.0
* A weak interval may rarely pass the core rule but still be highly reproducible as a supporting interval.

In [ ]:
# Cell 01.94
# Bootstrap all candidate-framework adjacent intervals.
#
# This does NOT reorder markers.
# It asks how stable the evidence is for each chosen adjacency.

N_ADJ_BOOT = 500
ADJ_BOOT_SEED = 20260910

rng_adj = np.random.default_rng(
    ADJ_BOOT_SEED
)


adj_boot_records = []

n_rils = len(
    mapping_geno
)


for boot_id in range(
    1,
    N_ADJ_BOOT + 1
):

    sample_idx = rng_adj.integers(
        0,
        n_rils,
        size=n_rils
    )


    for _, interval in framework_adjacencies_v1.iterrows():

        group = interval["working_group"]
        a = interval["marker_a"]
        b = interval["marker_b"]


        n, R, lod = (
            pair_stats_bootstrap_arrays(
                a,
                b,
                sample_idx
            )
        )


        pass_core = (
            n >= 40
            and R <= 0.30
            and lod >= 3.5
        )


        pass_supporting = (
            n >= 40
            and R <= 0.325
            and lod >= 2.0
        )


        pass_strong = (
            n >= 40
            and R <= 0.25
            and lod >= 5.0
        )


        adj_boot_records.append({
            "bootstrap_id":
                boot_id,

            "working_group":
                group,

            "marker_a":
                a,

            "marker_b":
                b,

            "n_overlap":
                n,

            "R_ril":
                R,

            "lod":
                lod,

            "pass_strong":
                pass_strong,

            "pass_core":
                pass_core,

            "pass_supporting":
                pass_supporting
        })


framework_adjacency_bootstrap = pd.DataFrame(
    adj_boot_records
)


print(
    "Framework adjacency bootstrap completed."
)

print(
    "Bootstrap replicates:",
    N_ADJ_BOOT
)

print(
    "Intervals evaluated:",
    framework_adjacencies_v1.shape[0]
)

### Cell 01.95 — summarize bootstrap stability of every interval

In [ ]:
# Cell 01.95
# Summarize bootstrap stability for every framework adjacency.

adjacency_bootstrap_summary = (
    framework_adjacency_bootstrap
    .groupby(
        [
            "working_group",
            "marker_a",
            "marker_b"
        ]
    )
    .agg(
        n_boot=(
            "bootstrap_id",
            "size"
        ),

        strong_pass_rate=(
            "pass_strong",
            "mean"
        ),

        core_pass_rate=(
            "pass_core",
            "mean"
        ),

        supporting_pass_rate=(
            "pass_supporting",
            "mean"
        ),

        R_q025=(
            "R_ril",
            lambda x:
                x.quantile(0.025)
        ),

        R_median=(
            "R_ril",
            "median"
        ),

        R_q975=(
            "R_ril",
            lambda x:
                x.quantile(0.975)
        ),

        lod_q025=(
            "lod",
            lambda x:
                x.quantile(0.025)
        ),

        lod_median=(
            "lod",
            "median"
        ),

        lod_q975=(
            "lod",
            lambda x:
                x.quantile(0.975)
        )
    )
    .reset_index()
)


# Add original interval statistics
adjacency_bootstrap_summary = (
    framework_adjacencies_v1[
        [
            "working_group",
            "marker_a",
            "marker_b",
            "R_ril",
            "lod",
            "interval_class"
        ]
    ]
    .merge(
        adjacency_bootstrap_summary,
        on=[
            "working_group",
            "marker_a",
            "marker_b"
        ],
        how="left"
    )
)


def bootstrap_interval_class(row):

    support = (
        row["supporting_pass_rate"]
    )

    core = (
        row["core_pass_rate"]
    )


    if core >= 0.80:
        return "very_stable_core"

    elif core >= 0.60:
        return "stable_core"

    elif support >= 0.80:
        return "stable_supporting"

    elif support >= 0.60:
        return "moderate_support"

    else:
        return "unstable_interval"


adjacency_bootstrap_summary[
    "bootstrap_interval_class"
] = (
    adjacency_bootstrap_summary.apply(
        bootstrap_interval_class,
        axis=1
    )
)


print("FRAMEWORK ADJACENCY BOOTSTRAP SUMMARY")
print("=" * 145)

print(
    adjacency_bootstrap_summary[
        "bootstrap_interval_class"
    ].value_counts()
)


print("\nLEAST STABLE FRAMEWORK INTERVALS")
print("-" * 145)

display(
    adjacency_bootstrap_summary
    .sort_values(
        [
            "supporting_pass_rate",
            "core_pass_rate",
            "lod_median"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
    .head(30)
)

### Cell 01.96 — group-level order stability and targeted review list

In [ ]:
# Cell 01.96
# Summarize adjacency stability by linkage group and
# identify intervals requiring targeted review.

group_order_bootstrap_summary = (
    adjacency_bootstrap_summary
    .groupby(
        "working_group"
    )
    .agg(
        n_intervals=(
            "marker_a",
            "size"
        ),

        median_core_pass_rate=(
            "core_pass_rate",
            "median"
        ),

        min_core_pass_rate=(
            "core_pass_rate",
            "min"
        ),

        median_supporting_pass_rate=(
            "supporting_pass_rate",
            "median"
        ),

        min_supporting_pass_rate=(
            "supporting_pass_rate",
            "min"
        ),

        n_unstable=(
            "bootstrap_interval_class",
            lambda x:
                (
                    x
                    ==
                    "unstable_interval"
                ).sum()
        ),

        n_moderate_support=(
            "bootstrap_interval_class",
            lambda x:
                (
                    x
                    ==
                    "moderate_support"
                ).sum()
        )
    )
    .reset_index()
)


print("GROUP-LEVEL FRAMEWORK ORDER STABILITY")
print("=" * 125)

display(
    group_order_bootstrap_summary
    .sort_values(
        [
            "n_unstable",
            "min_supporting_pass_rate",
            "min_core_pass_rate"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)


review_intervals = (
    adjacency_bootstrap_summary[
        (
            adjacency_bootstrap_summary[
                "supporting_pass_rate"
            ] < 0.60
        )
        |
        (
            adjacency_bootstrap_summary[
                "bootstrap_interval_class"
            ] == "unstable_interval"
        )
    ]
    .copy()
)


print("\nINTERVALS REQUIRING TARGETED REVIEW")
print("-" * 135)

print(
    "Number requiring review:",
    len(review_intervals)
)


display(
    review_intervals[
        [
            "working_group",
            "marker_a",
            "marker_b",
            "R_ril",
            "lod",
            "interval_class",
            "strong_pass_rate",
            "core_pass_rate",
            "supporting_pass_rate",
            "R_median",
            "lod_median",
            "bootstrap_interval_class"
        ]
    ]
    .sort_values(
        [
            "supporting_pass_rate",
            "core_pass_rate"
        ]
    )
)

### Cell 01.97 — inspect the local neighborhood around the unstable pLG02 interval

In [ ]:
# Cell 01.97
# Examine all pairwise relationships involving the markers
# around the unstable pLG02 adjacency.

target_group = "pLG02"

target_markers = [
    "Satt163c",
    "OG13_490",
    "OG13"
]


plg02_order = (
    framework_order_v1.loc[
        framework_order_v1[
            "working_group"
        ] == target_group
    ]
    .sort_values(
        "order_position"
    )
    .reset_index(drop=True)
)


print("CURRENT pLG02 ORDER")
print("=" * 110)

display(
    plg02_order
)


local_records = []


for marker in target_markers:

    partners = [
        m
        for m in plg02_order["marker"]
        if m != marker
    ]


    for partner in partners:

        R = R_matrices[
            target_group
        ].loc[
            marker,
            partner
        ]

        lod = LOD_matrices[
            target_group
        ].loc[
            marker,
            partner
        ]

        n = N_matrices[
            target_group
        ].loc[
            marker,
            partner
        ]


        cls = edge_support_class(
            target_group,
            marker,
            partner
        )


        local_records.append({
            "marker":
                marker,

            "partner":
                partner,

            "n_overlap":
                n,

            "R_ril":
                R,

            "lod":
                lod,

            "support_class":
                cls
        })


plg02_local_links = pd.DataFrame(
    local_records
)


print("\nSTRONGEST AVAILABLE PARTNERS")
print("=" * 120)

display(
    plg02_local_links
    .sort_values(
        [
            "marker",
            "R_ril",
            "lod"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .groupby(
        "marker",
        group_keys=False
    )
    .head(12)
)

### Cell 01.98 — test every possible insertion position for OG13_490
* Here we temporarily remove OG13_490 from pLG02 and insert it into every possible position while keeping the relative order of the other 32 markers unchanged.

In [ ]:
# Cell 01.98
# Exhaustive single-marker insertion test for OG13_490.

marker_to_test = "OG13_490"


base_order = (
    plg02_order[
        plg02_order[
            "marker"
        ] != marker_to_test
    ][
        "marker"
    ]
    .tolist()
)


insertion_records = []


for insert_position in range(
    len(base_order) + 1
):

    candidate = (
        base_order[:insert_position]
        +
        [marker_to_test]
        +
        base_order[insert_position:]
    )


    score = support_aware_path_score(
        target_group,
        candidate
    )


    # Identify immediate flanking markers
    left_marker = (
        candidate[
            insert_position - 1
        ]
        if insert_position > 0
        else None
    )

    right_marker = (
        candidate[
            insert_position + 1
        ]
        if insert_position < len(candidate) - 1
        else None
    )


    flank_details = []


    for flank in [
        left_marker,
        right_marker
    ]:

        if flank is None:
            continue


        flank_details.append({
            "flank":
                flank,

            "R":
                R_matrices[
                    target_group
                ].loc[
                    marker_to_test,
                    flank
                ],

            "lod":
                LOD_matrices[
                    target_group
                ].loc[
                    marker_to_test,
                    flank
                ],

            "class":
                edge_support_class(
                    target_group,
                    marker_to_test,
                    flank
                )
        })


    insertion_records.append({
        "insert_position":
            insert_position + 1,

        "left_marker":
            left_marker,

        "right_marker":
            right_marker,

        "n_poor":
            score[0],

        "n_weak":
            score[1],

        "n_core":
            score[2],

        "total_R":
            score[3],

        "left_R":
            (
                flank_details[0]["R"]
                if len(flank_details) >= 1
                else np.nan
            ),

        "left_lod":
            (
                flank_details[0]["lod"]
                if len(flank_details) >= 1
                else np.nan
            ),

        "left_class":
            (
                flank_details[0]["class"]
                if len(flank_details) >= 1
                else None
            ),

        "right_R":
            (
                flank_details[1]["R"]
                if len(flank_details) == 2
                else np.nan
            ),

        "right_lod":
            (
                flank_details[1]["lod"]
                if len(flank_details) == 2
                else np.nan
            ),

        "right_class":
            (
                flank_details[1]["class"]
                if len(flank_details) == 2
                else None
            )
    })


og13_490_insertion_test = pd.DataFrame(
    insertion_records
)


print("OG13_490 INSERTION TEST")
print("=" * 140)

display(
    og13_490_insertion_test
    .sort_values(
        [
            "n_poor",
            "n_weak",
            "n_core",
            "total_R"
        ]
    )
    .head(15)
)

### Cell 01.99 — repeat the insertion test for OG13

In [ ]:
# Cell 01.99
# Exhaustive single-marker insertion test for OG13.

marker_to_test = "OG13"


base_order = (
    plg02_order[
        plg02_order[
            "marker"
        ] != marker_to_test
    ][
        "marker"
    ]
    .tolist()
)


insertion_records = []


for insert_position in range(
    len(base_order) + 1
):

    candidate = (
        base_order[:insert_position]
        +
        [marker_to_test]
        +
        base_order[insert_position:]
    )


    score = support_aware_path_score(
        target_group,
        candidate
    )


    left_marker = (
        candidate[
            insert_position - 1
        ]
        if insert_position > 0
        else None
    )

    right_marker = (
        candidate[
            insert_position + 1
        ]
        if insert_position < len(candidate) - 1
        else None
    )


    def flank_stats(flank):

        if flank is None:

            return (
                np.nan,
                np.nan,
                None
            )


        return (
            R_matrices[
                target_group
            ].loc[
                marker_to_test,
                flank
            ],

            LOD_matrices[
                target_group
            ].loc[
                marker_to_test,
                flank
            ],

            edge_support_class(
                target_group,
                marker_to_test,
                flank
            )
        )


    left_R, left_lod, left_class = (
        flank_stats(
            left_marker
        )
    )

    right_R, right_lod, right_class = (
        flank_stats(
            right_marker
        )
    )


    insertion_records.append({
        "insert_position":
            insert_position + 1,

        "left_marker":
            left_marker,

        "right_marker":
            right_marker,

        "n_poor":
            score[0],

        "n_weak":
            score[1],

        "n_core":
            score[2],

        "total_R":
            score[3],

        "left_R":
            left_R,

        "left_lod":
            left_lod,

        "left_class":
            left_class,

        "right_R":
            right_R,

        "right_lod":
            right_lod,

        "right_class":
            right_class
    })


og13_insertion_test = pd.DataFrame(
    insertion_records
)


print("OG13 INSERTION TEST")
print("=" * 140)

display(
    og13_insertion_test
    .sort_values(
        [
            "n_poor",
            "n_weak",
            "n_core",
            "total_R"
        ]
    )
    .head(15)
)

### Cell 02.00 — compare current pLG02 order with the best local alternatives
* This cell identifies the best insertion-based alternatives and compares them directly with the current framework.

In [ ]:
# Cell 02.00
# Compare current pLG02 order against the best local
# single-marker insertion alternatives.

current_order = (
    plg02_order[
        "marker"
    ]
    .tolist()
)


current_score = (
    support_aware_path_score(
        target_group,
        current_order
    )
)


best_490 = (
    og13_490_insertion_test
    .sort_values(
        [
            "n_poor",
            "n_weak",
            "n_core",
            "total_R"
        ]
    )
    .iloc[0]
)


best_og13 = (
    og13_insertion_test
    .sort_values(
        [
            "n_poor",
            "n_weak",
            "n_core",
            "total_R"
        ]
    )
    .iloc[0]
)


comparison = pd.DataFrame([
    {
        "scenario":
            "current_framework_v1",

        "n_poor":
            current_score[0],

        "n_weak":
            current_score[1],

        "n_core":
            current_score[2],

        "total_R":
            current_score[3],

        "tested_marker":
            None,

        "best_position":
            None,

        "left_marker":
            None,

        "right_marker":
            None
    },

    {
        "scenario":
            "best_reinsert_OG13_490",

        "n_poor":
            best_490[
                "n_poor"
            ],

        "n_weak":
            best_490[
                "n_weak"
            ],

        "n_core":
            best_490[
                "n_core"
            ],

        "total_R":
            best_490[
                "total_R"
            ],

        "tested_marker":
            "OG13_490",

        "best_position":
            best_490[
                "insert_position"
            ],

        "left_marker":
            best_490[
                "left_marker"
            ],

        "right_marker":
            best_490[
                "right_marker"
            ]
    },

    {
        "scenario":
            "best_reinsert_OG13",

        "n_poor":
            best_og13[
                "n_poor"
            ],

        "n_weak":
            best_og13[
                "n_weak"
            ],

        "n_core":
            best_og13[
                "n_core"
            ],

        "total_R":
            best_og13[
                "total_R"
            ],

        "tested_marker":
            "OG13",

        "best_position":
            best_og13[
                "insert_position"
            ],

        "left_marker":
            best_og13[
                "left_marker"
            ],

        "right_marker":
            best_og13[
                "right_marker"
            ]
    }
])


print("pLG02 LOCAL ORDER COMPARISON")
print("=" * 120)

display(
    comparison
)

### Cell 02.01 — convert RIL recombination to provisional map distance
* Because this population is described as an F5-derived RIL population but we have not yet nailed down the exact generation at genotyping, I recommend using the infinitely selfed-RIL approximation for now and labeling the resulting distances provisional.
* For an infinitely selfed RIL:

$$ r=\frac{R}{2(1-R)} $$

* where \(R\) is observed RIL recombination and \(r\) is the inferred meiotic recombination fraction.
*Then we calculate both Haldane and Kosambi distances.

In [ ]:
# Cell 02.01
# Build provisional genetic-map intervals from framework order v1.
#
# IMPORTANT:
# Distances use the infinitely-selfed RIL approximation:
#
#     r = R / [2(1 - R)]
#
# These cM values are PROVISIONAL until the exact RIL generation
# at genotyping is established.

import numpy as np
import pandas as pd


def ril_R_to_meiotic_r(R):

    R = np.asarray(R, dtype=float)

    r = R / (
        2.0 * (1.0 - R)
    )

    # Numerical safeguard
    r = np.clip(
        r,
        0.0,
        0.499999
    )

    return r


def haldane_cm(r):

    return (
        -50.0
        *
        np.log(
            1.0 - 2.0 * r
        )
    )


def kosambi_cm(r):

    return (
        25.0
        *
        np.log(
            (1.0 + 2.0 * r)
            /
            (1.0 - 2.0 * r)
        )
    )


map_interval_records = []


for _, row in framework_adjacencies_v1.iterrows():

    R = float(
        row["R_ril"]
    )

    r = float(
        ril_R_to_meiotic_r(
            R
        )
    )


    map_interval_records.append({
        "working_group":
            row["working_group"],

        "left_position":
            row["left_position"],

        "right_position":
            row["right_position"],

        "marker_a":
            row["marker_a"],

        "marker_b":
            row["marker_b"],

        "n_overlap":
            row["n_overlap"],

        "R_ril":
            R,

        "meiotic_r_provisional":
            r,

        "lod":
            row["lod"],

        "interval_class":
            row["interval_class"],

        "haldane_cm_provisional":
            float(
                haldane_cm(r)
            ),

        "kosambi_cm_provisional":
            float(
                kosambi_cm(r)
            ),

        "uncertain_interval":
            (
                row["working_group"] == "pLG02"
                and
                row["marker_a"] == "OG13_490"
                and
                row["marker_b"] == "OG13"
            )
    })


provisional_map_intervals = pd.DataFrame(
    map_interval_records
)


print("PROVISIONAL GENETIC-MAP INTERVALS")
print("=" * 125)

display(
    provisional_map_intervals.head(20)
)


print(
    "\nTotal intervals:",
    len(provisional_map_intervals)
)

### Cell 02.02 — calculate cumulative marker positions
* I suggest using Kosambi as the displayed provisional map coordinate for now, while retaining Haldane in the table.

In [ ]:
# Cell 02.02
# Convert interval distances into cumulative marker positions.

map_marker_records = []


for group in sorted(
    framework_order_v1[
        "working_group"
    ].unique()
):

    order = (
        framework_order_v1.loc[
            framework_order_v1[
                "working_group"
            ] == group
        ]
        .sort_values(
            "order_position"
        )[
            "marker"
        ]
        .tolist()
    )


    group_intervals = (
        provisional_map_intervals.loc[
            provisional_map_intervals[
                "working_group"
            ] == group
        ]
        .sort_values(
            "left_position"
        )
        .reset_index(drop=True)
    )


    cumulative_haldane = 0.0
    cumulative_kosambi = 0.0


    # First marker
    map_marker_records.append({
        "working_group":
            group,

        "order_position":
            1,

        "marker":
            order[0],

        "haldane_cm_provisional":
            0.0,

        "kosambi_cm_provisional":
            0.0
    })


    for i, interval in group_intervals.iterrows():

        cumulative_haldane += (
            interval[
                "haldane_cm_provisional"
            ]
        )

        cumulative_kosambi += (
            interval[
                "kosambi_cm_provisional"
            ]
        )


        map_marker_records.append({
            "working_group":
                group,

            "order_position":
                i + 2,

            "marker":
                interval[
                    "marker_b"
                ],

            "haldane_cm_provisional":
                cumulative_haldane,

            "kosambi_cm_provisional":
                cumulative_kosambi
        })


provisional_framework_map = pd.DataFrame(
    map_marker_records
)


print("PROVISIONAL FRAMEWORK GENETIC MAP")
print("=" * 115)

display(
    provisional_framework_map.head(40)
)


print(
    "\nMarkers mapped:",
    provisional_framework_map[
        "marker"
    ].nunique()
)

print(
    "Linkage groups:",
    provisional_framework_map[
        "working_group"
    ].nunique()
)

### Cell 02.03 — map summary for all 20 linkage groups

In [ ]:
# Cell 02.03
# Summarize the provisional 20-group framework map.

map_group_summary = (
    provisional_framework_map
    .groupby(
        "working_group"
    )
    .agg(
        n_markers=(
            "marker",
            "size"
        ),

        map_length_haldane_cm=(
            "haldane_cm_provisional",
            "max"
        ),

        map_length_kosambi_cm=(
            "kosambi_cm_provisional",
            "max"
        )
    )
    .reset_index()
)


interval_summary = (
    provisional_map_intervals
    .groupby(
        "working_group"
    )
    .agg(
        n_intervals=(
            "marker_a",
            "size"
        ),

        median_interval_kosambi_cm=(
            "kosambi_cm_provisional",
            "median"
        ),

        max_interval_kosambi_cm=(
            "kosambi_cm_provisional",
            "max"
        ),

        min_interval_lod=(
            "lod",
            "min"
        ),

        n_uncertain_intervals=(
            "uncertain_interval",
            "sum"
        )
    )
    .reset_index()
)


provisional_map_summary = (
    map_group_summary
    .merge(
        interval_summary,
        on="working_group",
        how="left"
    )
)


print("PROVISIONAL DE NOVO GENETIC MAP SUMMARY")
print("=" * 130)

display(
    provisional_map_summary
    .sort_values(
        "working_group"
    )
)


print("\nWHOLE FRAMEWORK")
print("-" * 70)

print(
    "Linkage groups:",
    len(
        provisional_map_summary
    )
)

print(
    "Markers:",
    provisional_framework_map[
        "marker"
    ].nunique()
)

print(
    "Total Kosambi map length:",
    round(
        provisional_map_summary[
            "map_length_kosambi_cm"
        ].sum(),
        2
    ),
    "cM"
)

print(
    "Total Haldane map length:",
    round(
        provisional_map_summary[
            "map_length_haldane_cm"
        ].sum(),
        2
    ),
    "cM"
)

print(
    "Intervals explicitly flagged uncertain:",
    provisional_map_intervals[
        "uncertain_interval"
    ].sum()
)

### Cell 02.04 — draw the actual genetic map
* This will finally give you a visual map of all 20 provisional linkage groups.

In [ ]:
# Cell 02.04
# Plot the provisional 20-linkage-group framework map.

import matplotlib.pyplot as plt


groups = sorted(
    provisional_framework_map[
        "working_group"
    ].unique()
)


fig, ax = plt.subplots(
    figsize=(18, 14)
)


x_positions = {
    group: i
    for i, group in enumerate(
        groups,
        start=1
    )
}


for group in groups:

    temp = (
        provisional_framework_map.loc[
            provisional_framework_map[
                "working_group"
            ] == group
        ]
        .sort_values(
            "kosambi_cm_provisional"
        )
    )


    x = x_positions[group]

    max_cm = (
        temp[
            "kosambi_cm_provisional"
        ].max()
    )


    # LG vertical line
    ax.plot(
        [x, x],
        [0, max_cm],
        linewidth=2
    )


    # Marker ticks
    for _, row in temp.iterrows():

        y = row[
            "kosambi_cm_provisional"
        ]

        ax.plot(
            [
                x - 0.08,
                x + 0.08
            ],
            [
                y,
                y
            ],
            linewidth=1
        )


ax.set_xticks(
    list(
        x_positions.values()
    )
)

ax.set_xticklabels(
    groups,
    rotation=45,
    ha="right"
)

ax.set_ylabel(
    "Provisional Kosambi position (cM)"
)

ax.set_xlabel(
    "Provisional linkage group"
)

ax.set_title(
    "Flyer × Hartwig RIL Population\n"
    "Provisional De Novo Framework Genetic Map — 166 Markers"
)

ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Cell 02.05 — prepare the 16 satellite markers for insertion

In [ ]:
# Cell 02.05
# Prepare the 16 held-out satellite markers for ordered-map insertion.

satellite_candidates = (
    ordering_holdouts[
        [
            "working_group",
            "consensus_group",
            "marker",
            "component_role",
            "attachment_rate"
        ]
    ]
    .copy()
    .sort_values(
        [
            "working_group",
            "consensus_group",
            "marker"
        ]
    )
    .reset_index(drop=True)
)


print("SATELLITE MARKERS TO TEST FOR INSERTION")
print("=" * 110)

display(
    satellite_candidates
)


print(
    "\nNumber of satellites:",
    len(satellite_candidates)
)

print(
    "\nBy working group:"
)

print(
    satellite_candidates[
        "working_group"
    ].value_counts()
)

### Cell 02.06 — score every possible insertion position
* This evaluates each satellite at every possible gap in its assigned working linkage group.
* The important quantity is the incremental path cost:
$$ \Delta R = R(L,M)+R(M,R)-R(L,R) $$
* for internal positions.
* Smaller $$ \Delta R $$ means the marker fits more naturally between those flanking markers.

In [ ]:
# Cell 02.06
# Exhaustively test every possible insertion position
# for each held-out satellite marker.

satellite_insertion_records = []


for _, sat_row in satellite_candidates.iterrows():

    group = sat_row["working_group"]
    marker = sat_row["marker"]


    framework_order = (
        framework_order_v1.loc[
            framework_order_v1[
                "working_group"
            ] == group
        ]
        .sort_values(
            "order_position"
        )[
            "marker"
        ]
        .tolist()
    )


    for insert_idx in range(
        len(framework_order) + 1
    ):

        left_marker = (
            framework_order[
                insert_idx - 1
            ]
            if insert_idx > 0
            else None
        )

        right_marker = (
            framework_order[
                insert_idx
            ]
            if insert_idx < len(
                framework_order
            )
            else None
        )


        # Marker-to-flank statistics
        left_stats = (
            pair_stats_from_df(
                mapping_geno,
                marker,
                left_marker
            )
            if left_marker is not None
            else (
                np.nan,
                np.nan,
                np.nan
            )
        )

        right_stats = (
            pair_stats_from_df(
                mapping_geno,
                marker,
                right_marker
            )
            if right_marker is not None
            else (
                np.nan,
                np.nan,
                np.nan
            )
        )


        left_n, left_R, left_lod = (
            left_stats
        )

        right_n, right_R, right_lod = (
            right_stats
        )


        # Existing flank-to-flank interval
        if (
            left_marker is not None
            and
            right_marker is not None
        ):

            old_R = (
                R_matrices[
                    group
                ].loc[
                    left_marker,
                    right_marker
                ]
            )

            delta_R = (
                left_R
                +
                right_R
                -
                old_R
            )

        elif left_marker is not None:

            old_R = np.nan
            delta_R = left_R

        elif right_marker is not None:

            old_R = np.nan
            delta_R = right_R

        else:

            old_R = np.nan
            delta_R = np.nan


        # Classify each flank separately
        def classify_pair(
            R,
            lod,
            n
        ):

            if np.isnan(R):
                return None

            if (
                n >= 40
                and
                R <= 0.25
                and
                lod >= 5
            ):
                return "strong"

            elif (
                n >= 40
                and
                R <= 0.30
                and
                lod >= 3.5
            ):
                return "core_supported"

            elif (
                n >= 40
                and
                R <= 0.325
                and
                lod >= 2.0
            ):
                return "weak_supported"

            else:
                return "poor"


        left_class = classify_pair(
            left_R,
            left_lod,
            left_n
        )

        right_class = classify_pair(
            right_R,
            right_lod,
            right_n
        )


        class_rank = {
            "strong": 0,
            "core_supported": 1,
            "weak_supported": 2,
            "poor": 3,
            None: 0
        }


        worst_flank_rank = max(
            class_rank[
                left_class
            ],
            class_rank[
                right_class
            ]
        )


        satellite_insertion_records.append({
            "working_group":
                group,

            "marker":
                marker,

            "consensus_group":
                sat_row[
                    "consensus_group"
                ],

            "component_role":
                sat_row[
                    "component_role"
                ],

            "attachment_rate":
                sat_row[
                    "attachment_rate"
                ],

            "insert_position":
                insert_idx + 1,

            "left_marker":
                left_marker,

            "right_marker":
                right_marker,

            "left_n":
                left_n,

            "left_R":
                left_R,

            "left_lod":
                left_lod,

            "left_class":
                left_class,

            "right_n":
                right_n,

            "right_R":
                right_R,

            "right_lod":
                right_lod,

            "right_class":
                right_class,

            "worst_flank_rank":
                worst_flank_rank,

            "delta_R":
                delta_R
        })


satellite_insertion_tests = pd.DataFrame(
    satellite_insertion_records
)


print("SATELLITE INSERTION TESTS COMPLETE")
print("=" * 100)

print(
    "Candidate placements evaluated:",
    len(
        satellite_insertion_tests
    )
)

### Cell 02.07 — choose the best insertion position for each satellite
* We prioritize:
    - best worst-flank support
    - smallest $$ \Delta R $$
    - strongest minimum LOD.

In [ ]:
# Cell 02.07
# Select best candidate insertion position for each satellite.

satellite_best_placements = (
    satellite_insertion_tests
    .assign(
        min_flank_lod=lambda x:
            x[
                [
                    "left_lod",
                    "right_lod"
                ]
            ]
            .min(
                axis=1,
                skipna=True
            )
    )
    .sort_values(
        [
            "marker",
            "worst_flank_rank",
            "delta_R",
            "min_flank_lod"
        ],
        ascending=[
            True,
            True,
            True,
            False
        ]
    )
    .groupby(
        "marker",
        as_index=False
    )
    .first()
)


def placement_confidence(row):

    rank = row[
        "worst_flank_rank"
    ]

    delta = row[
        "delta_R"
    ]


    if (
        rank == 0
        and
        delta <= 0.10
    ):
        return "high"

    elif (
        rank <= 1
        and
        delta <= 0.15
    ):
        return "moderate"

    elif (
        rank <= 2
        and
        delta <= 0.20
    ):
        return "tentative"

    else:
        return "poor"


satellite_best_placements[
    "placement_confidence"
] = (
    satellite_best_placements.apply(
        placement_confidence,
        axis=1
    )
)


print("BEST SATELLITE INSERTION POSITIONS")
print("=" * 145)

display(
    satellite_best_placements[
        [
            "working_group",
            "marker",
            "component_role",
            "attachment_rate",
            "insert_position",
            "left_marker",
            "right_marker",
            "left_R",
            "left_lod",
            "left_class",
            "right_R",
            "right_lod",
            "right_class",
            "delta_R",
            "placement_confidence"
        ]
    ]
    .sort_values(
        [
            "placement_confidence",
            "working_group",
            "marker"
        ]
    )
)


print("\nPLACEMENT CONFIDENCE COUNTS")
print("-" * 70)

print(
    satellite_best_placements[
        "placement_confidence"
    ].value_counts()
)

### Cell 02.08 — inspect ambiguity of placement
* A marker may have one attractive position or several nearly equivalent positions. We need to know the difference before inserting anything permanently.

In [ ]:
# Cell 02.08
# Compare best and second-best placement for each satellite.
#
# Small differences indicate ambiguous local order.

ranked_satellite_tests = (
    satellite_insertion_tests
    .assign(
        min_flank_lod=lambda x:
            x[
                [
                    "left_lod",
                    "right_lod"
                ]
            ]
            .min(
                axis=1,
                skipna=True
            )
    )
    .sort_values(
        [
            "marker",
            "worst_flank_rank",
            "delta_R",
            "min_flank_lod"
        ],
        ascending=[
            True,
            True,
            True,
            False
        ]
    )
)


top_two = (
    ranked_satellite_tests
    .groupby(
        "marker"
    )
    .head(2)
    .copy()
)


top_two[
    "placement_rank"
] = (
    top_two
    .groupby(
        "marker"
    )
    .cumcount()
    + 1
)


placement_ambiguity = (
    top_two.pivot(
        index=[
            "working_group",
            "marker"
        ],
        columns="placement_rank",
        values=[
            "insert_position",
            "worst_flank_rank",
            "delta_R",
            "min_flank_lod"
        ]
    )
)


placement_ambiguity.columns = [
    f"{metric}_rank{rank}"
    for metric, rank
    in placement_ambiguity.columns
]


placement_ambiguity = (
    placement_ambiguity
    .reset_index()
)


placement_ambiguity[
    "delta_R_gap"
] = (
    placement_ambiguity[
        "delta_R_rank2"
    ]
    -
    placement_ambiguity[
        "delta_R_rank1"
    ]
)


print("SATELLITE PLACEMENT AMBIGUITY")
print("=" * 135)

display(
    placement_ambiguity
    .sort_values(
        "delta_R_gap"
    )
)

### Cell 02.09 — verify the insertion-test table itself

In [ ]:
# Cell 02.09
# Sanity-check the raw satellite insertion table before re-ranking.

print("RAW INSERTION TABLE CHECK")
print("=" * 100)

print(
    "Rows:",
    len(satellite_insertion_tests)
)

print(
    "Markers:",
    satellite_insertion_tests["marker"].nunique()
)

print(
    "\nCandidate-position counts per marker:"
)

display(
    satellite_insertion_tests
    .groupby(
        ["working_group", "marker"]
    )
    .size()
    .reset_index(
        name="n_candidate_positions"
    )
)


# Terminal-position sanity check:
# position 1 must have no left flank;
# final position must have no right flank.

terminal_check = []


for (
    group,
    marker
), temp in satellite_insertion_tests.groupby(
    ["working_group", "marker"]
):

    temp = temp.sort_values(
        "insert_position"
    )

    first = temp.iloc[0]
    last = temp.iloc[-1]


    terminal_check.append({
        "working_group":
            group,

        "marker":
            marker,

        "first_position":
            first["insert_position"],

        "first_left_is_missing":
            pd.isna(
                first["left_marker"]
            ),

        "last_position":
            last["insert_position"],

        "last_right_is_missing":
            pd.isna(
                last["right_marker"]
            )
    })


terminal_check = pd.DataFrame(
    terminal_check
)


print("\nTERMINAL FLANK CHECK")
print("-" * 100)

display(
    terminal_check
)


print(
    "\nAll first positions have no left flank:",
    terminal_check[
        "first_left_is_missing"
    ].all()
)

print(
    "All final positions have no right flank:",
    terminal_check[
        "last_right_is_missing"
    ].all()
)

### Cell 02.10 — correctly select the best complete row
* Here we use groupby(...).head(1) instead of .first().

In [ ]:
# Cell 02.10
# Correct ranking of satellite placements.
#
# IMPORTANT:
# head(1) preserves an entire candidate row.
# groupby().first() must NOT be used here.

ranked_satellite_tests = (
    satellite_insertion_tests
    .copy()
)


ranked_satellite_tests[
    "min_flank_lod"
] = (
    ranked_satellite_tests[
        [
            "left_lod",
            "right_lod"
        ]
    ]
    .min(
        axis=1,
        skipna=True
    )
)


ranked_satellite_tests = (
    ranked_satellite_tests
    .sort_values(
        [
            "working_group",
            "marker",
            "worst_flank_rank",
            "delta_R",
            "min_flank_lod",
            "insert_position"
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


satellite_best_placements = (
    ranked_satellite_tests
    .groupby(
        [
            "working_group",
            "marker"
        ],
        group_keys=False
    )
    .head(1)
    .copy()
    .reset_index(drop=True)
)


def placement_confidence(row):

    rank = row[
        "worst_flank_rank"
    ]

    delta = row[
        "delta_R"
    ]


    if (
        rank == 0
        and
        delta <= 0.10
    ):
        return "high"

    elif (
        rank <= 1
        and
        delta <= 0.15
    ):
        return "moderate"

    elif (
        rank <= 2
        and
        delta <= 0.20
    ):
        return "tentative"

    else:
        return "poor"


satellite_best_placements[
    "placement_confidence"
] = (
    satellite_best_placements.apply(
        placement_confidence,
        axis=1
    )
)


print("CORRECTED BEST SATELLITE INSERTION POSITIONS")
print("=" * 150)

display(
    satellite_best_placements[
        [
            "working_group",
            "marker",
            "component_role",
            "attachment_rate",
            "insert_position",
            "left_marker",
            "right_marker",
            "left_R",
            "left_lod",
            "left_class",
            "right_R",
            "right_lod",
            "right_class",
            "delta_R",
            "placement_confidence"
        ]
    ]
)


print("\nCORRECTED CONFIDENCE COUNTS")
print("-" * 70)

print(
    satellite_best_placements[
        "placement_confidence"
    ].value_counts()
)

### Cell 02.11 — correctly calculate first-vs-second placement ambiguity

In [ ]:
# Cell 02.11
# Obtain true first- and second-best COMPLETE candidate rows.

top_two_corrected = (
    ranked_satellite_tests
    .groupby(
        [
            "working_group",
            "marker"
        ],
        group_keys=False
    )
    .head(2)
    .copy()
)


top_two_corrected[
    "placement_rank"
] = (
    top_two_corrected
    .groupby(
        [
            "working_group",
            "marker"
        ]
    )
    .cumcount()
    + 1
)


best_rows = (
    top_two_corrected[
        top_two_corrected[
            "placement_rank"
        ] == 1
    ]
    .copy()
)


second_rows = (
    top_two_corrected[
        top_two_corrected[
            "placement_rank"
        ] == 2
    ]
    .copy()
)


placement_ambiguity_corrected = (
    best_rows[
        [
            "working_group",
            "marker",
            "insert_position",
            "worst_flank_rank",
            "delta_R",
            "min_flank_lod",
            "left_marker",
            "right_marker"
        ]
    ]
    .rename(
        columns={
            "insert_position":
                "best_position",

            "worst_flank_rank":
                "best_worst_flank_rank",

            "delta_R":
                "best_delta_R",

            "min_flank_lod":
                "best_min_flank_lod",

            "left_marker":
                "best_left_marker",

            "right_marker":
                "best_right_marker"
        }
    )
    .merge(
        second_rows[
            [
                "working_group",
                "marker",
                "insert_position",
                "worst_flank_rank",
                "delta_R",
                "min_flank_lod",
                "left_marker",
                "right_marker"
            ]
        ]
        .rename(
            columns={
                "insert_position":
                    "second_position",

                "worst_flank_rank":
                    "second_worst_flank_rank",

                "delta_R":
                    "second_delta_R",

                "min_flank_lod":
                    "second_min_flank_lod",

                "left_marker":
                    "second_left_marker",

                "right_marker":
                    "second_right_marker"
            }
        ),
        on=[
            "working_group",
            "marker"
        ],
        how="left"
    )
)


placement_ambiguity_corrected[
    "delta_R_gap"
] = (
    placement_ambiguity_corrected[
        "second_delta_R"
    ]
    -
    placement_ambiguity_corrected[
        "best_delta_R"
    ]
)


print("CORRECTED SATELLITE PLACEMENT AMBIGUITY")
print("=" * 155)

display(
    placement_ambiguity_corrected
    .sort_values(
        [
            "best_worst_flank_rank",
            "delta_R_gap"
        ]
    )
)

### Cell 02.12 — show the top five real placements for each satellite
* This is more informative than immediately classifying them. It lets us see whether a satellite has a genuine local placement or just one strong connection.

In [ ]:
# Cell 02.12
# Display top 5 genuine candidate positions for every satellite.

top5_satellite_placements = (
    ranked_satellite_tests
    .groupby(
        [
            "working_group",
            "marker"
        ],
        group_keys=False
    )
    .head(5)
    .copy()
)


top5_satellite_placements[
    "placement_rank"
] = (
    top5_satellite_placements
    .groupby(
        [
            "working_group",
            "marker"
        ]
    )
    .cumcount()
    + 1
)


display(
    top5_satellite_placements[
        [
            "working_group",
            "marker",
            "placement_rank",
            "insert_position",
            "left_marker",
            "right_marker",
            "left_R",
            "left_lod",
            "left_class",
            "right_R",
            "right_lod",
            "right_class",
            "worst_flank_rank",
            "delta_R"
        ]
    ]
)

### Cell 02.13 — classify the 16 satellites

In [ ]:
# Cell 02.13
# Final working classification of the 16 held-out satellites.
#
# These markers retain linkage-group association from the earlier
# graph/resampling analysis, but are NOT assigned genetic-map
# coordinates unless positional evidence is convincing.

satellite_final_status = (
    satellite_best_placements[
        [
            "working_group",
            "marker",
            "consensus_group",
            "component_role",
            "attachment_rate",
            "insert_position",
            "left_marker",
            "right_marker",
            "left_R",
            "left_lod",
            "left_class",
            "right_R",
            "right_lod",
            "right_class",
            "delta_R"
        ]
    ]
    .copy()
)


def classify_satellite_status(row):

    marker = row["marker"]

    # Reasonably convincing group attachment,
    # but not sufficiently resolved for a cM coordinate
    if marker == "Satt444":
        return "associated_unordered_supported"

    # Some positional/linkage evidence, but still weak/ambiguous
    elif marker in [
        "Satt132",
        "Satt305a",
        "Satt424b"
    ]:
        return "associated_unordered_tentative"

    else:
        return "associated_unordered"


satellite_final_status[
    "map_status"
] = (
    satellite_final_status.apply(
        classify_satellite_status,
        axis=1
    )
)


print("FINAL SATELLITE STATUS")
print("=" * 130)

display(
    satellite_final_status[
        [
            "working_group",
            "marker",
            "component_role",
            "attachment_rate",
            "left_marker",
            "right_marker",
            "left_R",
            "left_lod",
            "left_class",
            "right_R",
            "right_lod",
            "right_class",
            "map_status"
        ]
    ]
    .sort_values(
        [
            "working_group",
            "marker"
        ]
    )
)


print("\nSTATUS COUNTS")
print("-" * 70)

print(
    satellite_final_status[
        "map_status"
    ].value_counts()
)

### Cell 02.14 — create the final provisional framework-map table
* This separates ordered map position from linkage-group association.

In [ ]:
# Cell 02.14
# Assemble the primary provisional map table.

provisional_framework_map_final = (
    provisional_framework_map
    .copy()
)


provisional_framework_map_final[
    "map_status"
] = "ordered_framework"


# Add interval information for the marker to its left.
interval_for_marker = (
    provisional_map_intervals[
        [
            "working_group",
            "marker_b",
            "R_ril",
            "meiotic_r_provisional",
            "lod",
            "interval_class",
            "kosambi_cm_provisional",
            "uncertain_interval"
        ]
    ]
    .rename(
        columns={
            "marker_b":
                "marker",

            "R_ril":
                "R_from_previous",

            "meiotic_r_provisional":
                "meiotic_r_from_previous",

            "lod":
                "lod_from_previous",

            "interval_class":
                "interval_class_from_previous",

            "kosambi_cm_provisional":
                "interval_kosambi_cm",

            "uncertain_interval":
                "uncertain_interval_from_previous"
        }
    )
)


provisional_framework_map_final = (
    provisional_framework_map_final
    .merge(
        interval_for_marker,
        on=[
            "working_group",
            "marker"
        ],
        how="left"
    )
)


print("FINAL PROVISIONAL FRAMEWORK MAP")
print("=" * 140)

display(
    provisional_framework_map_final.head(40)
)


print(
    "\nOrdered framework markers:",
    len(
        provisional_framework_map_final
    )
)

### Cell 02.15 — create a master marker-status table
* This is useful later for QTL mapping, supplementary tables, and manuscript reporting.

In [ ]:
# Cell 02.15
# Combine ordered framework markers and associated-but-unordered
# satellites into one master map-status table.

ordered_status = (
    provisional_framework_map_final[
        [
            "working_group",
            "marker",
            "order_position",
            "kosambi_cm_provisional",
            "haldane_cm_provisional",
            "map_status"
        ]
    ]
    .copy()
)


unordered_status = (
    satellite_final_status[
        [
            "working_group",
            "marker",
            "map_status"
        ]
    ]
    .copy()
)


unordered_status[
    "order_position"
] = np.nan

unordered_status[
    "kosambi_cm_provisional"
] = np.nan

unordered_status[
    "haldane_cm_provisional"
] = np.nan


master_linkage_map_status = pd.concat(
    [
        ordered_status,
        unordered_status
    ],
    ignore_index=True
)


master_linkage_map_status = (
    master_linkage_map_status
    .sort_values(
        [
            "working_group",
            "order_position",
            "marker"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


print("MASTER LINKAGE-MAP STATUS")
print("=" * 110)

print(
    "Total group-associated markers:",
    len(
        master_linkage_map_status
    )
)

print(
    "Ordered:",
    (
        master_linkage_map_status[
            "map_status"
        ]
        ==
        "ordered_framework"
    ).sum()
)

print(
    "Associated but unordered:",
    (
        master_linkage_map_status[
            "map_status"
        ]
        !=
        "ordered_framework"
    ).sum()
)


display(
    master_linkage_map_status
)

### Cell 02.16 — export the map checkpoint

In [ ]:
# Cell 02.16
# Export the provisional de novo linkage-map checkpoint.

output_dir = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


xlsx_path = (
    output_dir
    / "flyer_hartwig_provisional_framework_map.xlsx"
)


with pd.ExcelWriter(
    xlsx_path,
    engine="openpyxl"
) as writer:

    provisional_framework_map_final.to_excel(
        writer,
        sheet_name="ordered_framework",
        index=False
    )

    provisional_map_intervals.to_excel(
        writer,
        sheet_name="intervals",
        index=False
    )

    provisional_map_summary.to_excel(
        writer,
        sheet_name="group_summary",
        index=False
    )

    satellite_final_status.to_excel(
        writer,
        sheet_name="unordered_satellites",
        index=False
    )

    master_linkage_map_status.to_excel(
        writer,
        sheet_name="all_group_associated",
        index=False
    )

    adjacency_bootstrap_summary.to_excel(
        writer,
        sheet_name="bootstrap_support",
        index=False
    )


print("MAP CHECKPOINT EXPORTED")
print("=" * 100)

print(
    xlsx_path
)